In [ ]:
from pathlib import Path

zip_files = list(Path("/kaggle/input").rglob("*.zip"))

if zip_files:
    for path in zip_files:
        print(path)
else:
    print("No ZIP found. These are the available files:")
    for path in Path("/kaggle/input").rglob("*"):
        if path.is_file():
            print(path)

In [ ]:
from pathlib import Path
import shutil
import os

source = Path(
    "/kaggle/input/datasets/tejaswinirammohan/current/something_unique"
)
project = Path("/kaggle/working/something_unique")

shutil.copytree(source, project, dirs_exist_ok=True)
os.chdir(project)

print("Project ready:", Path.cwd())
print("\nRequired packages:")
print(Path("requirements.txt").read_text())

In [ ]:
import importlib

packages = [
    "numpy", "pandas", "sklearn",
    "scipy", "joblib", "matplotlib"
]

for name in packages:
    try:
        module = importlib.import_module(name)
        print(f"OK: {name} — {module.__version__}")
    except ImportError as error:
        print(f"MISSING: {name} — {error}")

try:
    from scipy.optimize import milp
    print("OK: MIP solver available")
except ImportError:
    print("MISSING: scipy.optimize.milp")

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable, "-u", "main.py",
        "--duration", "30",
        "--seed", "42",
        "--log-file", "kaggle_test.log"
    ],
    cwd="/kaggle/working/something_unique",
    capture_output=True,
    text=True,
    timeout=120
)

print(result.stdout)

if result.stderr:
    print("Messages / errors:")
    print(result.stderr)

print("Exit code:", result.returncode)

In [ ]:
import subprocess
import sys

for title, extra_args in [
    ("FULL SYSTEM", []),
    ("NO COORDINATION", ["--baseline"])
]:
    print(f"\n===== {title} =====", flush=True)

    result = subprocess.run(
        [
            sys.executable, "-u", "main.py",
            "--duration", "150",
            "--seed", "42",
            "--log-file", f"kaggle_{title.lower().replace(' ', '_')}.log",
            *extra_args
        ],
        cwd="/kaggle/working/something_unique",
        capture_output=True,
        text=True,
        timeout=180
    )

    print(result.stdout)
    if result.stderr:
        print(result.stderr)

    print("Exit code:", result.returncode)

In [ ]:
from pathlib import Path

project = Path("/kaggle/working/something_unique")

for filename in [
    "agents/cluster_placement.py",
    "agents/monitoring.py"
]:
    print(f"\n===== {filename} =====\n")
    print((project / filename).read_text())

In [ ]:
from pathlib import Path

project = Path("/kaggle/working/something_unique")

print("===== SIMULATION CODE =====")
print((project / "simulation.py").read_text())

print("\n===== FORECASTING CODE =====")
print((project / "agents/forecasting.py").read_text())

In [ ]:
from pathlib import Path
import shutil
import py_compile

project = Path("/kaggle/working/something_unique")
file = project / "agents/cluster_placement.py"
backup = file.with_suffix(".py.bak")

# Keep the original file as a backup.
if not backup.exists():
    shutil.copy2(file, backup)

code = file.read_text()

old = """        c_adj = np.where(feasible_mask, c, 1e6)

        res = milp(c=c_adj, constraints=[con_assign], integrality=integrality, bounds=bounds)"""

new = """        # Hard constraint: infeasible nodes cannot be selected.
        bounds = Bounds(
            lb=np.zeros(n),
            ub=feasible_mask.astype(float)
        )

        res = milp(
            c=c,
            constraints=[con_assign],
            integrality=integrality,
            bounds=bounds
        )"""

if old in code:
    updated = code.replace(old, new, 1)
    compile(updated, str(file), "exec")
    file.write_text(updated)
    print("Fixed: nodes without enough CPU are now forbidden.")
elif "ub=feasible_mask.astype(float)" in code:
    print("This fix is already applied.")
else:
    raise RuntimeError("File differs from the expected code. No changes made.")

py_compile.compile(str(file), doraise=True)
print("Syntax check passed.")
print("Original backup:", backup)

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "placement_check", file
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

agent = module.ClusterPlacementAgent()

# A is too full. B is feasible but deliberately expensive.
chosen, _ = agent.solve(
    candidate_nodes=["A", "B"],
    predicted_costs=[1.0, 2_000_000.0],
    remaining_capacity=[0.0, 10.0],
    task_size=2.0
)
assert chosen == "B", f"Wrong destination: {chosen}"

chosen, _ = agent.solve(
    candidate_nodes=["A", "B"],
    predicted_costs=[1.0, 2.0],
    remaining_capacity=[0.0, 1.0],
    task_size=2.0
)
assert chosen is None, "Should reject when neither node fits."

print("Both checks passed: CPU feasibility is enforced.")

In [ ]:
from pathlib import Path
import shutil

project = Path("/kaggle/working/something_unique")
file = project / "agents/monitoring.py"
backup = file.with_suffix(".py.bak")

if not backup.exists():
    shutil.copy2(file, backup)

code = '''"""
Monitoring snapshots of waiting-plus-running resource demand.
These are demand ratios, not measured execution utilization.
"""

from dataclasses import dataclass


@dataclass
class Snapshot:
    time: float
    cpu_load: float
    queue_len: int
    bandwidth_mbps: float
    response_time_s: float
    mem_load: float = 0.0
    combined_load: float = 0.0


class MonitoringAgent:
    def __init__(self, node):
        self.node = node
        self.history = []

    def sample(self, env_time):
        # Keep raw ratios: values above 1 mean demand exceeds capacity.
        cpu = self.node.used_cpu() / self.node.capacity
        memory = self.node.used_mem() / self.node.mem_capacity

        snap = Snapshot(
            time=env_time,
            cpu_load=cpu,
            mem_load=memory,
            combined_load=min(1.0, max(cpu, memory)),
            queue_len=len(self.node.queue),
            bandwidth_mbps=self.node.bandwidth_mbps,
            response_time_s=self.node.avg_response_time(),
        )
        self.history.append(snap)
        return snap

    def recent_load_series(self, window=20):
        # Preserve the input expected by the existing forecasting model.
        return [s.combined_load for s in self.history[-window:]]
'''

compile(code, str(file), "exec")
file.write_text(code)

# Check CPU and memory are separate, while forecasting stays compatible.
namespace = {}
exec(compile(code, str(file), "exec"), namespace)

class ExampleNode:
    capacity = 10.0
    mem_capacity = 8.0
    bandwidth_mbps = 40.0
    queue = []

    def used_cpu(self):
        return 4.0

    def used_mem(self):
        return 7.2

    def avg_response_time(self):
        return 0.0

monitor = namespace["MonitoringAgent"](ExampleNode())
snapshot = monitor.sample(0.0)

assert abs(snapshot.cpu_load - 0.4) < 1e-9
assert abs(snapshot.mem_load - 0.9) < 1e-9
assert abs(monitor.recent_load_series()[0] - 0.9) < 1e-9

print("CPU demand: 40%")
print("Memory demand: 90%")
print("Existing forecaster still receives: 90%")
print("Monitoring checks passed.")

In [ ]:
from pathlib import Path
import shutil

file = Path("/kaggle/working/something_unique/simulation.py")
backup = file.with_suffix(".py.before_migration_fix.bak")

if not backup.exists():
    shutil.copy2(file, backup)

code = file.read_text()

old = '''                if task is not None:
                    self.queue.remove(task)
                    yield self.env.process(self.parent.handle_escalation(self, task, predicted))'''

new = '''                if task is not None:
                    # A task can be considered only once at this timestamp.
                    if task.get("_last_escalation_time") == self.env.now:
                        self.scheduler.enqueue(task, self.env.now)
                        continue

                    task["_last_escalation_time"] = self.env.now
                    self.queue.remove(task)
                    yield self.env.process(self.parent.handle_escalation(self, task, predicted))'''

if old in code:
    updated = code.replace(old, new, 1)
    compile(updated, str(file), "exec")
    file.write_text(updated)
    print("Applied: same-timestamp repeated escalation is blocked.")
elif 'task["_last_escalation_time"] = self.env.now' in code:
    print("Fix already applied.")
else:
    raise RuntimeError("Unexpected code layout. No changes made.")

In [ ]:
from pathlib import Path
import shutil

file = Path("/kaggle/working/something_unique/simulation.py")
backup = file.with_suffix(".py.before_cloud_fix.bak")

code = file.read_text()

old = '''        self.metrics.record_finish(task, finish)
        self.log(now, f"CLOUD: accepted task {task['id']} (will finish at ~t={finish:.1f}s)")'''

new = '''        self.env.process(self._finish_task(task, finish))
        self.log(now, f"CLOUD: accepted task {task['id']} (will finish at ~t={finish:.1f}s)")

    def _finish_task(self, task, finish):
        yield self.env.timeout(max(0.0, finish - self.env.now))
        self.metrics.record_finish(task, self.env.now)
        self.log(self.env.now, f"CLOUD: finished task {task['id']}")'''

old_setup = '''    cloud = CloudTier(log)
    spec = ['''

new_setup = '''    cloud = CloudTier(log)
    cloud.env = env
    spec = ['''

if "def _finish_task(self, task, finish):" in code:
    print("Cloud timing fix already applied.")
else:
    assert code.count(old) == 1, "Unexpected cloud code. No changes made."
    assert code.count(old_setup) == 1, "Unexpected setup. No changes made."

    updated = code.replace(old, new, 1)
    updated = updated.replace(old_setup, new_setup, 1)
    compile(updated, str(file), "exec")

    if not backup.exists():
        shutil.copy2(file, backup)

    file.write_text(updated)
    print("Applied: cloud tasks finish through a timed event.")

In [ ]:
import os
import sys
import subprocess

test_code = """
import desim
from simulation import CloudTier, Metrics

env = desim.Environment()
cloud = CloudTier(lambda *args: None)
cloud.env = env
cloud.metrics = Metrics()

task = {
    "id": "cloud_check",
    "cpu_request": 10.0,
    "arrival_time": 0.0,
    "deadline": 20.0,
}

cloud.accept(task, 0.0)
assert cloud.metrics.completed == 0, "Counted completion too early!"

env.run(until=8.0)
assert cloud.metrics.completed == 1, "Completion event did not run!"
assert cloud.metrics.response_times == [7.0]

print("Passed: task finishes at 7 simulated seconds, not immediately.")
"""

result = subprocess.run(
    [sys.executable, "-c", test_code],
    cwd="/kaggle/working/something_unique",
    capture_output=True,
    text=True,
    timeout=60
)

print(result.stdout)
if result.stderr:
    print(result.stderr)
print("Exit code:", result.returncode)

In [ ]:
from pathlib import Path
import shutil

project = Path("/kaggle/working/something_unique")
placement = project / "agents/cluster_placement.py"
simulation = project / "simulation.py"

changes = {
    placement: [
        (
            "def solve(self, candidate_nodes, predicted_costs, remaining_capacity, task_size=1.0):",
            """def solve(self, candidate_nodes, predicted_costs, remaining_capacity,
              task_size=1.0, remaining_memory=None, task_memory=0.0):"""
        ),
        (
            "        feasible_mask = np.array(remaining_capacity) >= task_size",
            """        feasible_mask = np.array(remaining_capacity) >= task_size
        if remaining_memory is not None:
            feasible_mask &= np.array(remaining_memory) >= task_memory"""
        ),
    ],
    simulation: [
        (
            '[c.remaining_capacity() for c in candidates], task_size=task["cpu_request"])',
            '''[c.remaining_capacity() for c in candidates],
            task_size=task["cpu_request"],
            remaining_memory=[
                max(0.0, c.mem_capacity - c.used_mem())
                for c in candidates
            ],
            task_memory=task["mem_request"])'''
        ),
    ],
}

# Prepare and validate all changes before writing.
prepared = {}

for path, replacements in changes.items():
    code = path.read_text()
    for old, new in replacements:
        if new in code:
            continue
        if code.count(old) != 1:
            raise RuntimeError(
                f"Unexpected code in {path.name}. No files changed."
            )
        code = code.replace(old, new, 1)

    compile(code, str(path), "exec")
    prepared[path] = code

for path, code in prepared.items():
    backup = path.with_suffix(".py.before_memory_fix.bak")
    if not backup.exists():
        shutil.copy2(path, backup)
    path.write_text(code)

print("CPU and memory constraints connected to placement.")

In [ ]:
import importlib.util

path = "/kaggle/working/something_unique/agents/cluster_placement.py"
spec = importlib.util.spec_from_file_location("memory_check", path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

agent = module.ClusterPlacementAgent()

chosen, _ = agent.solve(
    candidate_nodes=["A", "B"],
    predicted_costs=[1.0, 10.0],
    remaining_capacity=[10.0, 10.0],
    task_size=2.0,
    remaining_memory=[1.0, 8.0],
    task_memory=3.0,
)

assert chosen == "B", f"Incorrect destination: {chosen}"

print("Passed: A rejected for insufficient memory; B selected.")

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable, "-u", "main.py",
        "--duration", "150",
        "--seed", "42",
        "--log-file", "after_fixes.log"
    ],
    cwd="/kaggle/working/something_unique",
    capture_output=True,
    text=True,
    timeout=180
)

# Show only the final summary.
output = result.stdout
marker = "----- MULTI-AGENT SYSTEM -----"
print(output[output.find(marker):] if marker in output else output)

if result.stderr:
    print(result.stderr)

print("Exit code:", result.returncode)

In [ ]:
from pathlib import Path

file = Path("/kaggle/working/something_unique/agents/batch_placement.py")

code = '''
import numpy as np
from scipy.optimize import milp, Bounds, LinearConstraint
from scipy.sparse import lil_matrix, csr_matrix


def solve_batch(costs, cpu, memory, free_cpu, free_memory,
                feasible=None, time_limit=30.0):
    """
    costs[i, j]: predicted placement cost for task i on node j.
    feasible[i, j]: whether placement passes deadline/path checks.
    Returns node indices per task; -1 means unassigned.

    Objective:
      1. Place as many tasks as possible.
      2. Minimize cost without reducing that count.
    """
    costs = np.asarray(costs, dtype=float)
    cpu = np.asarray(cpu, dtype=float)
    memory = np.asarray(memory, dtype=float)
    free_cpu = np.asarray(free_cpu, dtype=float)
    free_memory = np.asarray(free_memory, dtype=float)

    if costs.ndim != 2:
        raise ValueError("costs must be a task-by-node matrix.")

    tasks, nodes = costs.shape
    if tasks == 0 or nodes == 0:
        raise ValueError("Supply at least one task and one node.")

    for values, shape in [
        (cpu, (tasks,)), (memory, (tasks,)),
        (free_cpu, (nodes,)), (free_memory, (nodes,))
    ]:
        if values.shape != shape:
            raise ValueError("Resource array has incorrect shape.")

    for values in [costs, cpu, memory, free_cpu, free_memory]:
        if not np.isfinite(values).all() or (values < 0).any():
            raise ValueError("Inputs must be finite and nonnegative.")

    allowed = np.ones((tasks, nodes), dtype=bool)
    if feasible is not None:
        supplied = np.asarray(feasible, dtype=bool)
        if supplied.shape != costs.shape:
            raise ValueError("feasible must match costs.shape.")
        allowed &= supplied

    allowed &= cpu[:, None] <= free_cpu[None, :]
    allowed &= memory[:, None] <= free_memory[None, :]

    variables = tasks * nodes
    matrix = lil_matrix((tasks + 2 * nodes, variables))

    # Each task goes to at most one node.
    for i in range(tasks):
        matrix[i, i * nodes:(i + 1) * nodes] = 1

    # All tasks assigned to a node share its available resources.
    for j in range(nodes):
        columns = np.arange(tasks) * nodes + j
        matrix[tasks + j, columns] = cpu
        matrix[tasks + nodes + j, columns] = memory

    upper = np.concatenate([
        np.ones(tasks), free_cpu, free_memory
    ])
    constraints = LinearConstraint(
        matrix.tocsr(), np.full(len(upper), -np.inf), upper
    )
    bounds = Bounds(np.zeros(variables), allowed.ravel().astype(float))
    options = {"time_limit": time_limit, "mip_rel_gap": 0.0}

    # Stage 1: maximize number placed.
    first = milp(
        c=-np.ones(variables),
        integrality=np.ones(variables),
        bounds=bounds,
        constraints=constraints,
        options=options
    )
    if not first.success:
        raise RuntimeError("Admission solve not optimal: " + first.message)

    placed = int(np.rint(first.x).sum())

    # Stage 2: minimize cost while preserving the placement count.
    count_constraint = LinearConstraint(
        csr_matrix(np.ones((1, variables))), placed, placed
    )
    second = milp(
        c=costs.ravel(),
        integrality=np.ones(variables),
        bounds=bounds,
        constraints=[constraints, count_constraint],
        options=options
    )
    if not second.success:
        raise RuntimeError("Cost solve not optimal: " + second.message)

    assignment = np.rint(second.x).astype(int).reshape(tasks, nodes)
    destinations = np.full(tasks, -1, dtype=int)

    for i in range(tasks):
        selected = np.flatnonzero(assignment[i])
        if len(selected):
            destinations[i] = selected[0]

    return {
        "destinations": destinations.tolist(),
        "placed": placed,
        "unassigned": tasks - placed,
        "cost": float((assignment * costs).sum()),
        "optimal": True
    }
'''

compile(code, str(file), "exec")
file.write_text(code)
print("Batch placement module created.")

In [ ]:
import importlib.util

path = "/kaggle/working/something_unique/agents/batch_placement.py"
spec = importlib.util.spec_from_file_location("batch_check", path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

result = module.solve_batch(
    costs=[[1, 5], [2, 1], [3, 3]],
    cpu=[3, 3, 3],
    memory=[2, 2, 2],
    free_cpu=[4, 4],
    free_memory=[4, 4]
)

assert result["placed"] == 2
assert result["destinations"] == [0, 1, -1]
assert abs(result["cost"] - 2.0) < 1e-6

# Plenty of CPU, but insufficient combined memory for both tasks.
memory_check = module.solve_batch(
    costs=[[1], [2]],
    cpu=[1, 1],
    memory=[3, 3],
    free_cpu=[10],
    free_memory=[4]
)
assert memory_check["placed"] == 1

# A forbidden task-node pair must never be selected.
blocked = module.solve_batch(
    costs=[[1]],
    cpu=[1],
    memory=[1],
    free_cpu=[10],
    free_memory=[10],
    feasible=[[False]]
)
assert blocked["placed"] == 0

print("Passed: combined CPU, combined memory and forbidden placements.")
print("Example:", result)

In [ ]:
import importlib.util
import numpy as np
from time import perf_counter

path = "/kaggle/working/something_unique/agents/batch_placement.py"
spec = importlib.util.spec_from_file_location("batch_baseline", path)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

# Fixed seed makes this example reproducible.
rng = np.random.default_rng(42)

node_names = ["F1", "F2", "F3", "F4", "F5", "F6"]

# Available resources at this scheduling instant.
free_cpu = np.array([2, 3, 2, 8, 9, 10], dtype=float)
free_memory = np.array([2, 2, 3, 6, 7, 8], dtype=float)

task_count = 12
cpu = rng.integers(1, 5, size=task_count).astype(float)
memory = rng.integers(1, 4, size=task_count).astype(float)

# Example placement times for testing the baseline plumbing.
# These are synthetic, not measured or learned predictions.
costs = rng.uniform(1, 10, size=(task_count, 6))

# Each task has this much time left before its deadline.
slack = rng.uniform(4, 12, size=task_count)
feasible = costs <= slack[:, None]

start = perf_counter()

flat_result = module.solve_batch(
    costs=costs,
    cpu=cpu,
    memory=memory,
    free_cpu=free_cpu,
    free_memory=free_memory,
    feasible=feasible
)

elapsed = perf_counter() - start

# Independently verify the returned assignment.
used_cpu = np.zeros(6)
used_memory = np.zeros(6)

for i, node in enumerate(flat_result["destinations"]):
    if node >= 0:
        used_cpu[node] += cpu[i]
        used_memory[node] += memory[i]
        assert feasible[i, node], "Deadline feasibility violated!"

assert np.all(used_cpu <= free_cpu + 1e-6)
assert np.all(used_memory <= free_memory + 1e-6)

print("FLAT-MIP BASELINE — synthetic snapshot")
print("Tasks:", task_count)
print("Candidate nodes:", len(node_names))
print("Placed:", flat_result["placed"])
print("Unassigned:", flat_result["unassigned"])
print("Total example cost:", round(flat_result["cost"], 3))
print("Placement computation time:", round(elapsed, 4), "seconds")
print("CPU, memory and feasibility checks passed.")

print("\nAssignments:")
for i, node in enumerate(flat_result["destinations"]):
    destination = node_names[node] if node >= 0 else "unassigned"
    print(f"Task {i + 1} -> {destination}")

In [ ]:
# Reuse the exact arrays from the flat-baseline cell.
# Assign reproducible task origins: about 65% in Cluster-A.
origin_rng = np.random.default_rng(7)
origins = (origin_rng.random(task_count) >= 0.65).astype(int)

clusters = [
    np.array([0, 1, 2]),  # Cluster-A
    np.array([3, 4, 5])   # Cluster-B
]

remaining_cpu = free_cpu.copy()
remaining_memory = free_memory.copy()
destinations = np.full(task_count, -1, dtype=int)
placement_stage = ["unassigned"] * task_count
solver_calls = 0


def place_group(task_ids, node_ids, stage):
    global solver_calls

    if len(task_ids) == 0:
        return

    result = module.solve_batch(
        costs=costs[np.ix_(task_ids, node_ids)],
        cpu=cpu[task_ids],
        memory=memory[task_ids],
        free_cpu=remaining_cpu[node_ids],
        free_memory=remaining_memory[node_ids],
        feasible=feasible[np.ix_(task_ids, node_ids)]
    )
    solver_calls += 1

    for task_id, local_node in zip(task_ids, result["destinations"]):
        if local_node >= 0:
            node = node_ids[local_node]
            destinations[task_id] = node
            placement_stage[task_id] = stage
            remaining_cpu[node] -= cpu[task_id]
            remaining_memory[node] -= memory[task_id]


start = perf_counter()

# First give both clusters a chance to serve their own tasks.
for cluster_id, nodes in enumerate(clusters):
    task_ids = np.flatnonzero(origins == cluster_id)
    place_group(task_ids, nodes, "own cluster")

# Then try the other cluster for leftovers.
# Fixed processing order is an explicit policy choice.
for cluster_id in range(2):
    leftovers = np.flatnonzero(
        (origins == cluster_id) & (destinations < 0)
    )
    place_group(leftovers, clusters[1 - cluster_id], "other cluster")

hierarchical_elapsed = perf_counter() - start

# Independently verify the final placements.
check_cpu = np.zeros(6)
check_memory = np.zeros(6)
hierarchical_cost = 0.0

for i, node in enumerate(destinations):
    if node >= 0:
        assert feasible[i, node]
        check_cpu[node] += cpu[i]
        check_memory[node] += memory[i]
        hierarchical_cost += costs[i, node]

assert np.all(check_cpu <= free_cpu + 1e-6)
assert np.all(check_memory <= free_memory + 1e-6)

placed = int((destinations >= 0).sum())

print("TWO-CLUSTER PLACEMENT — same synthetic snapshot")
print("Placed:", placed)
print("Unassigned:", task_count - placed)
print("Total example cost:", round(hierarchical_cost, 3))
print("Computation time:", round(hierarchical_elapsed, 4), "seconds")
print("Solver calls:", solver_calls)
print("Maximum nodes per solve: 3")
print("CPU, memory and feasibility checks passed.")

print("\nCompared with flat MIP:")
print("Flat placed:", flat_result["placed"])
print("Cluster method placed:", placed)

if placed == flat_result["placed"] and flat_result["cost"] > 0:
    gap = 100 * (
        hierarchical_cost - flat_result["cost"]
    ) / flat_result["cost"]
    print("Cost increase versus flat:", round(gap, 2), "%")
else:
    print("Placement counts differ; cost alone is not comparable.")

print("\nAssignments:")
for i, node in enumerate(destinations):
    destination = node_names[node] if node >= 0 else "unassigned"
    origin = "A" if origins[i] == 0 else "B"
    print(
        f"Task {i + 1}: origin {origin} -> "
        f"{destination} ({placement_stage[i]})"
    )

In [ ]:
task_id = 6  # Task 7; Python starts counting at zero.

print("Task 7 CPU request:", cpu[task_id])
print("Task 7 memory request:", memory[task_id])

print("\nSpace remaining after cluster placement:")

for j, name in enumerate(node_names):
    fits_cpu = remaining_cpu[j] >= cpu[task_id] - 1e-6
    fits_memory = remaining_memory[j] >= memory[task_id] - 1e-6
    deadline_ok = bool(feasible[task_id, j])

    print(
        f"{name}: "
        f"CPU free={remaining_cpu[j]:.1f}, "
        f"memory free={remaining_memory[j]:.1f}, "
        f"CPU fits={fits_cpu}, "
        f"memory fits={fits_memory}, "
        f"deadline feasible={deadline_ok}"
    )

In [ ]:
from pathlib import Path
import shutil

project = Path("/kaggle/working/something_unique")

archive = shutil.make_archive(
    "/kaggle/working/fog_scheduler_checkpoint",
    "zip",
    root_dir=project.parent,
    base_dir=project.name
)

print("Checkpoint created:", archive)

In [ ]:
import numpy as np
from time import perf_counter

def run_cluster_placement(
    costs, cpu, memory, free_cpu, free_memory,
    feasible, origins, clusters
):
    """Try each task's own cluster, then other clusters for leftovers."""
    remaining_cpu = free_cpu.copy()
    remaining_memory = free_memory.copy()
    destinations = np.full(len(cpu), -1, dtype=int)
    solver_calls = 0

    def place(task_ids, node_ids):
        nonlocal solver_calls

        if len(task_ids) == 0:
            return

        result = module.solve_batch(
            costs=costs[np.ix_(task_ids, node_ids)],
            cpu=cpu[task_ids],
            memory=memory[task_ids],
            free_cpu=remaining_cpu[node_ids],
            free_memory=remaining_memory[node_ids],
            feasible=feasible[np.ix_(task_ids, node_ids)]
        )
        solver_calls += 1

        for task_id, local_node in zip(
            task_ids, result["destinations"]
        ):
            if local_node >= 0:
                node = node_ids[local_node]
                destinations[task_id] = node
                remaining_cpu[node] -= cpu[task_id]
                remaining_memory[node] -= memory[task_id]

    start = perf_counter()

    # Own-cluster placements happen first.
    for cluster_id, nodes in enumerate(clusters):
        place(
            np.flatnonzero(origins == cluster_id),
            np.asarray(nodes)
        )

    # Fixed-order fallback to other clusters.
    for origin_id in range(len(clusters)):
        for target_id, nodes in enumerate(clusters):
            if target_id == origin_id:
                continue

            leftovers = np.flatnonzero(
                (origins == origin_id) & (destinations < 0)
            )
            place(leftovers, np.asarray(nodes))

    elapsed = perf_counter() - start

    used_cpu = np.zeros(len(free_cpu))
    used_memory = np.zeros(len(free_memory))
    total_cost = 0.0

    for i, node in enumerate(destinations):
        if node >= 0:
            assert feasible[i, node]
            used_cpu[node] += cpu[i]
            used_memory[node] += memory[i]
            total_cost += costs[i, node]

    assert np.all(used_cpu <= free_cpu + 1e-6)
    assert np.all(used_memory <= free_memory + 1e-6)

    return {
        "destinations": destinations.tolist(),
        "placed": int((destinations >= 0).sum()),
        "cost": float(total_cost),
        "seconds": elapsed,
        "solver_calls": solver_calls
    }


# Confirm the reusable version reproduces our previous example.
check = run_cluster_placement(
    costs, cpu, memory,
    free_cpu, free_memory,
    feasible, origins, clusters
)

assert check["placed"] == 11
assert abs(check["cost"] - hierarchical_cost) < 1e-6

print("Reusable cluster placement ready.")
print("Previous example reproduced: 11 of 12 tasks placed.")

In [ ]:
import pandas as pd

rows = []

for seed in range(5):
    rng = np.random.default_rng(seed)

    n_tasks = 12
    test_cpu = rng.integers(1, 5, n_tasks).astype(float)
    test_memory = rng.integers(1, 4, n_tasks).astype(float)

    test_costs = rng.uniform(1, 10, (n_tasks, 6))
    test_slack = rng.uniform(4, 12, n_tasks)
    test_feasible = test_costs <= test_slack[:, None]

    test_origins = (rng.random(n_tasks) >= 0.65).astype(int)

    def run_flat():
        start = perf_counter()
        result = module.solve_batch(
            costs=test_costs,
            cpu=test_cpu,
            memory=test_memory,
            free_cpu=free_cpu,
            free_memory=free_memory,
            feasible=test_feasible
        )
        result["seconds"] = perf_counter() - start
        return result

    def run_clusters():
        return run_cluster_placement(
            test_costs, test_cpu, test_memory,
            free_cpu, free_memory,
            test_feasible, test_origins, clusters
        )

    # Alternate execution order to reduce order effects.
    if seed % 2 == 0:
        flat = run_flat()
        scoped = run_clusters()
    else:
        scoped = run_clusters()
        flat = run_flat()

    rows.append({
        "seed": seed,
        "flat_placed": flat["placed"],
        "cluster_placed": scoped["placed"],
        "flat_ms": flat["seconds"] * 1000,
        "cluster_ms": scoped["seconds"] * 1000,
        "flat_cost": flat["cost"],
        "cluster_cost": scoped["cost"],
        "same_count": flat["placed"] == scoped["placed"]
    })

results = pd.DataFrame(rows)

print(results.round(3).to_string(index=False))

print("\nAverage tasks placed out of 12:")
print("Flat:", results["flat_placed"].mean())
print("Cluster:", results["cluster_placed"].mean())

print("\nMedian computation time:")
print("Flat:", round(results["flat_ms"].median(), 3), "ms")
print("Cluster:", round(results["cluster_ms"].median(), 3), "ms")

results.to_csv(
    "/kaggle/working/six_node_snapshot_results.csv",
    index=False
)

print("\nResults written to six_node_snapshot_results.csv")

In [ ]:
def run_tree_snapshot(
    costs, cpu, memory, free_cpu, free_memory,
    feasible, origins
):
    """
    Columns: F1, F2, F3, F4, F5, F6, P1, P2.
    origins: 0 for Cluster-A, 1 for Cluster-B.

    Unassigned tasks are returned as cloud candidates.
    Cloud execution is not simulated here.
    """
    costs = np.asarray(costs, dtype=float)
    cpu = np.asarray(cpu, dtype=float)
    memory = np.asarray(memory, dtype=float)
    free_cpu = np.asarray(free_cpu, dtype=float)
    free_memory = np.asarray(free_memory, dtype=float)
    feasible = np.asarray(feasible, dtype=bool)
    origins = np.asarray(origins)

    assert costs.shape == (len(cpu), 8)
    assert feasible.shape == costs.shape
    assert free_cpu.shape == free_memory.shape == (8,)
    assert origins.shape == (len(cpu),)
    assert np.isin(origins, [0, 1]).all()

    remaining_cpu = free_cpu.copy()
    remaining_memory = free_memory.copy()
    destinations = np.full(len(cpu), -1, dtype=int)
    stages = np.full(len(cpu), "cloud_candidate", dtype=object)
    solve_sizes = []

    def place(task_ids, node_ids, stage):
        if len(task_ids) == 0:
            return

        node_ids = np.asarray(node_ids)
        result = module.solve_batch(
            costs=costs[np.ix_(task_ids, node_ids)],
            cpu=cpu[task_ids],
            memory=memory[task_ids],
            free_cpu=remaining_cpu[node_ids],
            free_memory=remaining_memory[node_ids],
            feasible=feasible[np.ix_(task_ids, node_ids)]
        )
        solve_sizes.append(len(node_ids))

        for task_id, local_node in zip(
            task_ids, result["destinations"]
        ):
            if local_node >= 0:
                node = int(node_ids[local_node])
                destinations[task_id] = node
                stages[task_id] = stage
                remaining_cpu[node] -= cpu[task_id]
                remaining_memory[node] -= memory[task_id]

    start = perf_counter()

    # Tier 1: each controller pools its leaf group's tasks.
    place(np.flatnonzero(origins == 0), [0, 1, 2], "leaf")
    place(np.flatnonzero(origins == 1), [3, 4, 5], "leaf")

    # Tier 2: only leftovers climb to the parent sibling group.
    leftovers = np.flatnonzero(destinations < 0)
    place(leftovers, [6, 7], "parent")

    elapsed = perf_counter() - start

    # Independently check all placements.
    used_cpu = np.zeros(8)
    used_memory = np.zeros(8)
    total_cost = 0.0

    for i, node in enumerate(destinations):
        if node >= 0:
            assert feasible[i, node]
            used_cpu[node] += cpu[i]
            used_memory[node] += memory[i]
            total_cost += costs[i, node]

    assert np.all(used_cpu <= free_cpu + 1e-6)
    assert np.all(used_memory <= free_memory + 1e-6)

    return {
        "destinations": destinations.tolist(),
        "leaf_placed": int((stages == "leaf").sum()),
        "parent_placed": int((stages == "parent").sum()),
        "cloud_candidates": np.flatnonzero(
            destinations < 0
        ).tolist(),
        "cost": float(total_cost),
        "seconds": elapsed,
        "nodes_per_solve": solve_sizes
    }

print("Two-tier tree placement function ready.")

In [ ]:
tree_check = run_tree_snapshot(
    costs=np.ones((4, 8)),
    cpu=np.array([2., 2., 2., 2.]),
    memory=np.array([2., 2., 2., 2.]),
    free_cpu=np.array([2., 0., 0., 2., 0., 0., 2., 2.]),
    free_memory=np.array([2., 0., 0., 2., 0., 0., 2., 2.]),
    feasible=np.ones((4, 8), dtype=bool),
    origins=np.array([0, 0, 1, 1])
)

assert tree_check["leaf_placed"] == 2
assert tree_check["parent_placed"] == 2
assert len(tree_check["cloud_candidates"]) == 0
assert max(tree_check["nodes_per_solve"]) <= 3

print("Leaf tier placed:", tree_check["leaf_placed"])
print("Parent tier placed:", tree_check["parent_placed"])
print("Cloud candidates:", len(tree_check["cloud_candidates"]))
print("Nodes searched per solve:", tree_check["nodes_per_solve"])
print("Tree traversal check passed.")

In [ ]:
import pandas as pd
import numpy as np
from time import perf_counter

tree_rows = []

for seed in range(5):
    rng = np.random.default_rng(seed)

    n_tasks = 16
    cpu_test = rng.integers(1, 5, n_tasks).astype(float)
    mem_test = rng.integers(1, 4, n_tasks).astype(float)
    origins_test = (rng.random(n_tasks) >= 0.65).astype(int)

    # Six leaves plus two compute-capable parents.
    cpu_free = np.array([2, 3, 2, 8, 9, 10, 6, 6], dtype=float)
    mem_free = np.array([2, 2, 3, 6, 7, 8, 5, 5], dtype=float)

    # Synthetic example times, including a parent-tier penalty.
    cost_test = rng.uniform(1, 10, (n_tasks, 8))
    cost_test[:, 6:] += 2.0

    slack_test = rng.uniform(4, 14, n_tasks)
    feasible_test = cost_test <= slack_test[:, None]

    def flat_run():
        start = perf_counter()
        result = module.solve_batch(
            costs=cost_test,
            cpu=cpu_test,
            memory=mem_test,
            free_cpu=cpu_free,
            free_memory=mem_free,
            feasible=feasible_test
        )
        return result, perf_counter() - start

    def tree_run():
        return run_tree_snapshot(
            cost_test, cpu_test, mem_test,
            cpu_free, mem_free,
            feasible_test, origins_test
        )

    if seed % 2 == 0:
        flat, flat_seconds = flat_run()
        tree = tree_run()
    else:
        tree = tree_run()
        flat, flat_seconds = flat_run()

    tree_placed = tree["leaf_placed"] + tree["parent_placed"]

    # Flat has all tree options available, so its optimum
    # cannot admit fewer tasks.
    assert flat["placed"] >= tree_placed

    tree_rows.append({
        "seed": seed,
        "flat_placed": flat["placed"],
        "tree_placed": tree_placed,
        "tree_leaf": tree["leaf_placed"],
        "tree_parent": tree["parent_placed"],
        "tree_cloud_candidates": len(tree["cloud_candidates"]),
        "flat_ms": flat_seconds * 1000,
        "tree_ms": tree["seconds"] * 1000,
        "flat_cost": flat["cost"],
        "tree_cost": tree["cost"]
    })

tree_results = pd.DataFrame(tree_rows)

print(tree_results.round(3).to_string(index=False))

print("\nAverage fog placements out of 16:")
print("Flat:", tree_results["flat_placed"].mean())
print("Tree:", tree_results["tree_placed"].mean())

print("\nMedian computation time:")
print("Flat:", round(tree_results["flat_ms"].median(), 3), "ms")
print("Tree:", round(tree_results["tree_ms"].median(), 3), "ms")

tree_results.to_csv(
    "/kaggle/working/eight_node_tree_snapshot_results.csv",
    index=False
)

In [ ]:
def run_rebalancing_tree(
    costs, cpu, memory, free_cpu, free_memory,
    feasible, origins
):
    start = perf_counter()

    # Stage 1: own leaf cluster, then neighbouring leaf cluster.
    leaves = run_cluster_placement(
        costs[:, :6],
        cpu,
        memory,
        free_cpu[:6],
        free_memory[:6],
        feasible[:, :6],
        origins,
        [np.array([0, 1, 2]), np.array([3, 4, 5])]
    )

    destinations = np.array(leaves["destinations"], dtype=int)
    leaf_placed = leaves["placed"]
    parent_placed = 0
    solver_calls = leaves["solver_calls"]

    # Stage 2: only remaining tasks try the parent tier.
    leftovers = np.flatnonzero(destinations < 0)

    if len(leftovers):
        parents = module.solve_batch(
            costs=costs[leftovers, 6:8],
            cpu=cpu[leftovers],
            memory=memory[leftovers],
            free_cpu=free_cpu[6:8],
            free_memory=free_memory[6:8],
            feasible=feasible[leftovers, 6:8]
        )
        solver_calls += 1

        for task_id, parent_id in zip(
            leftovers, parents["destinations"]
        ):
            if parent_id >= 0:
                destinations[task_id] = 6 + parent_id
                parent_placed += 1

    # Validate the complete assignment.
    used_cpu = np.zeros(8)
    used_memory = np.zeros(8)
    total_cost = 0.0

    for i, node in enumerate(destinations):
        if node >= 0:
            assert feasible[i, node]
            used_cpu[node] += cpu[i]
            used_memory[node] += memory[i]
            total_cost += costs[i, node]

    assert np.all(used_cpu <= free_cpu + 1e-6)
    assert np.all(used_memory <= free_memory + 1e-6)

    return {
        "destinations": destinations.tolist(),
        "leaf_placed": leaf_placed,
        "parent_placed": parent_placed,
        "cloud_candidates": np.flatnonzero(
            destinations < 0
        ).tolist(),
        "cost": float(total_cost),
        "seconds": perf_counter() - start,
        "solver_calls": solver_calls
    }

print("Cross-cluster rebalancing added.")

In [ ]:
import pandas as pd
import numpy as np
from time import perf_counter

rebalancing_rows = []

for seed in range(5):
    rng = np.random.default_rng(seed)

    n_tasks = 16
    cpu_test = rng.integers(1, 5, n_tasks).astype(float)
    mem_test = rng.integers(1, 4, n_tasks).astype(float)
    origins_test = (rng.random(n_tasks) >= 0.65).astype(int)

    # Same six leaves and two compute-capable parents.
    cpu_free = np.array(
        [2, 3, 2, 8, 9, 10, 6, 6], dtype=float
    )
    mem_free = np.array(
        [2, 2, 3, 6, 7, 8, 5, 5], dtype=float
    )

    # Same synthetic costs and deadline feasibility.
    cost_test = rng.uniform(1, 10, (n_tasks, 8))
    cost_test[:, 6:] += 2.0

    slack_test = rng.uniform(4, 14, n_tasks)
    feasible_test = cost_test <= slack_test[:, None]

    def flat_run():
        start = perf_counter()

        result = module.solve_batch(
            costs=cost_test,
            cpu=cpu_test,
            memory=mem_test,
            free_cpu=cpu_free,
            free_memory=mem_free,
            feasible=feasible_test
        )

        return result, perf_counter() - start

    def tree_run():
        return run_rebalancing_tree(
            cost_test,
            cpu_test,
            mem_test,
            cpu_free,
            mem_free,
            feasible_test,
            origins_test
        )

    # Alternate execution order.
    if seed % 2 == 0:
        flat, flat_seconds = flat_run()
        tree = tree_run()
    else:
        tree = tree_run()
        flat, flat_seconds = flat_run()

    tree_placed = tree["leaf_placed"] + tree["parent_placed"]

    assert flat["placed"] >= tree_placed

    rebalancing_rows.append({
        "seed": seed,
        "flat_placed": flat["placed"],
        "tree_placed": tree_placed,
        "tree_leaf": tree["leaf_placed"],
        "tree_parent": tree["parent_placed"],
        "tree_cloud_candidates": len(tree["cloud_candidates"]),
        "flat_ms": flat_seconds * 1000,
        "tree_ms": tree["seconds"] * 1000,
        "flat_cost": flat["cost"],
        "tree_cost": tree["cost"],
        "tree_solver_calls": tree["solver_calls"]
    })

rebalancing_results = pd.DataFrame(rebalancing_rows)

print(rebalancing_results.round(3).to_string(index=False))

print("\nAverage fog placements out of 16:")
print("Flat:", rebalancing_results["flat_placed"].mean())
print("Tree:", rebalancing_results["tree_placed"].mean())

print("\nMedian computation time:")
print(
    "Flat:",
    round(rebalancing_results["flat_ms"].median(), 3),
    "ms"
)
print(
    "Tree:",
    round(rebalancing_results["tree_ms"].median(), 3),
    "ms"
)

output_path = "/kaggle/working/eight_node_rebalancing_results.csv"
rebalancing_results.to_csv(output_path, index=False)

print("\nResults saved:", output_path)

In [ ]:
from pathlib import Path

project = Path("/kaggle/working/something_unique")

for name in [
    "main.py",
    "agents/cost_prediction.py",
    "agents/local_scheduling.py"
]:
    print(f"\n===== {name} =====\n")
    print((project / name).read_text())

In [ ]:
from pathlib import Path
import random
import json
import os
import sys

project = Path("/kaggle/working/something_unique")
os.chdir(project)

if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from agents.task_classification import TaskClassificationAgent

classifier = TaskClassificationAgent().load(
    str(project / "models/task_classification.joblib")
)

def create_replay(duration=150.0, rate=1.4, seed=42, hotspot=0.65):
    rng = random.Random(seed)
    tasks = []
    now = 0.0

    weights = [
        hotspot / 3,
        hotspot / 3,
        hotspot / 3,
        (1 - hotspot) / 3,
        (1 - hotspot) / 3,
        (1 - hotspot) / 3
    ]

    while True:
        now += rng.expovariate(rate)

        if now >= duration:
            break

        origin = rng.choices(range(6), weights=weights)[0]
        cpu = rng.gammavariate(2.0, 1.2)

        task = {
            "id": f"T{len(tasks) + 1}",
            "arrival_time": now,
            "origin_node": origin,
            "cpu_request": cpu,
            "mem_request": cpu * rng.uniform(0.3, 0.8),
            "data_size_mb": rng.expovariate(1 / 40),
            "urgency_flag": int(rng.random() < 0.2),
            "source_device": origin % 3,
            "deadline": now + rng.uniform(10, 90)
        }

        task["label"] = classifier.classify(task)
        tasks.append(task)

    return tasks


replay_tasks = create_replay()

output = project / "data/replay_seed42.json"
output.parent.mkdir(parents=True, exist_ok=True)

payload = {
    "config": {
        "workload": "synthetic",
        "duration": 150.0,
        "arrival_rate": 1.4,
        "seed": 42,
        "hotspot": 0.65
    },
    "tasks": replay_tasks
}

output.write_text(json.dumps(payload, indent=2))

# Verify reproducibility and original deadlines.
assert replay_tasks == create_replay()
assert all(t["deadline"] > t["arrival_time"] for t in replay_tasks)
assert len({t["id"] for t in replay_tasks}) == len(replay_tasks)

print("Fixed replay workload saved.")
print("Total tasks:", len(replay_tasks))
print(
    "Cluster-A tasks:",
    sum(t["origin_node"] < 3 for t in replay_tasks)
)
print(
    "Cluster-B tasks:",
    sum(t["origin_node"] >= 3 for t in replay_tasks)
)
print("Reproducibility checks passed.")
print("File:", output)

In [ ]:
import pandas as pd
import sklearn
import shutil
import json

from agents.task_classification import TaskClassificationAgent

model_path = project / "models/task_classification.joblib"
backup = model_path.with_suffix(".joblib.original")

if not backup.exists():
    shutil.copy2(model_path, backup)

# Train using the existing example dataset.
training_data = pd.read_csv(
    project / "data/task_classification.csv"
)

classifier = TaskClassificationAgent()
accuracy = classifier.train(training_data)
classifier.save(str(model_path))

# Reload the freshly trained model.
classifier = TaskClassificationAgent().load(str(model_path))

# Update labels only: arrivals and resource requests stay the same.
replay_path = project / "data/replay_seed42.json"
payload = json.loads(replay_path.read_text())

for task in payload["tasks"]:
    task["label"] = classifier.classify(task)

payload["config"]["classifier_sklearn_version"] = sklearn.__version__
replay_path.write_text(json.dumps(payload, indent=2))

replay_tasks = payload["tasks"]

print("Classifier rebuilt with scikit-learn:", sklearn.__version__)
print("Example-data test accuracy:", round(accuracy, 4))
print("Replay labels updated.")
print("Total tasks:", len(replay_tasks))

In [ ]:
from pathlib import Path
import pandas as pd
import shutil
import sklearn

from agents.cost_prediction import CostPredictionAgent

project = Path("/kaggle/working/something_unique")

compute_path = project / "models/cost_compute.joblib"
transfer_path = project / "models/cost_transfer.joblib"

for path in [compute_path, transfer_path]:
    backup = path.with_suffix(".joblib.original")
    if path.exists() and not backup.exists():
        shutil.copy2(path, backup)

training_data = pd.read_csv(project / "data/cost_data.csv")

cost_agent = CostPredictionAgent()
compute_mse, transfer_mse = cost_agent.train(training_data)

cost_agent.save(str(compute_path), str(transfer_path))

# Reload the freshly trained models.
cost_agent = CostPredictionAgent().load(
    str(compute_path),
    str(transfer_path)
)

print("Cost models rebuilt with scikit-learn:", sklearn.__version__)
print("Compute test RMSE:", round(compute_mse ** 0.5, 4))
print("Transfer test RMSE:", round(transfer_mse ** 0.5, 4))
print("Both models saved and reloaded.")

In [ ]:
import heapq
import itertools
import copy
import numpy as np
from time import perf_counter
from agents.local_scheduling import LocalSchedulingAgent


def run_replay_comparison(tasks, policy, round_seconds=5.0):
    """
    Six execution nodes; parents coordinate only.
    Reservations cover tasks in transfer, waiting and running.
    Arrivals stop after the replay; every submitted task is drained.

    Fixed routing isolates placement scope.
    Forecasting and learned escalation are not enabled in this experiment.
    """
    if policy not in ("flat", "hierarchical"):
        raise ValueError("Use flat or hierarchical.")

    tasks = copy.deepcopy(tasks)
    cpu_capacity = np.array([8, 10, 12, 8, 10, 12], dtype=float)
    mem_capacity = cpu_capacity * 0.8
    bandwidth = np.array([40, 60, 100, 40, 60, 100], dtype=float)

    reserved_cpu = np.zeros(6)
    reserved_mem = np.zeros(6)
    queues = [LocalSchedulingAgent() for _ in range(6)]
    busy = [False] * 6

    events = []
    sequence = itertools.count()
    pending = []
    completed = []
    scheduling_times = []
    solver_calls = 0

    def event(time, kind, payload):
        heapq.heappush(
            events, (float(time), next(sequence), kind, payload)
        )

    for task in tasks:
        event(task["arrival_time"], "arrival", task)

    if not tasks:
        return {
            "tasks": [], "scheduling_times": [], "solver_calls": 0
        }

    last_arrival = max(t["arrival_time"] for t in tasks)
    event(0.0, "round", None)

    def start_worker(node, now):
        if busy[node] or len(queues[node]) == 0:
            return

        queues[node].rescore_all(now)
        task = queues[node].pop_next()
        task["start_time"] = now
        busy[node] = True
        event(now + task["actual_compute_s"], "finish", (node, task))

    def finish_record(task, now):
        task["finish_time"] = now
        task["response_s"] = now - task["arrival_time"]
        task["deadline_met"] = now <= task["deadline"]
        task["batch_wait_s"] = (
            task["placement_time"] - task["arrival_time"]
        )
        task["queue_wait_s"] = (
            task["start_time"] - task["ready_time"]
        )
        completed.append(task)

    while events:
        now, _, kind, payload = heapq.heappop(events)

        if kind == "arrival":
            pending.append(payload)

        elif kind == "ready":
            node, task = payload
            task["ready_time"] = now
            queues[node].enqueue(task, now)
            start_worker(node, now)

        elif kind == "finish":
            node, task = payload
            reserved_cpu[node] -= task["cpu_request"]
            reserved_mem[node] -= task["mem_request"]
            busy[node] = False
            finish_record(task, now)
            start_worker(node, now)

        elif kind == "cloud_finish":
            task = payload
            finish_record(task, now)

        elif kind == "round":
            if pending:
                batch = pending[:]
                pending.clear()
                count = len(batch)

                t0 = perf_counter()

                cpu = np.array([t["cpu_request"] for t in batch])
                memory = np.array([t["mem_request"] for t in batch])
                origins = np.array([t["origin_node"] for t in batch])

                load = np.minimum(
                    1.0,
                    np.maximum(
                        reserved_cpu / cpu_capacity,
                        reserved_mem / mem_capacity
                    )
                )

                predicted_compute = np.zeros((count, 6))
                transfer = np.zeros((count, 6))

                for i, task in enumerate(batch):
                    for j in range(6):
                        predicted_compute[i, j] = max(
                            0.2,
                            float(cost_agent.compute_model.predict(
                                [[task["cpu_request"], load[j]]]
                            )[0])
                        )

                        if j != origins[i]:
                            cross = j // 3 != origins[i] // 3
                            hops = 3 if cross else 1
                            link = min(bandwidth[origins[i]], bandwidth[j])
                            transfer[i, j] = (
                                8.0 * task["data_size_mb"] / link
                            ) * (1.0 + 0.5 * hops)

                # Approximate existing waiting work, including transfers.
                waiting_bound = np.zeros(6)
                

                # Use current scheduled finish and reserved queued work.
                for time, _, queued_kind, queued_payload in events:
                    if queued_kind == "finish":
                        node, _task = queued_payload
                        waiting_bound[node] += max(0.0, time - now)
                    elif queued_kind == "ready":
                        node, _task = queued_payload
                        waiting_bound[node] += (
                            max(0.0, time - now)
                            + _task["actual_compute_s"]
                        )

                for j in range(6):
                    waiting_bound[j] += sum(
                        t["actual_compute_s"]
                        for t in queues[j].peek_all()
                    )

                costs = predicted_compute + transfer
                slack = np.array([t["deadline"] - now for t in batch])

                # Pairwise estimate; jointly admitted tasks can add more wait.
                # Actual deadline success is measured at completion.
                feasible = (
                    costs + waiting_bound[None, :]
                    <= slack[:, None]
                )

                free_cpu = np.maximum(0.0, cpu_capacity - reserved_cpu)
                free_mem = np.maximum(0.0, mem_capacity - reserved_mem)
                destinations = np.full(count, -1, dtype=int)

                def place(task_ids, nodes):
                    nonlocal solver_calls
                    if len(task_ids) == 0:
                        return
                    nodes = np.asarray(nodes)
                    result = module.solve_batch(
                        costs[np.ix_(task_ids, nodes)],
                        cpu[task_ids],
                        memory[task_ids],
                        free_cpu[nodes],
                        free_mem[nodes],
                        feasible[np.ix_(task_ids, nodes)]
                    )
                    solver_calls += 1

                    for i, local_node in zip(
                        task_ids, result["destinations"]
                    ):
                        if local_node >= 0:
                            j = int(nodes[local_node])
                            destinations[i] = j
                            free_cpu[j] -= cpu[i]
                            free_mem[j] -= memory[i]

                if policy == "flat":
                    place(np.arange(count), np.arange(6))
                else:
                    for group in range(2):
                        place(
                            np.flatnonzero(origins // 3 == group),
                            np.arange(group * 3, group * 3 + 3)
                        )

                    for group in range(2):
                        leftovers = np.flatnonzero(
                            (origins // 3 == group)
                            & (destinations < 0)
                        )
                        other = 1 - group
                        place(
                            leftovers,
                            np.arange(other * 3, other * 3 + 3)
                        )

                scheduling_times.append(perf_counter() - t0)

                for i, task in enumerate(batch):
                    task["placement_time"] = now
                    j = destinations[i]

                    if j < 0:
                        task["route"] = "cloud"
                        task["destination"] = "cloud"
                        task["transfer_s"] = (
                            4.0 + 8.0 * task["data_size_mb"] / 20.0
                        )
                        task["actual_compute_s"] = (
                            0.3 * task["cpu_request"]
                        )
                        task["ready_time"] = now + task["transfer_s"]
                        task["start_time"] = task["ready_time"]
                        event(
                            task["start_time"] + task["actual_compute_s"],
                            "cloud_finish", task
                        )
                    else:
                        task["destination"] = f"F{j + 1}"
                        task["route"] = (
                            "local" if j == origins[i]
                            else "sibling" if j // 3 == origins[i] // 3
                            else "cross_cluster"
                        )
                        task["transfer_s"] = float(transfer[i, j])

                        # Explicit synthetic execution model.
                        task["actual_compute_s"] = max(
                            0.2, task["cpu_request"] * (1 + 2 * load[j])
                        )

                        reserved_cpu[j] += cpu[i]
                        reserved_mem[j] += memory[i]
                        event(
                            now + task["transfer_s"],
                            "ready", (j, task)
                        )

                assert np.all(reserved_cpu <= cpu_capacity + 1e-6)
                assert np.all(reserved_mem <= mem_capacity + 1e-6)

            if now < last_arrival:
                event(now + round_seconds, "round", None)

    assert len(completed) == len(tasks)
    assert len({t["id"] for t in completed}) == len(tasks)
    assert np.allclose(reserved_cpu, 0.0)
    assert np.allclose(reserved_mem, 0.0)

    return {
        "tasks": completed,
        "scheduling_times": scheduling_times,
        "solver_calls": solver_calls
    }

print("Timed replay comparison function ready.")

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

project = Path("/kaggle/working/something_unique")

payload = json.loads(
    (project / "data/replay_seed42.json").read_text()
)
tasks = payload["tasks"]

comparison_runs = {}
summary_rows = []

for policy in ["flat", "hierarchical"]:
    print(f"Running {policy}...", flush=True)

    result = run_replay_comparison(tasks, policy)
    comparison_runs[policy] = result

    df = pd.DataFrame(result["tasks"])

    # Check task accounting and response-time decomposition.
    assert len(df) == len(tasks)
    assert df["id"].is_unique

    reconstructed = (
        df["batch_wait_s"]
        + df["transfer_s"]
        + df["queue_wait_s"]
        + df["actual_compute_s"]
    )
    assert np.allclose(df["response_s"], reconstructed)
    assert (df["queue_wait_s"] >= -1e-6).all()

    summary_rows.append({
        "policy": policy,
        "submitted": len(tasks),
        "completed": len(df),
        "deadline_met": int(df["deadline_met"].sum()),
        "deadline_missed": int((~df["deadline_met"]).sum()),
        "success_pct": 100 * df["deadline_met"].mean(),
        "mean_response_s": df["response_s"].mean(),
        "p95_response_s": df["response_s"].quantile(0.95),
        "cloud_tasks": int((df["route"] == "cloud").sum()),
        "scheduler_total_ms": 1000 * sum(
            result["scheduling_times"]
        ),
        "solver_calls": result["solver_calls"]
    })

    results_dir = project / "results"
    results_dir.mkdir(exist_ok=True)
    df.to_csv(results_dir / f"replay_{policy}_seed42.csv", index=False)

    print(f"\n{policy.upper()} — response by route:")
    print(
        df.groupby("route").agg(
            tasks=("id", "count"),
            mean_response_s=("response_s", "mean"),
            mean_batch_wait_s=("batch_wait_s", "mean"),
            mean_transfer_s=("transfer_s", "mean"),
            mean_queue_wait_s=("queue_wait_s", "mean"),
            mean_compute_s=("actual_compute_s", "mean"),
            success_pct=("deadline_met", lambda x: 100 * x.mean())
        ).round(3).to_string()
    )

summary = pd.DataFrame(summary_rows)
summary.to_csv(
    project / "results/replay_comparison_seed42.csv",
    index=False
)

print("\n===== OVERALL COMPARISON =====")
print(summary.round(3).to_string(index=False))
print("\nTask accounting and timing checks passed.")

In [ ]:
flat_tasks = pd.DataFrame(comparison_runs["flat"]["tasks"])
hier_tasks = pd.DataFrame(comparison_runs["hierarchical"]["tasks"])

paired = flat_tasks[
    ["id", "route", "response_s", "deadline_met"]
].merge(
    hier_tasks[["id", "route", "response_s", "deadline_met"]],
    on="id",
    suffixes=("_flat", "_hier"),
    validate="one_to_one"
)

print("Same tasks: flat route versus hierarchical route")
print(pd.crosstab(paired["route_flat"], paired["route_hier"]))

print("\nDeadline outcomes:")
print(pd.crosstab(
    paired["deadline_met_flat"],
    paired["deadline_met_hier"]
))

print("\nMean response change, hierarchical minus flat:")
print(round(
    (paired["response_s_hier"] - paired["response_s_flat"]).mean(),
    3
), "seconds")

In [ ]:
import inspect
import textwrap

source = textwrap.dedent(inspect.getsource(run_replay_comparison))

old_prediction = '''                        predicted_compute[i, j] = max(
                            0.2,
                            float(cost_agent.compute_model.predict(
                                [[task["cpu_request"], load[j]]]
                            )[0])
                        )'''

batch_prediction = '''                # Predict every task-node pair in one RF call.
                features = np.column_stack([
                    np.repeat(cpu, 6),
                    np.tile(load, count)
                ])
                predicted_compute = np.maximum(
                    0.2,
                    cost_agent.compute_model.predict(features)
                ).reshape(count, 6)

'''

assert old_prediction in source, (
    "Function differs from the expected version. No changes made."
)

updated = source.replace(old_prediction, "", 1)

anchor = '''                for i, task in enumerate(batch):
                    for j in range(6):'''

assert anchor in updated
updated = updated.replace(anchor, batch_prediction + anchor, 1)

# Keep a separate timer for each batch-solver call.
updated = updated.replace(
    "    solver_calls = 0",
    "    solver_calls = 0\n    mip_times = []",
    1
)

solver_anchor = '''                    result = module.solve_batch('''
assert solver_anchor in updated

updated = updated.replace(
    solver_anchor,
    '''                    mip_start = perf_counter()
                    result = module.solve_batch(''',
    1
)

count_anchor = '''                    solver_calls += 1'''
assert count_anchor in updated

updated = updated.replace(
    count_anchor,
    '''                    mip_times.append(perf_counter() - mip_start)
                    solver_calls += 1''',
    1
)

return_anchor = '''        "solver_calls": solver_calls'''
assert return_anchor in updated

updated = updated.replace(
    return_anchor,
    '''        "solver_calls": solver_calls,
        "mip_times": mip_times''',
    1
)

# Compile before replacing the function.
compile(updated, "<batched_replay>", "exec")

# Preserve the old implementation for a behavior check.
run_replay_before_batching = run_replay_comparison

exec(compile(updated, "<batched_replay>", "exec"), globals())

print("RF predictions batched.")
print("Separate batch-MIP timing added.")

In [ ]:
new_runs = {}
timing_rows = []

for policy in ["flat", "hierarchical"]:
    print("Running", policy, flush=True)
    result = run_replay_comparison(tasks, policy)
    new_runs[policy] = result

    frame = pd.DataFrame(result["tasks"])

    # Confirm predictions and decisions remain equivalent.
    old_frame = pd.DataFrame(
        comparison_runs[policy]["tasks"]
    ).set_index("id").sort_index()

    new_frame = frame.set_index("id").sort_index()

    assert old_frame.index.equals(new_frame.index)
    assert old_frame["destination"].equals(new_frame["destination"])
    assert np.allclose(
        old_frame["response_s"], new_frame["response_s"]
    )

    total_ms = 1000 * sum(result["scheduling_times"])
    mip_ms = 1000 * sum(result["mip_times"])

    timing_rows.append({
        "policy": policy,
        "completed": len(frame),
        "success_pct": 100 * frame["deadline_met"].mean(),
        "scheduler_total_ms": total_ms,
        "batch_mip_ms": mip_ms,
        "other_scheduling_ms": total_ms - mip_ms,
        "solver_calls": result["solver_calls"]
    })

print(pd.DataFrame(timing_rows).round(3).to_string(index=False))
print("\nBehavior unchanged after batching predictions.")

In [ ]:
from pathlib import Path

project = Path("/kaggle/working/something_unique")
runner_path = project / "replay_comparison.py"

# `updated` contains the function source from the batching cell.
header = """import heapq
import itertools
import copy
import numpy as np
from time import perf_counter
from agents.local_scheduling import LocalSchedulingAgent
from agents.cost_prediction import CostPredictionAgent
import agents.batch_placement as module
from pathlib import Path

ROOT = Path(__file__).resolve().parent
cost_agent = CostPredictionAgent().load(
    str(ROOT / "models/cost_compute.joblib"),
    str(ROOT / "models/cost_transfer.joblib")
)

"""

compile(header + updated, str(runner_path), "exec")
runner_path.write_text(header + updated)

print("Comparison function saved:", runner_path)

In [ ]:
from pathlib import Path
import inspect
import textwrap

project = Path("/kaggle/working/something_unique")
runner_path = project / "replay_comparison.py"

if "run_replay_comparison" not in globals():
    raise RuntimeError(
        "Please rerun the cell defining run_replay_comparison first."
    )

# Read the currently active function directly.
function_source = textwrap.dedent(
    inspect.getsource(run_replay_comparison)
)

header = """import heapq
import itertools
import copy
import numpy as np
from time import perf_counter
from pathlib import Path
from agents.local_scheduling import LocalSchedulingAgent
from agents.cost_prediction import CostPredictionAgent
import agents.batch_placement as module

ROOT = Path(__file__).resolve().parent
cost_agent = CostPredictionAgent().load(
    str(ROOT / "models/cost_compute.joblib"),
    str(ROOT / "models/cost_transfer.joblib")
)

"""

code = header + function_source
compile(code, str(runner_path), "exec")
runner_path.write_text(code)

print("Comparison function saved:", runner_path)
print("Separate MIP timing included:", '"mip_times"' in function_source)

In [ ]:
from pathlib import Path

project = Path("/kaggle/working/something_unique")

for name in [
    "agents/batch_placement.py",
    "models/cost_compute.joblib",
    "data/replay_seed42.json",
    "replay_comparison.py"
]:
    path = project / name
    print(f"{'FOUND' if path.exists() else 'MISSING'}: {name}")

In [ ]:
from pathlib import Path

wanted = {
    "main.py",
    "batch_placement.py",
    "cost_compute.joblib",
    "replay_seed42.json",
    "replay_comparison.py",
}

for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    print(f"\n===== {root} =====")
    found = False

    if root.exists():
        for path in root.rglob("*"):
            if path.is_file() and (
                path.name in wanted
                or path.suffix.lower() == ".zip"
            ):
                print(path)
                found = True

    if not found:
        print("No matching project files or ZIPs found.")

In [ ]:
from pathlib import Path
import shutil

file = Path("/kaggle/working/something_unique/simulation.py")
backup = file.with_suffix(".py.before_migration_fix.bak")

code = file.read_text()

old = '''                if task is not None:
                    self.queue.remove(task)
                    yield self.env.process(self.parent.handle_escalation(self, task, predicted))'''

new = '''                if task is not None:
                    if task.get("_last_escalation_time") == self.env.now:
                        self.scheduler.enqueue(task, self.env.now)
                        continue

                    task["_last_escalation_time"] = self.env.now
                    self.queue.remove(task)
                    yield self.env.process(self.parent.handle_escalation(self, task, predicted))'''

if 'task["_last_escalation_time"] = self.env.now' in code:
    print("Fix already applied.")
elif code.count(old) == 1:
    updated_code = code.replace(old, new, 1)
    compile(updated_code, str(file), "exec")

    if not backup.exists():
        shutil.copy2(file, backup)

    file.write_text(updated_code)
    print("Applied: same-timestamp repeated escalation is blocked.")
else:
    raise RuntimeError("Code differs from expected. No changes made.")

In [ ]:
from pathlib import Path
import shutil

file = Path("/kaggle/working/something_unique/simulation.py")
code = file.read_text()

old = '''        self.metrics.record_finish(task, finish)
        self.log(now, f"CLOUD: accepted task {task['id']} (will finish at ~t={finish:.1f}s)")'''

new = '''        self.env.process(self._finish_task(task, finish))
        self.log(now, f"CLOUD: accepted task {task['id']} (will finish at ~t={finish:.1f}s)")

    def _finish_task(self, task, finish):
        yield self.env.timeout(max(0.0, finish - self.env.now))
        self.metrics.record_finish(task, self.env.now)
        self.log(self.env.now, f"CLOUD: finished task {task['id']}")'''

old_setup = '''    cloud = CloudTier(log)
    spec = ['''

new_setup = '''    cloud = CloudTier(log)
    cloud.env = env
    spec = ['''

if "def _finish_task(self, task, finish):" in code:
    print("Cloud timing fix already applied.")
else:
    assert code.count(old) == 1, "Unexpected code. No changes made."
    assert code.count(old_setup) == 1, "Unexpected setup. No changes made."

    updated_code = code.replace(old, new, 1)
    updated_code = updated_code.replace(old_setup, new_setup, 1)
    compile(updated_code, str(file), "exec")

    backup = file.with_suffix(".py.before_cloud_fix.bak")
    if not backup.exists():
        shutil.copy2(file, backup)

    file.write_text(updated_code)
    print("Applied: cloud tasks finish through a timed event.")

In [ ]:
from pathlib import Path
import shutil

project = Path("/kaggle/working/something_unique")

changes = {
    project / "agents/cluster_placement.py": [
        (
            "def solve(self, candidate_nodes, predicted_costs, remaining_capacity, task_size=1.0):",
            """def solve(self, candidate_nodes, predicted_costs, remaining_capacity,
              task_size=1.0, remaining_memory=None, task_memory=0.0):"""
        ),
        (
            "        feasible_mask = np.array(remaining_capacity) >= task_size",
            """        feasible_mask = np.array(remaining_capacity) >= task_size
        if remaining_memory is not None:
            feasible_mask &= np.array(remaining_memory) >= task_memory"""
        ),
    ],
    project / "simulation.py": [
        (
            '[c.remaining_capacity() for c in candidates], task_size=task["cpu_request"])',
            '''[c.remaining_capacity() for c in candidates],
            task_size=task["cpu_request"],
            remaining_memory=[
                max(0.0, c.mem_capacity - c.used_mem())
                for c in candidates
            ],
            task_memory=task["mem_request"])'''
        ),
    ],
}

prepared = {}

for path, replacements in changes.items():
    code = path.read_text()

    for old, new in replacements:
        if new in code:
            continue
        if code.count(old) != 1:
            raise RuntimeError(
                f"Unexpected code in {path.name}. No files changed."
            )
        code = code.replace(old, new, 1)

    compile(code, str(path), "exec")
    prepared[path] = code

for path, code in prepared.items():
    backup = path.with_suffix(".py.before_memory_fix.bak")
    if not backup.exists():
        shutil.copy2(path, backup)
    path.write_text(code)

print("Restored: placement checks both CPU and memory.")

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import sklearn

project = Path("/kaggle/working/something_unique")

if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from agents.task_classification import TaskClassificationAgent

training_data = pd.read_csv(
    project / "data/task_classification.csv"
)

classifier = TaskClassificationAgent()
accuracy = classifier.train(training_data)
classifier.save(str(project / "models/task_classification.joblib"))

print("Classifier rebuilt with scikit-learn:", sklearn.__version__)
print("Synthetic-label test accuracy:", round(accuracy, 4))

In [ ]:
import random
import json

def create_replay(duration=150.0, rate=1.4, seed=42, hotspot=0.65):
    rng = random.Random(seed)
    tasks = []
    now = 0.0

    weights = [hotspot / 3] * 3 + [(1 - hotspot) / 3] * 3

    while True:
        now += rng.expovariate(rate)
        if now >= duration:
            break

        origin = rng.choices(range(6), weights=weights)[0]
        cpu = rng.gammavariate(2.0, 1.2)

        task = {
            "id": f"T{len(tasks) + 1}",
            "arrival_time": now,
            "origin_node": origin,
            "cpu_request": cpu,
            "mem_request": cpu * rng.uniform(0.3, 0.8),
            "data_size_mb": rng.expovariate(1 / 40),
            "urgency_flag": int(rng.random() < 0.2),
            "source_device": origin % 3,
            "deadline": now + rng.uniform(10, 90)
        }
        task["label"] = classifier.classify(task)
        tasks.append(task)

    return tasks


replay_tasks = create_replay()
tasks = replay_tasks

payload = {
    "config": {
        "workload": "synthetic",
        "duration": 150.0,
        "arrival_rate": 1.4,
        "seed": 42,
        "hotspot": 0.65,
        "classifier_sklearn_version": sklearn.__version__
    },
    "tasks": replay_tasks
}

output = project / "data/replay_seed42.json"
output.write_text(json.dumps(payload, indent=2))

assert replay_tasks == create_replay()
assert len(replay_tasks) == 189

print("Replay restored:", len(replay_tasks), "tasks")
print("Reproducibility check passed.")

In [ ]:
from pathlib import Path

matches = [
    p for p in Path("/kaggle/input").rglob("*")
    if p.is_file()
    and (
        "batch_task" in p.name.lower()
        or "alibaba" in p.name.lower()
    )
]

if matches:
    for p in matches[:30]:
        print(p)
else:
    print("No Alibaba files found. Attach the dataset first.")

In [ ]:
from pathlib import Path
import pandas as pd

alibaba_path = Path(
    "/kaggle/input/datasets/rohitdurbha/"
    "alibaba-cluster-data/batch_task.csv"
)

sample = pd.read_csv(
    alibaba_path,
    header=None,
    nrows=5
)

print("Number of columns:", sample.shape[1])
print(sample.to_string(index=False, header=False))

In [ ]:
columns = [
    "task_name", "instance_num", "job_name", "task_type",
    "status", "start_time", "end_time", "plan_cpu", "plan_mem"
]

alibaba_sample = pd.read_csv(
    alibaba_path,
    header=None,
    names=columns,
    nrows=100_000
)

numeric_columns = [
    "instance_num", "start_time", "end_time",
    "plan_cpu", "plan_mem"
]

for column in numeric_columns:
    alibaba_sample[column] = pd.to_numeric(
        alibaba_sample[column], errors="coerce"
    )

alibaba_sample["recorded_duration_s"] = (
    alibaba_sample["end_time"] - alibaba_sample["start_time"]
)

valid = (
    alibaba_sample["status"].eq("Terminated")
    & alibaba_sample[numeric_columns].notna().all(axis=1)
    & alibaba_sample["start_time"].gt(0)
    & alibaba_sample["recorded_duration_s"].gt(0)
    & alibaba_sample["instance_num"].ge(1)
    & alibaba_sample["plan_cpu"].gt(0)
    & alibaba_sample["plan_mem"].gt(0)
)

clean_sample = alibaba_sample.loc[valid].copy()

print("Rows inspected:", len(alibaba_sample))
print("Valid replay candidates:", len(clean_sample))

print("\nResource and duration ranges:")
print(
    clean_sample[
        ["instance_num", "plan_cpu", "plan_mem",
         "recorded_duration_s"]
    ].describe().round(3).to_string()
)

print("\nStatus counts:")
print(alibaba_sample["status"].value_counts().to_string())

In [ ]:
import json
import random
from pathlib import Path

project = Path("/kaggle/working/something_unique")
(project / "data").mkdir(parents=True, exist_ok=True)

# First integration subset: single-instance tasks from our 100k-row sample.
single = clean_sample.loc[
    clean_sample["instance_num"].eq(1)
].copy()

single = single.sort_values(
    ["start_time", "job_name", "task_name"]
)

# Keep one 150-second interval, including every eligible row in it.
window_start = float(single["start_time"].min())
window_seconds = 150.0

selected = single.loc[
    single["start_time"].ge(window_start)
    & single["start_time"].lt(window_start + window_seconds)
].copy()

rng = random.Random(42)
alibaba_tasks = []

# Explicit modeling conversion, NOT a conversion to GB.
# Memory remains in the trace's normalized units.
MEMORY_SCALE = 1.0

for index, row in enumerate(selected.itertuples(index=False)):
    arrival = float(row.start_time - window_start)
    duration = float(row.recorded_duration_s)

    # Modeled fog origin: 65% Cluster-A, 35% Cluster-B.
    cluster = 0 if rng.random() < 0.65 else 1
    origin = cluster * 3 + rng.randrange(3)

    task = {
        "id": f"A{index + 1}",
        "job_name": str(row.job_name),
        "task_name": str(row.task_name),
        "arrival_time": arrival,
        "origin_node": origin,
        "cpu_request": float(row.plan_cpu) / 100.0,
        "mem_request": float(row.plan_mem) * MEMORY_SCALE,
        "reference_runtime_s": duration,

        # Assumed fields absent from this trace file.
        "data_size_mb": 40.0,
        "urgency_flag": int(rng.random() < 0.2),
        "source_device": origin % 3,
        "deadline": arrival + 2.0 * duration + 30.0
    }

    task["label"] = classifier.classify(task)
    alibaba_tasks.append(task)

assert alibaba_tasks, "No tasks selected."
assert all(t["reference_runtime_s"] > 0 for t in alibaba_tasks)

alibaba_payload = {
    "config": {
        "workload": "Alibaba trace-derived diagnostic subset",
        "source": str(alibaba_path),
        "source_rows_inspected": len(alibaba_sample),
        "selection": "single-instance, terminated, valid records",
        "trace_window_start": window_start,
        "window_seconds": window_seconds,
        "arrival_time_scale": 1.0,
        "memory_scale": MEMORY_SCALE,
        "memory_units": "normalized trace units; not GB",
        "arrival_proxy": "recorded start_time",
        "runtime_proxy": "recorded end_time minus start_time",
        "assumed_fields": [
            "fog origin", "payload size", "urgency",
            "source device", "deadline"
        ],
        "dependencies": "treated as independent tasks",
        "seed": 42
    },
    "tasks": alibaba_tasks
}

output = project / "data/alibaba_replay.json"
output.write_text(json.dumps(alibaba_payload, indent=2))

print("Alibaba replay created.")
print("Tasks selected:", len(alibaba_tasks))
print("Arrival window:", window_seconds, "seconds")
print("Time compression: none")
print("CPU range:",
      min(t["cpu_request"] for t in alibaba_tasks),
      "to",
      max(t["cpu_request"] for t in alibaba_tasks))
print("Memory range:",
      min(t["mem_request"] for t in alibaba_tasks),
      "to",
      max(t["mem_request"] for t in alibaba_tasks))
print("Saved:", output)

In [ ]:
print((project / "agents/escalation.py").read_text())

In [ ]:
try:
    import torch
    print("PyTorch available:", torch.__version__)
    print("GPU available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch is not installed.")

In [ ]:
from pathlib import Path

project = Path("/kaggle/working/something_unique")
file = project / "agents/escalation_ddqn.py"

code = '''
"""PyTorch Dueling Double DQN for escalation.

Creating this agent does not train it or connect it to the simulator.
"""

import random
from collections import deque

import numpy as np
import torch
from torch import nn


ACTIONS = [
    "keep_local",
    "siblings",
    "parent",
    "cloud",
    "hold_in_queue"
]

STATE_DIM = 6
N_ACTIONS = len(ACTIONS)


def make_state(predicted_own_load, best_sibling_load,
               other_cluster_load, urgency_level,
               time_to_deadline_norm, queue_len_norm):
    return np.array([
        predicted_own_load,
        best_sibling_load,
        other_cluster_load,
        urgency_level / 2.0,
        time_to_deadline_norm,
        queue_len_norm
    ], dtype=np.float32)


class DuelingQNet(nn.Module):
    def __init__(self, state_dim=STATE_DIM, n_actions=N_ACTIONS):
        super().__init__()

        self.shared = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU()
        )

        self.value_head = nn.Linear(64, 1)
        self.advantage_head = nn.Linear(64, n_actions)

    def forward(self, state):
        features = self.shared(state)
        value = self.value_head(features)
        advantage = self.advantage_head(features)

        return value + advantage - advantage.mean(
            dim=-1, keepdim=True
        )


class EscalationDDQN:
    def __init__(
        self,
        seed=42,
        learning_rate=1e-3,
        gamma=0.99,
        epsilon=1.0,
        buffer_size=10000,
        target_sync_every=200
    ):
        if target_sync_every < 1:
            raise ValueError("target_sync_every must be positive.")

        self.device = torch.device("cpu")
        self.rng = random.Random(seed)

        # Initialize reproducibly without changing global torch RNG state.
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(seed)
            self.online = DuelingQNet().to(self.device)
            self.target = DuelingQNet().to(self.device)

        self.target.load_state_dict(self.online.state_dict())
        self.target.eval()
        self.target.requires_grad_(False)

        self.optimizer = torch.optim.Adam(
            self.online.parameters(), lr=learning_rate
        )
        self.loss_function = nn.SmoothL1Loss()

        self.gamma = gamma
        self.epsilon = epsilon
        self.buffer = deque(maxlen=buffer_size)
        self.target_sync_every = target_sync_every
        self.training_steps = 0

    def act(self, state, greedy=False, valid_actions=None):
        valid = (
            list(range(N_ACTIONS))
            if valid_actions is None
            else list(valid_actions)
        )

        if not valid or any(a not in range(N_ACTIONS) for a in valid):
            raise ValueError("Supply valid action indices.")

        if not greedy and self.rng.random() < self.epsilon:
            return self.rng.choice(valid)

        state_tensor = torch.as_tensor(
            np.asarray(state, dtype=np.float32),
            device=self.device
        ).reshape(1, STATE_DIM)

        with torch.no_grad():
            values = self.online(state_tensor)[0]
            mask = torch.full_like(values, -torch.inf)
            mask[valid] = values[valid]

        return int(mask.argmax().item())

    def remember(
        self, state, action, reward, next_state, done,
        next_valid_actions=None
    ):
        if action not in range(N_ACTIONS):
            raise ValueError("Invalid action.")

        mask = np.zeros(N_ACTIONS, dtype=bool)

        if next_valid_actions is None:
            mask[:] = True
        else:
            for a in next_valid_actions:
                if a not in range(N_ACTIONS):
                    raise ValueError("Invalid next action.")
                mask[a] = True

        if not done and not mask.any():
            raise ValueError("Nonterminal state needs a valid next action.")

        state = np.asarray(state, dtype=np.float32).reshape(STATE_DIM)
        next_state = np.asarray(
            next_state, dtype=np.float32
        ).reshape(STATE_DIM)

        if (
            not np.isfinite(state).all()
            or not np.isfinite(next_state).all()
            or not np.isfinite(reward)
        ):
            raise ValueError("States and reward must be finite.")

        self.buffer.append((
            state.copy(), int(action), float(reward),
            next_state.copy(), bool(done), mask.copy()
        ))

    def train_step(self, batch_size=64):
        if len(self.buffer) < batch_size:
            return None

        batch = self.rng.sample(list(self.buffer), batch_size)

        states = torch.as_tensor(
            np.stack([b[0] for b in batch]), device=self.device
        )
        actions = torch.tensor(
            [b[1] for b in batch],
            dtype=torch.long, device=self.device
        )
        rewards = torch.tensor(
            [b[2] for b in batch],
            dtype=torch.float32, device=self.device
        )
        next_states = torch.as_tensor(
            np.stack([b[3] for b in batch]), device=self.device
        )
        dones = torch.tensor(
            [b[4] for b in batch],
            dtype=torch.bool, device=self.device
        )
        next_masks = torch.as_tensor(
            np.stack([b[5] for b in batch]),
            dtype=torch.bool, device=self.device
        )

        # Q-values for actions actually taken.
        predicted = self.online(states).gather(
            1, actions[:, None]
        ).squeeze(1)

        with torch.no_grad():
            # Double DQN: online selects; target evaluates.
            online_next = self.online(next_states)
            online_next = online_next.masked_fill(
                ~next_masks, -torch.inf
            )
            next_actions = online_next.argmax(dim=1)

            next_values = self.target(next_states).gather(
                1, next_actions[:, None]
            ).squeeze(1)

            next_values = torch.where(
                dones, torch.zeros_like(next_values), next_values
            )
            targets = rewards + self.gamma * next_values

        loss = self.loss_function(predicted, targets)

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.online.parameters(), 10.0)
        self.optimizer.step()

        self.training_steps += 1

        if self.training_steps % self.target_sync_every == 0:
            self.sync_target()

        return float(loss.item())

    def sync_target(self):
        self.target.load_state_dict(self.online.state_dict())

    def save(self, path):
        """Save model and optimizer; replay memory is not included."""
        torch.save({
            "online": self.online.state_dict(),
            "target": self.target.state_dict(),
            "optimizer": self.optimizer.state_dict(),
            "epsilon": self.epsilon,
            "gamma": self.gamma,
            "training_steps": self.training_steps,
            "target_sync_every": self.target_sync_every,
            "rng_state": self.rng.getstate()
        }, path)

    def load(self, path):
        checkpoint = torch.load(
            path, map_location=self.device, weights_only=True
        )

        self.online.load_state_dict(checkpoint["online"])
        self.target.load_state_dict(checkpoint["target"])
        self.optimizer.load_state_dict(checkpoint["optimizer"])
        self.epsilon = checkpoint["epsilon"]
        self.gamma = checkpoint["gamma"]
        self.training_steps = checkpoint["training_steps"]
        self.target_sync_every = checkpoint["target_sync_every"]
        self.rng.setstate(checkpoint["rng_state"])
        self.target.eval()

        return self
'''

# Check syntax before saving; preserve an existing version.
compile(code, str(file), "exec")

if file.exists():
    import shutil
    backup = file.with_suffix(".py.bak")
    if not backup.exists():
        shutil.copy2(file, backup)

file.parent.mkdir(parents=True, exist_ok=True)
file.write_text(code)

print("Created:", file)
print("Old escalation.py preserved.")
print("Status: implemented; not yet trained or connected.")

In [ ]:
import importlib.util
import torch
import numpy as np

spec = importlib.util.spec_from_file_location(
    "ddqn_check", file
)
ddqn_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ddqn_module)

agent = ddqn_module.EscalationDDQN(seed=42)

state = np.array(
    [0.9, 0.3, 0.5, 1.0, 0.4, 0.6],
    dtype=np.float32
)

with torch.no_grad():
    q_values = agent.online(torch.tensor(state).unsqueeze(0))

assert q_values.shape == (1, 5)

# Artificial transitions only test the learning machinery.
for i in range(64):
    agent.remember(
        state=state,
        action=i % 5,
        reward=0.5,
        next_state=state * 0.9,
        done=False,
        next_valid_actions=[0, 1, 2, 3, 4]
    )

loss = agent.train_step(batch_size=64)

assert loss is not None and np.isfinite(loss)
assert agent.training_steps == 1

print("Network output shape:", tuple(q_values.shape))
print("One training update passed.")
print("Loss:", round(loss, 6))
print("This is a code check, not policy training.")

In [ ]:
from pathlib import Path

project = Path("/kaggle/working/something_unique")

for name in [
    "agents/escalation_ddqn.py",
    "agents/batch_placement.py",
    "models/cost_compute.joblib",
    "models/cost_transfer.joblib",
    "data/alibaba_replay.json",
    "replay_comparison.py"
]:
    path = project / name
    print(f"{'FOUND' if path.exists() else 'MISSING'}: {name}")

print(
    "\nReplay function loaded:",
    "run_replay_comparison" in globals()
)

In [ ]:
from pathlib import Path

project = Path("/kaggle/working/something_unique")
runner_path = project / "replay_comparison.py"

code = r'''
"""Fixed-policy replay comparison.

Parents coordinate; six leaves execute.
DDQN and forecasting are not connected yet.
Resource reservations include transfer, waiting and execution.
"""

import copy
import heapq
import itertools
from time import perf_counter

import numpy as np

from agents.batch_placement import solve_batch
from agents.local_scheduling import LocalSchedulingAgent


def run_replay_comparison(tasks, policy, cost_agent, round_seconds=5.0):
    if policy not in ("flat", "hierarchical"):
        raise ValueError("Policy must be flat or hierarchical.")
    if round_seconds <= 0:
        raise ValueError("round_seconds must be positive.")

    tasks = copy.deepcopy(tasks)
    if not tasks:
        raise ValueError("Replay is empty.")

    if len({t["id"] for t in tasks}) != len(tasks):
        raise ValueError("Task IDs must be unique.")

    for task in tasks:
        values = [
            task["arrival_time"], task["deadline"],
            task["cpu_request"], task["mem_request"],
            task["data_size_mb"], task["reference_runtime_s"]
        ]
        if not np.isfinite(values).all():
            raise ValueError("Task fields must be finite.")
        if (
            task["arrival_time"] < 0
            or task["cpu_request"] <= 0
            or task["mem_request"] <= 0
            or task["data_size_mb"] < 0
            or task["reference_runtime_s"] <= 0
            or task["origin_node"] not in range(6)
        ):
            raise ValueError("Invalid task fields.")

    cpu_cap = np.array([8, 10, 12, 8, 10, 12], dtype=float)
    mem_cap = cpu_cap * 0.8
    bandwidth = np.array([40, 60, 100, 40, 60, 100], dtype=float)

    cpu_reserved = np.zeros(6)
    mem_reserved = np.zeros(6)
    busy = [False] * 6
    active = [None] * 6
    transit = [{} for _ in range(6)]
    queues = [LocalSchedulingAgent() for _ in range(6)]

    events = []
    sequence = itertools.count()
    pending = []
    completed = []
    scheduling_times = []
    mip_times = []

    def push(time, kind, payload):
        heapq.heappush(
            events, (float(time), next(sequence), kind, payload)
        )

    for task in tasks:
        push(task["arrival_time"], "arrival", task)

    last_arrival = max(t["arrival_time"] for t in tasks)
    push(0.0, "round", None)

    def start_worker(node, now):
        if busy[node] or not len(queues[node]):
            return
        queues[node].rescore_all(now)
        task = queues[node].pop_next()
        task["start_time"] = now
        busy[node] = True
        active[node] = task

        # Recorded runtime is execution ground truth only.
        push(
            now + task["actual_compute_s"],
            "finish", (node, task)
        )

    def record(task, now):
        task["finish_time"] = now
        task["response_s"] = now - task["arrival_time"]
        task["deadline_met"] = now <= task["deadline"]
        task["batch_wait_s"] = (
            task["placement_time"] - task["arrival_time"]
        )
        task["queue_wait_s"] = (
            task["start_time"] - task["ready_time"]
        )
        completed.append(task)

    while events:
        now, _, kind, payload = heapq.heappop(events)

        if kind == "arrival":
            pending.append(payload)

        elif kind == "ready":
            node, task = payload
            transit[node].pop(task["id"])
            task["ready_time"] = now
            queues[node].enqueue(task, now)
            start_worker(node, now)

        elif kind == "finish":
            node, task = payload
            cpu_reserved[node] -= task["cpu_request"]
            mem_reserved[node] -= task["mem_request"]
            busy[node] = False
            active[node] = None
            record(task, now)
            start_worker(node, now)

        elif kind == "cloud_finish":
            record(payload, now)

        elif kind == "round":
            if pending:
                batch = pending[:]
                pending.clear()
                begin = perf_counter()
                m = len(batch)

                cpu = np.array([t["cpu_request"] for t in batch])
                mem = np.array([t["mem_request"] for t in batch])
                origins = np.array([t["origin_node"] for t in batch])
                load = np.minimum(
                    1.0,
                    np.maximum(cpu_reserved / cpu_cap,
                               mem_reserved / mem_cap)
                )

                features = np.column_stack([
                    np.repeat(cpu, 6), np.tile(load, m)
                ])
                predicted = np.maximum(
                    0.2, cost_agent.compute_model.predict(features)
                ).reshape(m, 6)

                transfer = np.zeros((m, 6))
                for i, task in enumerate(batch):
                    for node in range(6):
                        if node != origins[i]:
                            cross = node // 3 != origins[i] // 3
                            hops = 3 if cross else 1
                            link = min(
                                bandwidth[origins[i]], bandwidth[node]
                            )
                            transfer[i, node] = (
                                8.0 * task["data_size_mb"] / link
                                * (1.0 + 0.5 * hops)
                            )

                # Estimate waiting using predictions, not future runtimes.
                waiting = np.zeros(6)
                for node in range(6):
                    if active[node] is not None:
                        task = active[node]
                        waiting[node] += max(
                            0.0,
                            task["predicted_compute_s"]
                            - (now - task["start_time"])
                        )

                    waiting[node] += sum(
                        t["predicted_compute_s"]
                        for t in queues[node].peek_all()
                    )

                    for task in transit[node].values():
                        waiting[node] += (
                            max(0.0, task["expected_ready_time"] - now)
                            + task["predicted_compute_s"]
                        )

                costs = predicted + transfer
                slack = np.array([t["deadline"] - now for t in batch])
                feasible = costs + waiting[None, :] <= slack[:, None]

                # This pairwise feasibility estimate does not guarantee
                # joint queue deadlines. Measure actual completion outcomes.
                free_cpu = np.maximum(0.0, cpu_cap - cpu_reserved)
                free_mem = np.maximum(0.0, mem_cap - mem_reserved)
                choices = np.full(m, -1, dtype=int)

                def place(ids, nodes):
                    if not len(ids):
                        return
                    nodes = np.asarray(nodes)
                    mip_begin = perf_counter()
                    result = solve_batch(
                        costs[np.ix_(ids, nodes)],
                        cpu[ids], mem[ids],
                        free_cpu[nodes], free_mem[nodes],
                        feasible[np.ix_(ids, nodes)]
                    )
                    mip_times.append(perf_counter() - mip_begin)

                    for i, local_node in zip(
                        ids, result["destinations"]
                    ):
                        if local_node >= 0:
                            node = int(nodes[local_node])
                            choices[i] = node
                            free_cpu[node] -= cpu[i]
                            free_mem[node] -= mem[i]

                if policy == "flat":
                    place(np.arange(m), np.arange(6))
                else:
                    for group in range(2):
                        place(
                            np.flatnonzero(origins // 3 == group),
                            np.arange(3 * group, 3 * group + 3)
                        )
                    for group in range(2):
                        ids = np.flatnonzero(
                            (origins // 3 == group) & (choices < 0)
                        )
                        other = 1 - group
                        place(
                            ids, np.arange(3 * other, 3 * other + 3)
                        )

                scheduling_times.append(perf_counter() - begin)

                for i, task in enumerate(batch):
                    task["placement_time"] = now
                    node = choices[i]

                    if node < 0:
                        task["destination"] = "cloud"
                        task["route"] = "cloud"
                        task["transfer_s"] = (
                            4.0 + 8.0 * task["data_size_mb"] / 20.0
                        )
                        # Explicit modeled cloud speedup, not measured.
                        task["actual_compute_s"] = (
                            0.3 * task["reference_runtime_s"]
                        )
                        task["ready_time"] = now + task["transfer_s"]
                        task["start_time"] = task["ready_time"]
                        push(
                            task["start_time"] + task["actual_compute_s"],
                            "cloud_finish", task
                        )
                    else:
                        task["destination"] = f"F{node + 1}"
                        task["route"] = (
                            "local" if node == origins[i]
                            else "sibling"
                            if node // 3 == origins[i] // 3
                            else "cross_cluster"
                        )
                        task["transfer_s"] = float(transfer[i, node])
                        task["predicted_compute_s"] = float(predicted[i, node])

                        # Same reference runtime on each fog node.
                        # This is a replay assumption, not fog calibration.
                        task["actual_compute_s"] = (
                            task["reference_runtime_s"]
                        )
                        task["expected_ready_time"] = (
                            now + task["transfer_s"]
                        )
                        cpu_reserved[node] += cpu[i]
                        mem_reserved[node] += mem[i]
                        transit[node][task["id"]] = task
                        push(
                            task["expected_ready_time"],
                            "ready", (node, task)
                        )

                assert np.all(cpu_reserved <= cpu_cap + 1e-6)
                assert np.all(mem_reserved <= mem_cap + 1e-6)

            if now < last_arrival:
                push(now + round_seconds, "round", None)

    assert len(completed) == len(tasks)
    assert len({t["id"] for t in completed}) == len(tasks)
    assert np.allclose(cpu_reserved, 0)
    assert np.allclose(mem_reserved, 0)

    return {
        "tasks": completed,
        "scheduling_times": scheduling_times,
        "mip_times": mip_times,
        "solver_calls": len(mip_times)
    }
'''

compile(code, str(runner_path), "exec")
runner_path.write_text(code)

print("Saved:", runner_path)
print("Alibaba reference runtimes connected to execution.")
print("DDQN training environment: still to be connected.")

In [ ]:
import sys
import json
import importlib.util
import pandas as pd

if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from agents.cost_prediction import CostPredictionAgent

cost_agent = CostPredictionAgent()
compute_mse, transfer_mse = cost_agent.train(
    pd.read_csv(project / "data/cost_data.csv")
)
cost_agent.save(
    str(project / "models/cost_compute.joblib"),
    str(project / "models/cost_transfer.joblib")
)

spec = importlib.util.spec_from_file_location(
    "replay_runner", runner_path
)
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)

alibaba_tasks = json.loads(
    (project / "data/alibaba_replay.json").read_text()
)["tasks"]

alibaba_runs = {}

for policy in ["flat", "hierarchical"]:
    result = runner.run_replay_comparison(
        alibaba_tasks, policy, cost_agent
    )
    alibaba_runs[policy] = result
    frame = pd.DataFrame(result["tasks"])

    assert len(frame) == len(alibaba_tasks)
    assert frame["id"].is_unique

    parts = (
        frame["batch_wait_s"] + frame["transfer_s"]
        + frame["queue_wait_s"] + frame["actual_compute_s"]
    )
    import numpy as np
    assert np.allclose(frame["response_s"], parts)

    fog = frame["route"] != "cloud"
    assert np.allclose(
        frame.loc[fog, "actual_compute_s"],
        frame.loc[fog, "reference_runtime_s"]
    )

    print(
        policy,
        "| completed:", len(frame),
        "| deadlines met:", int(frame["deadline_met"].sum()),
        "| cloud:", int((frame["route"] == "cloud").sum())
    )

print("Alibaba execution and timing checks passed.")

In [ ]:
import pandas as pd

columns = [
    "task_name", "instance_num", "job_name", "task_type",
    "status", "start_time", "end_time", "plan_cpu", "plan_mem"
]

# Keep the same starting point as the five-task replay.
window_start = float(alibaba_payload["config"]["trace_window_start"])
window_end = window_start + 3600.0

parts = []
scanned = 0

for chunk in pd.read_csv(
    alibaba_path,
    header=None,
    names=columns,
    chunksize=250_000
):
    scanned += len(chunk)

    for column in [
        "instance_num", "start_time", "end_time",
        "plan_cpu", "plan_mem"
    ]:
        chunk[column] = pd.to_numeric(
            chunk[column], errors="coerce"
        )

    duration = chunk["end_time"] - chunk["start_time"]

    keep = (
        chunk["status"].eq("Terminated")
        & chunk["instance_num"].eq(1)
        & chunk["start_time"].ge(window_start)
        & chunk["start_time"].lt(window_end)
        & duration.gt(0)
        & chunk["plan_cpu"].gt(0)
        & chunk["plan_mem"].gt(0)
    )

    selected_chunk = chunk.loc[keep].copy()
    selected_chunk["recorded_duration_s"] = duration.loc[keep]

    if len(selected_chunk):
        parts.append(selected_chunk)

    if scanned % 1_000_000 == 0:
        print(f"Scanned {scanned:,} rows...", flush=True)

if not parts:
    raise RuntimeError("No eligible tasks found in this window.")

hour_records = (
    pd.concat(parts, ignore_index=True)
    .sort_values(["start_time", "job_name", "task_name"])
    .reset_index(drop=True)
)

output = project / "data/alibaba_single_instance_1hour.csv"
hour_records.to_csv(output, index=False)

print("\nScan finished.")
print("Rows scanned:", scanned)
print("Eligible single-instance tasks:", len(hour_records))
print("Window length: 3,600 original seconds")
print("No time compression or task-count truncation.")
print("Saved:", output)

In [ ]:
import numpy as np
import pandas as pd
import joblib
import sklearn

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

records = pd.read_csv(
    project / "data/alibaba_single_instance_1hour.csv"
)

# Predictor inputs available before execution.
records["cpu_request"] = records["plan_cpu"] / 100.0
records["mem_request"] = records["plan_mem"]

features = ["cpu_request", "mem_request"]
target = "recorded_duration_s"

numeric = records[
    features + ["start_time", "end_time", target]
].to_numpy(dtype=float)

records = records.loc[
    np.isfinite(numeric).all(axis=1)
    & records[target].gt(0)
].copy()

# Split by timestamp, not random overlapping records.
cutoff = float(records["start_time"].quantile(0.70))

# Training outcomes must already be known at the cutoff.
train = records.loc[
    records["start_time"].lt(cutoff)
    & records["end_time"].le(cutoff)
].copy()

test = records.loc[
    records["start_time"].ge(cutoff)
].copy()

# Keep evaluation jobs separate from training jobs.
training_jobs = set(train["job_name"])
test = test.loc[
    ~test["job_name"].isin(training_jobs)
].copy()

assert len(train) > 0 and len(test) > 0
assert set(train["job_name"]).isdisjoint(set(test["job_name"]))

runtime_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=2
)

runtime_rf.fit(train[features], train[target])

predicted = np.maximum(0.001, runtime_rf.predict(test[features]))
actual = test[target].to_numpy()

# Simple baseline: always predict the training-set median duration.
baseline = np.full(
    len(test), float(train[target].median())
)

comparison = pd.DataFrame([
    {
        "model": "Training-median baseline",
        "MAE_seconds": mean_absolute_error(actual, baseline),
        "RMSE_seconds": np.sqrt(
            mean_squared_error(actual, baseline)
        )
    },
    {
        "model": "Alibaba runtime RF",
        "MAE_seconds": mean_absolute_error(actual, predicted),
        "RMSE_seconds": np.sqrt(
            mean_squared_error(actual, predicted)
        )
    }
])

model_path = project / "models/alibaba_runtime_rf.joblib"

joblib.dump({
    "model": runtime_rf,
    "features": features,
    "target": target,
    "cutoff": cutoff,
    "sklearn_version": sklearn.__version__,
    "training_jobs": sorted(training_jobs),
    "memory_scale": 1.0
}, model_path)

results_dir = project / "results"
results_dir.mkdir(exist_ok=True)

comparison.to_csv(
    results_dir / "alibaba_runtime_evaluation.csv",
    index=False
)

# Preserve held-out records for subsequent replay.
test.to_csv(
    project / "data/alibaba_runtime_heldout.csv",
    index=False
)

print("Training records:", len(train))
print("Held-out records:", len(test))
print("Train/test job overlap: 0")
print("\n", comparison.round(3).to_string(index=False))
print("\nSaved separate model:", model_path)

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import importlib.util

project = Path("/kaggle/working/something_unique")
source_path = project / "replay_comparison.py"
destination = project / "replay_alibaba.py"

code = source_path.read_text()

replacements = [
    (
        "def run_replay_comparison(tasks, policy, cost_agent, round_seconds=5.0):",
        "def run_replay_comparison(tasks, policy, runtime_model, round_seconds=5.0):"
    ),
    (
        "np.repeat(cpu, 6), np.tile(load, m)",
        "np.repeat(cpu, 6), np.repeat(mem, 6)"
    ),
    (
        "cost_agent.compute_model.predict(features)",
        """runtime_model.predict(pd.DataFrame(
                        features,
                        columns=["cpu_request", "mem_request"]
                    ))"""
    )
]

for old, new in replacements:
    if code.count(old) != 1:
        raise RuntimeError(
            "Runner differs from expected. No file changed."
        )
    code = code.replace(old, new, 1)

code = "import pandas as pd\n" + code

compile(code, str(destination), "exec")
destination.write_text(code)

bundle = joblib.load(
    project / "models/alibaba_runtime_rf.joblib"
)
runtime_model = bundle["model"]

spec = importlib.util.spec_from_file_location(
    "alibaba_runner", destination
)
alibaba_runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(alibaba_runner)

# Diagnostic only: use 20 training records, not held-out evaluation data.
check_records = train.sort_values(
    ["start_time", "job_name", "task_name"]
).head(20)

start = float(check_records["start_time"].min())
check_tasks = []

for i, row in enumerate(check_records.itertuples(index=False)):
    arrival = float(row.start_time - start)
    duration = float(row.recorded_duration_s)

    check_tasks.append({
        "id": f"check_{i}",
        "arrival_time": arrival,
        "origin_node": i % 6,
        "cpu_request": float(row.cpu_request),
        "mem_request": float(row.mem_request),
        "reference_runtime_s": duration,
        "data_size_mb": 40.0,
        "deadline": arrival + 2 * duration + 30,
        "label": "can_wait"
    })

for policy in ["flat", "hierarchical"]:
    result = alibaba_runner.run_replay_comparison(
        check_tasks, policy, runtime_model
    )
    frame = pd.DataFrame(result["tasks"])

    assert len(frame) == len(check_tasks)
    assert frame["id"].is_unique

    fog = frame["route"] != "cloud"
    assert np.allclose(
        frame.loc[fog, "actual_compute_s"],
        frame.loc[fog, "reference_runtime_s"]
    )

    print(
        policy,
        "| completed:", len(frame),
        "| deadlines met:", int(frame["deadline_met"].sum())
    )

print("Alibaba RF connected; recorded runtimes retained for execution.")
print("Saved:", destination)

In [ ]:
from pathlib import Path
import shutil

project = Path("/kaggle/working/something_unique")

required = [
    "agents/escalation_ddqn.py",
    "agents/batch_placement.py",
    "replay_comparison.py",
    "replay_alibaba.py",
    "models/alibaba_runtime_rf.joblib",
    "data/alibaba_single_instance_1hour.csv"
]

for name in required:
    assert (project / name).exists(), f"Missing: {name}"

archive = shutil.make_archive(
    "/kaggle/working/fog_scheduler_alibaba_checkpoint",
    "zip",
    root_dir=project.parent,
    base_dir=project.name
)

print("Checkpoint created:", archive)
print("Size:", round(Path(archive).stat().st_size / 1_000_000, 2), "MB")

In [ ]:
from pathlib import Path
from IPython.display import display, FileLink

checkpoint = Path(
    "/kaggle/working/fog_scheduler_alibaba_checkpoint.zip"
)

if checkpoint.exists():
    print("Checkpoint exists. Click below to download a backup:")
    display(FileLink(str(checkpoint)))
else:
    print("Checkpoint is missing from this session.")

In [ ]:
from pathlib import Path
import pandas as pd
import joblib
import random
import json

project = Path("/kaggle/working/something_unique")

records = pd.read_csv(
    project / "data/alibaba_single_instance_1hour.csv"
)
bundle = joblib.load(
    project / "models/alibaba_runtime_rf.joblib"
)

# Restrict episodes to the compute model's training records.
eligible = records.loc[
    records["start_time"].lt(bundle["cutoff"])
    & records["end_time"].le(bundle["cutoff"])
    & records["job_name"].isin(bundle["training_jobs"])
].copy()

episode_seconds = 60.0
first_time = float(eligible["start_time"].min())
episodes = []

for episode_id in range(5):
    start = first_time + episode_id * episode_seconds
    end = start + episode_seconds

    subset = eligible.loc[
        eligible["start_time"].ge(start)
        & eligible["start_time"].lt(end)
    ].sort_values(["start_time", "job_name", "task_name"])

    rng = random.Random(42 + episode_id)
    episode_tasks = []

    for i, row in enumerate(subset.itertuples(index=False)):
        arrival = float(row.start_time - start)
        duration = float(row.recorded_duration_s)
        cluster = 0 if rng.random() < 0.65 else 1

        task = {
            "id": f"E{episode_id}_T{i}",
            "job_name": str(row.job_name),
            "arrival_time": arrival,
            "origin_node": cluster * 3 + rng.randrange(3),
            "cpu_request": float(row.plan_cpu) / 100.0,
            "mem_request": float(row.plan_mem),
            "reference_runtime_s": duration,
            "data_size_mb": 40.0,
            "urgency_flag": int(rng.random() < 0.2),
            "source_device": rng.randrange(3),
            "deadline": arrival + 2 * duration + 30
        }

        task["label"] = classifier.classify(task)
        episode_tasks.append(task)

    if episode_tasks:
        episodes.append({
            "episode_id": episode_id,
            "trace_start": start,
            "arrival_window_seconds": episode_seconds,
            "tasks": episode_tasks
        })

assert episodes, "No nonempty training episodes found."

output = project / "data/ddqn_training_episodes.json"
output.write_text(json.dumps({
    "purpose": "DDQN training only",
    "time_compression": 1.0,
    "assumptions": {
        "tasks": "single-instance records treated independently",
        "arrival": "recorded start time used as arrival proxy",
        "deadline": "arrival + 2 * recorded duration + 30 seconds",
        "payload_MB": 40.0,
        "origins_and_urgency": "seeded synthetic assignments",
        "memory": "normalized trace units"
    },
    "episodes": episodes
}, indent=2))

for episode in episodes:
    print(
        f"Episode {episode['episode_id']}: "
        f"{len(episode['tasks'])} tasks"
    )

print("Saved:", output)
print("Original arrival spacing and recorded runtimes preserved.")
print("No DDQN training has run yet.")

In [ ]:
from pathlib import Path

project = Path("/kaggle/working/something_unique")
env_path = project / "ddqn_environment.py"

code = r'''
"""Sequential escalation environment using trace-derived execution.

Controllers coordinate; six fog leaves execute.
Current demand is a temporary proxy for forecast demand.
Network delays and cloud speedup are simulation assumptions.
"""

import copy
import heapq
import itertools

import numpy as np
import pandas as pd

from agents.batch_placement import solve_batch
from agents.local_scheduling import LocalSchedulingAgent


class FogEscalationEnv:
    def __init__(self, runtime_model, max_holds=3, hold_seconds=1.0):
        if max_holds < 0 or hold_seconds <= 0:
            raise ValueError("Invalid hold settings.")

        self.model = runtime_model
        self.max_holds = max_holds
        self.hold_seconds = hold_seconds
        self.cpu_cap = np.array([8, 10, 12, 8, 10, 12], float)
        self.mem_cap = self.cpu_cap * 0.8
        self.bandwidth = np.array([40, 60, 100, 40, 60, 100], float)

    def reset(self, tasks):
        self.tasks = copy.deepcopy(tasks)
        if not self.tasks:
            raise ValueError("Episode is empty.")
        if len({t["id"] for t in self.tasks}) != len(self.tasks):
            raise ValueError("Task IDs must be unique.")

        self.now = 0.0
        self.events = []
        self.seq = itertools.count()
        self.cpu = np.zeros(6)
        self.mem = np.zeros(6)
        self.queues = [LocalSchedulingAgent() for _ in range(6)]
        self.active = [None] * 6
        self.transit = [{} for _ in range(6)]
        self.completed = []
        self.current = None
        self.done = False
        self.reward = 0.0

        for task in self.tasks:
            if (
                task["origin_node"] not in range(6)
                or task["arrival_time"] < 0
                or task["cpu_request"] <= 0
                or task["mem_request"] <= 0
                or task["reference_runtime_s"] <= 0
            ):
                raise ValueError("Invalid task.")
            task["holds"] = 0
            self._push(task["arrival_time"], "decision", task)

        self._advance()
        return self._state()

    def _push(self, time, kind, payload):
        # Process resource/completion events before decisions at equal times.
        priority = 1 if kind == "decision" else 0
        heapq.heappush(
            self.events,
            (float(time), priority, next(self.seq), kind, payload)
        )

    def _load(self):
        return np.minimum(
            1.0, np.maximum(self.cpu / self.cpu_cap,
                            self.mem / self.mem_cap)
        )

    def _start_worker(self, node):
        if self.active[node] is not None or not len(self.queues[node]):
            return
        self.queues[node].rescore_all(self.now)
        task = self.queues[node].pop_next()
        task["start_time"] = self.now
        self.active[node] = task
        self._push(
            self.now + task["reference_runtime_s"],
            "finish", (node, task)
        )

    def _record(self, task):
        task["finish_time"] = self.now
        task["response_s"] = self.now - task["arrival_time"]
        task["deadline_met"] = self.now <= task["deadline"]

        weight = {
            "urgent": 3.0, "heavy_compute": 1.5, "can_wait": 1.0
        }.get(task["label"], 1.0)

        # Reward is based on actual simulated completion.
        self.reward += weight if task["deadline_met"] else -weight
        self.completed.append(task)

    def _advance(self):
        self.current = None

        while self.events:
            self.now, _, _, kind, payload = heapq.heappop(self.events)

            if kind == "decision":
                self.current = payload
                self._prepare_candidates()
                return

            if kind == "ready":
                node, task = payload
                self.transit[node].pop(task["id"])
                task["ready_time"] = self.now
                self.queues[node].enqueue(task, self.now)
                self._start_worker(node)

            elif kind == "finish":
                node, task = payload
                self.cpu[node] -= task["cpu_request"]
                self.mem[node] -= task["mem_request"]
                self.active[node] = None
                self._record(task)
                self._start_worker(node)

            elif kind == "cloud_finish":
                self._record(payload)

        self.done = True
        assert len(self.completed) == len(self.tasks)
        assert len({t["id"] for t in self.completed}) == len(self.tasks)
        assert np.allclose(self.cpu, 0)
        assert np.allclose(self.mem, 0)

    def _prepare_candidates(self):
        task = self.current
        origin = task["origin_node"]

        features = pd.DataFrame({
            "cpu_request": [task["cpu_request"]] * 6,
            "mem_request": [task["mem_request"]] * 6
        })
        self.predicted = np.maximum(
            0.2, self.model.predict(features)
        )
        self.transfer = np.zeros(6)

        for node in range(6):
            if node != origin:
                hops = 1 if node // 3 == origin // 3 else 3
                link = min(self.bandwidth[origin], self.bandwidth[node])
                self.transfer[node] = (
                    8 * task["data_size_mb"] / link
                    * (1 + 0.5 * hops)
                )

        # Estimate existing waiting from predictions, not future runtimes.
        waiting = np.zeros(6)

        for node in range(6):
            active = self.active[node]
            if active is not None:
                waiting[node] += max(
                    0.0,
                    active["predicted_compute_s"]
                    - (self.now - active["start_time"])
                )

            waiting[node] += sum(
                t["predicted_compute_s"]
                for t in self.queues[node].peek_all()
            )

            for t in self.transit[node].values():
                waiting[node] += (
                    max(0.0, t["expected_ready_time"] - self.now)
                    + t["predicted_compute_s"]
                )

        self.feasible = (
            (self.cpu + task["cpu_request"] <= self.cpu_cap + 1e-9)
            & (self.mem + task["mem_request"] <= self.mem_cap + 1e-9)
            & (
                self.now + waiting + self.transfer + self.predicted
                <= task["deadline"]
            )
        )

    def valid_actions(self):
        if self.done:
            return []

        origin = self.current["origin_node"]
        siblings = [
            j for j in range(6)
            if j // 3 == origin // 3 and j != origin
        ]
        peers = [
            j for j in range(6) if j // 3 != origin // 3
        ]

        valid = [3]  # Cloud always accepts; meeting deadline is not guaranteed.

        if self.feasible[origin]:
            valid.append(0)
        if self.feasible[siblings].any():
            valid.append(1)
        if self.feasible[peers].any():
            valid.append(2)
        if (
            self.current["holds"] < self.max_holds
            and self.now + self.hold_seconds < self.current["deadline"]
        ):
            valid.append(4)

        return sorted(valid)

    def _state(self):
        if self.done:
            return np.zeros(6, dtype=np.float32)

        task = self.current
        origin = task["origin_node"]
        load = self._load()

        siblings = [
            j for j in range(6)
            if j // 3 == origin // 3 and j != origin
        ]
        peers = [
            j for j in range(6) if j // 3 != origin // 3
        ]
        urgency = {
            "can_wait": 0.0, "heavy_compute": 0.5, "urgent": 1.0
        }.get(task["label"], 0.0)

        return np.array([
            load[origin],
            min(load[siblings]),
            min(load[peers]),
            urgency,
            np.clip((task["deadline"] - self.now) / 300.0, 0, 1),
            min(1.0, len(self.queues[origin]) / 20.0)
        ], dtype=np.float32)

    def step(self, action):
        if self.done:
            raise RuntimeError("Reset before stepping a finished episode.")
        if action not in self.valid_actions():
            raise ValueError("Action is masked as infeasible.")

        self.reward = 0.0
        task = self.current
        origin = task["origin_node"]

        if action == 4:
            task["holds"] += 1
            self.reward -= 0.01
            self._push(
                self.now + self.hold_seconds, "decision", task
            )

        elif action == 3:
            task["route"] = "cloud"
            task["destination"] = "cloud"
            task["placement_time"] = self.now
            task["transfer_s"] = (
                4 + 8 * task["data_size_mb"] / 20.0
            )
            task["start_time"] = self.now + task["transfer_s"]

            # Assumed cloud speedup; trace does not measure this.
            self._push(
                task["start_time"]
                + 0.3 * task["reference_runtime_s"],
                "cloud_finish", task
            )
            self.reward -= 0.2 + task["transfer_s"] / 100.0

        else:
            if action == 0:
                node = origin
            else:
                candidates = [
                    j for j in range(6)
                    if (
                        j // 3 == origin // 3 and j != origin
                        if action == 1
                        else j // 3 != origin // 3
                    )
                ]

                result = solve_batch(
                    costs=(self.predicted + self.transfer)[
                        candidates
                    ][None, :],
                    cpu=[task["cpu_request"]],
                    memory=[task["mem_request"]],
                    free_cpu=np.maximum(
                        0, self.cpu_cap[candidates] - self.cpu[candidates]
                    ),
                    free_memory=np.maximum(
                        0, self.mem_cap[candidates] - self.mem[candidates]
                    ),
                    feasible=self.feasible[candidates][None, :]
                )
                chosen = result["destinations"][0]
                assert chosen >= 0
                node = candidates[chosen]

            task["route"] = (
                "local" if node == origin
                else "sibling" if node // 3 == origin // 3
                else "cross_cluster"
            )
            task["destination"] = f"F{node + 1}"
            task["placement_time"] = self.now
            task["predicted_compute_s"] = float(self.predicted[node])
            task["transfer_s"] = float(self.transfer[node])
            task["expected_ready_time"] = self.now + task["transfer_s"]

            self.cpu[node] += task["cpu_request"]
            self.mem[node] += task["mem_request"]
            self.transit[node][task["id"]] = task

            self._push(
                task["expected_ready_time"], "ready", (node, task)
            )
            self.reward -= task["transfer_s"] / 100.0

        assert np.all(self.cpu <= self.cpu_cap + 1e-6)
        assert np.all(self.mem <= self.mem_cap + 1e-6)

        # Advance shared simulation until another decision is needed.
        # On the final decision, drain all remaining execution events.
        self._advance()

        return self._state(), float(self.reward), self.done, {
            "time": self.now,
            "completed": len(self.completed),
            "valid_actions": self.valid_actions()
        }
'''

compile(code, str(env_path), "exec")
env_path.write_text(code)

print("Created:", env_path)
print("Environment implemented; not yet validated or used for training.")

In [ ]:
import importlib.util
import json
import joblib

spec = importlib.util.spec_from_file_location("fog_env_check", env_path)
env_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(env_module)

bundle = joblib.load(project / "models/alibaba_runtime_rf.joblib")
episodes = json.loads(
    (project / "data/ddqn_training_episodes.json").read_text()
)["episodes"]

env = env_module.FogEscalationEnv(bundle["model"])
state = env.reset(episodes[0]["tasks"])

steps = 0
nonterminal = 0
total_reward = 0.0
done = False

while not done:
    valid = env.valid_actions()

    # Fixed test policy; this is NOT DDQN training.
    action = next(a for a in [0, 1, 2, 3] if a in valid)
    state, reward, done, info = env.step(action)

    steps += 1
    nonterminal += int(not done)
    total_reward += reward

    assert steps <= 4 * len(episodes[0]["tasks"])

print("Decisions:", steps)
print("Nonterminal transitions:", nonterminal)
print("Completed tasks:", len(env.completed))
print("Deadline successes:", sum(t["deadline_met"] for t in env.completed))
print("Total reward:", round(total_reward, 3))
print("Episode finished:", done)
print("Task and resource accounting checks passed.")

In [ ]:
import sys
import json
import importlib.util
import numpy as np
import pandas as pd
import torch
from pathlib import Path

project = Path("/kaggle/working/something_unique")

if str(project) not in sys.path:
    sys.path.insert(0, str(project))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

ddqn_module = load_module(
    "ddqn_training_agent",
    project / "agents/escalation_ddqn.py"
)
environment_module = load_module(
    "ddqn_training_environment",
    project / "ddqn_environment.py"
)

torch.set_num_threads(2)

episodes = json.loads(
    (project / "data/ddqn_training_episodes.json").read_text()
)["episodes"]

# Start a fresh agent—not the agent from the artificial code check.
agent = ddqn_module.EscalationDDQN(
    seed=42,
    epsilon=1.0,
    learning_rate=0.0003,
    target_sync_every=200
)

env = environment_module.FogEscalationEnv(
    runtime_model=bundle["model"]
)

checkpoint_dir = project / "models"
results_dir = project / "results"
checkpoint_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

training_log = []
total_decisions = 0
passes = 3

for training_pass in range(passes):
    # Reproducible variation in episode order.
    order = np.random.default_rng(
        42 + training_pass
    ).permutation(len(episodes))

    for episode_index in order:
        episode = episodes[int(episode_index)]
        state = env.reset(episode["tasks"])

        done = False
        episode_reward = 0.0
        losses = []
        decisions = 0
        action_counts = [0] * 5

        while not done:
            action = agent.act(
                state,
                valid_actions=env.valid_actions()
            )

            next_state, reward, done, info = env.step(action)

            agent.remember(
                state=state,
                action=action,
                reward=reward,
                next_state=next_state,
                done=done,
                next_valid_actions=info["valid_actions"]
            )

            loss = agent.train_step(batch_size=64)
            if loss is not None:
                assert np.isfinite(loss), "Nonfinite training loss."
                losses.append(loss)

            state = next_state
            episode_reward += reward
            decisions += 1
            total_decisions += 1
            action_counts[action] += 1

            # Gradually reduce random exploration.
            agent.epsilon = max(
                0.05, agent.epsilon * 0.9995
            )

            assert decisions <= 4 * len(episode["tasks"])

        successes = sum(
            t["deadline_met"] for t in env.completed
        )

        row = {
            "pass": training_pass + 1,
            "episode": episode["episode_id"],
            "tasks": len(env.completed),
            "decisions": decisions,
            "reward": episode_reward,
            "success_pct": 100 * successes / len(env.completed),
            "mean_loss": float(np.mean(losses)) if losses else None,
            "epsilon": agent.epsilon,
            "training_steps": agent.training_steps
        }

        for index, name in enumerate(ddqn_module.ACTIONS):
            row[name] = action_counts[index]

        training_log.append(row)

        # Save after every episode.
        agent.save(
            checkpoint_dir / "escalation_ddqn_latest.pt"
        )
        pd.DataFrame(training_log).to_csv(
            results_dir / "ddqn_training_log.csv",
            index=False
        )

        print(
            f"Pass {training_pass + 1}/{passes}"
            f" | episode {episode['episode_id']}"
            f" | decisions {decisions}"
            f" | success {row['success_pct']:.1f}%"
            f" | reward {episode_reward:.2f}"
            f" | epsilon {agent.epsilon:.3f}",
            flush=True
        )

print("\nInitial training finished.")
print("Total decisions:", total_decisions)
print("Gradient updates:", agent.training_steps)
print("Saved: models/escalation_ddqn_latest.pt")
print("Saved: results/ddqn_training_log.csv")

In [ ]:
import json
import random
import joblib
import importlib.util
import pandas as pd
from pathlib import Path

project = Path("/kaggle/working/something_unique")

def import_file(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

ddqn_module = import_file(
    "ddqn_evaluation",
    project / "agents/escalation_ddqn.py"
)
env_module = import_file(
    "env_evaluation",
    project / "ddqn_environment.py"
)

bundle = joblib.load(project / "models/alibaba_runtime_rf.joblib")

# New agent object verifies the saved checkpoint can be loaded.
trained_agent = ddqn_module.EscalationDDQN(seed=42).load(
    project / "models/escalation_ddqn_latest.pt"
)

heldout = pd.read_csv(
    project / "data/alibaba_runtime_heldout.csv"
)

assert set(heldout["job_name"]).isdisjoint(
    set(bundle["training_jobs"])
)

# Fixed first 60 seconds of held-out arrivals; no task-count truncation.
start = float(heldout["start_time"].min())
selected = heldout.loc[
    heldout["start_time"].ge(start)
    & heldout["start_time"].lt(start + 60.0)
].sort_values(["start_time", "job_name", "task_name"])

rng = random.Random(2026)
evaluation_tasks = []

for i, row in enumerate(selected.itertuples(index=False)):
    arrival = float(row.start_time - start)
    duration = float(row.recorded_duration_s)
    cluster = 0 if rng.random() < 0.65 else 1

    task = {
        "id": f"evaluation_{i}",
        "arrival_time": arrival,
        "origin_node": cluster * 3 + rng.randrange(3),
        "cpu_request": float(row.plan_cpu) / 100.0,
        "mem_request": float(row.plan_mem),
        "reference_runtime_s": duration,
        "data_size_mb": 40.0,
        "urgency_flag": int(rng.random() < 0.2),
        "source_device": rng.randrange(3),
        "deadline": arrival + 2 * duration + 30
    }
    task["label"] = classifier.classify(task)
    evaluation_tasks.append(task)

assert evaluation_tasks
print("Held-out tasks:", len(evaluation_tasks), flush=True)

evaluation_rows = []
evaluation_results = {}
results_dir = project / "results"
results_dir.mkdir(exist_ok=True)

for policy in ["trained_ddqn", "fixed_routing", "cloud_only"]:
    print("\nEvaluating:", policy, flush=True)

    env = env_module.FogEscalationEnv(bundle["model"])
    state = env.reset(evaluation_tasks)

    done = False
    decisions = 0
    reward_total = 0.0
    action_counts = [0] * 5

    while not done:
        valid = env.valid_actions()

        if policy == "trained_ddqn":
            action = trained_agent.act(
                state, greedy=True, valid_actions=valid
            )
        elif policy == "fixed_routing":
            # Local -> sibling -> neighbour -> cloud. No holding.
            action = next(a for a in [0, 1, 2, 3] if a in valid)
        else:
            action = 3

        state, reward, done, info = env.step(action)

        decisions += 1
        reward_total += reward
        action_counts[action] += 1

        assert decisions <= 4 * len(evaluation_tasks)

        if decisions % 500 == 0:
            print("Decisions:", decisions, flush=True)

    frame = pd.DataFrame(env.completed)
    assert len(frame) == len(evaluation_tasks)
    assert frame["id"].is_unique

    frame.to_csv(
        results_dir / f"ddqn_evaluation_{policy}.csv",
        index=False
    )
    evaluation_results[policy] = frame

    row = {
        "policy": policy,
        "tasks": len(frame),
        "success_pct": 100 * frame["deadline_met"].mean(),
        "mean_response_s": frame["response_s"].mean(),
        "p95_response_s": frame["response_s"].quantile(0.95),
        "cloud_tasks": int((frame["route"] == "cloud").sum()),
        "decisions": decisions,
        "hold_actions": action_counts[4],
        "reward": reward_total
    }
    evaluation_rows.append(row)

    print("Finished:", policy, flush=True)

evaluation_summary = pd.DataFrame(evaluation_rows)
evaluation_summary.to_csv(
    results_dir / "ddqn_heldout_evaluation.csv",
    index=False
)

print("\n===== HELD-OUT EVALUATION =====")
print(evaluation_summary.round(3).to_string(index=False))
print("\nNo learning updates were performed during evaluation.")

In [ ]:
ddqn = evaluation_results["trained_ddqn"]

print("Hold counts per task:")
print(ddqn["holds"].value_counts().sort_index())

print("\nFinal destinations:")
print(ddqn["route"].value_counts())

print("\nResponse and success by destination:")
print(
    ddqn.groupby("route").agg(
        tasks=("id", "count"),
        mean_response_s=("response_s", "mean"),
        success_pct=("deadline_met", lambda x: 100 * x.mean())
    ).round(3)
)

In [ ]:
import numpy as np
import pandas as pd

ddqn = evaluation_results["trained_ddqn"].copy()
fog = ddqn.loc[ddqn["route"] != "cloud"].copy()

# The evaluation contains arrivals from a 60-second window.
# Each fog node executes one task at a time in this simulator.
offered_runtime = ddqn["reference_runtime_s"].sum()
fog_service_budget = 6 * 60.0

print("Recorded runtime arriving in the window:",
      round(offered_runtime, 1), "seconds")
print("Six serial fog workers' service budget:",
      fog_service_budget, "seconds")
print("Offered work / fog service budget:",
      round(offered_runtime / fog_service_budget, 2))
print("(Above 1 means all-fog service cannot keep up sustainably.)")

fog["queue_wait_s"] = (
    fog["start_time"]
    - fog["placement_time"]
    - fog["transfer_s"]
)

print("\nFog tasks: prediction versus execution")
print(
    fog.groupby("route").agg(
        tasks=("id", "count"),
        mean_predicted_runtime=("predicted_compute_s", "mean"),
        mean_recorded_runtime=("reference_runtime_s", "mean"),
        mean_queue_wait=("queue_wait_s", "mean"),
        success_pct=("deadline_met", lambda x: 100 * x.mean())
    ).round(2).to_string()
)

In [ ]:
from pathlib import Path

project = Path("/kaggle/working/something_unique")
source = project / "ddqn_environment.py"
destination = project / "ddqn_environment_concurrent.py"

code = source.read_text()

old = "        self.active = [None] * 6"
new = """        self.active = [{} for _ in range(6)]
        self.running_cpu = np.zeros(6)
        self.running_mem = np.zeros(6)"""

assert code.count(old) == 1
code = code.replace(old, new, 1)

# Replace the single-task worker with capacity-aware concurrent starts.
start = code.index("    def _start_worker(self, node):")
end = code.index("    def _record(self, task):", start)

worker = '''    def _start_worker(self, node):
        self.queues[node].rescore_all(self.now)
        deferred = []

        while len(self.queues[node]):
            task = self.queues[node].pop_next()

            fits = (
                self.running_cpu[node] + task["cpu_request"]
                <= self.cpu_cap[node] + 1e-9
                and self.running_mem[node] + task["mem_request"]
                <= self.mem_cap[node] + 1e-9
            )

            if not fits:
                deferred.append(task)
                continue

            task["start_time"] = self.now
            self.active[node][task["id"]] = task
            self.running_cpu[node] += task["cpu_request"]
            self.running_mem[node] += task["mem_request"]

            self._push(
                self.now + task["reference_runtime_s"],
                "finish", (node, task)
            )

        for task in deferred:
            self.queues[node].enqueue(task, self.now)

        assert np.all(self.running_cpu <= self.cpu_cap + 1e-6)
        assert np.all(self.running_mem <= self.mem_cap + 1e-6)

'''

code = code[:start] + worker + code[end:]

old = """                self.active[node] = None
                self._record(task)"""

new = """                self.active[node].pop(task["id"])
                self.running_cpu[node] -= task["cpu_request"]
                self.running_mem[node] -= task["mem_request"]
                self._record(task)"""

assert code.count(old) == 1
code = code.replace(old, new, 1)

old = """            active = self.active[node]
            if active is not None:
                waiting[node] += max(
                    0.0,
                    active["predicted_compute_s"]
                    - (self.now - active["start_time"])
                )"""

new = """            # Running tasks overlap; use the longest estimated
            # remaining runtime rather than summing them.
            waiting[node] += max(
                (
                    max(
                        0.0,
                        task["predicted_compute_s"]
                        - (self.now - task["start_time"])
                    )
                    for task in self.active[node].values()
                ),
                default=0.0
            )"""

assert code.count(old) == 1
code = code.replace(old, new, 1)

old = """        assert np.allclose(self.mem, 0)"""
new = """        assert np.allclose(self.mem, 0)
        assert np.allclose(self.running_cpu, 0)
        assert np.allclose(self.running_mem, 0)
        assert all(not tasks for tasks in self.active)"""

assert code.count(old) == 1
code = code.replace(old, new, 1)

compile(code, str(destination), "exec")
destination.write_text(code)

print("Created:", destination)
print("Original environment preserved.")

In [ ]:
import importlib.util
import numpy as np

spec = importlib.util.spec_from_file_location(
    "concurrent_env", destination
)
concurrent_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(concurrent_module)

class TestPredictor:
    def predict(self, features):
        return np.full(len(features), 10.0)

def test_task(task_id, cpu):
    return {
        "id": task_id,
        "arrival_time": 0.0,
        "origin_node": 0,
        "cpu_request": cpu,
        "mem_request": 1.0,
        "reference_runtime_s": 10.0,
        "data_size_mb": 0.0,
        "deadline": 100.0,
        "label": "can_wait"
    }

# Two tasks fit together at F1: both should finish at t=10.
env = concurrent_module.FogEscalationEnv(TestPredictor())
env.reset([test_task("A", 3.0), test_task("B", 3.0)])

_, _, done, _ = env.step(0)
assert not done

_, _, done, _ = env.step(0)
assert done

assert all(t["start_time"] == 0.0 for t in env.completed)
assert all(t["finish_time"] == 10.0 for t in env.completed)

print("Passed: two tasks execute concurrently and finish at t=10.")

# Two 6-CPU tasks cannot both reserve F1's 8-CPU capacity.
env.reset([test_task("C", 6.0), test_task("D", 6.0)])
env.step(0)

assert 0 not in env.valid_actions()

# Complete the second task through the guaranteed cloud fallback.
env.step(3)

print("Passed: insufficient CPU blocks another local placement.")
print("Completion and resource-release checks passed.")

In [ ]:
from pathlib import Path
import sys

project = Path("/kaggle/working/something_unique")

if str(project) not in sys.path:
    sys.path.insert(0, str(project))

# Create a separate 10-input agent.
source = project / "agents/escalation_ddqn.py"
destination = project / "agents/escalation_ddqn_v2.py"

code = source.read_text()

assert code.count("STATE_DIM = 6") == 1
code = code.replace("STATE_DIM = 6", "STATE_DIM = 10", 1)

# Remove the old six-field state helper.
# The environment now constructs the complete observation.
start = code.index("def make_state(")
end = code.index("class DuelingQNet", start)
code = code[:start] + code[end:]

compile(code, str(destination), "exec")
destination.write_text(code)

# Extend the concurrent environment without changing its original file.
environment_code = '''
import numpy as np
from ddqn_environment_concurrent import FogEscalationEnv as BaseEnv


STATE_FEATURES = [
    "own_reserved_demand",
    "best_sibling_reserved_demand",
    "best_peer_reserved_demand",
    "urgency",
    "remaining_deadline_scaled",
    "source_queue_length_scaled",
    "hold_fraction_used",
    "task_cpu_over_source_capacity",
    "task_memory_over_source_capacity",
    "predicted_runtime_scaled"
]


class FogEscalationEnvV2(BaseEnv):
    def _state(self):
        if self.done:
            return np.zeros(10, dtype=np.float32)

        base = super()._state()
        task = self.current
        origin = task["origin_node"]

        extra = np.array([
            task["holds"] / max(1, self.max_holds),
            task["cpu_request"] / self.cpu_cap[origin],
            task["mem_request"] / self.mem_cap[origin],
            self.predicted[origin] / 300.0
        ], dtype=np.float32)

        observation = np.concatenate([base, extra])
        assert observation.shape == (10,)
        assert np.isfinite(observation).all()
        return observation
'''

env_v2_path = project / "ddqn_environment_v2.py"
compile(environment_code, str(env_v2_path), "exec")
env_v2_path.write_text(environment_code)

print("Created: agents/escalation_ddqn_v2.py")
print("Created: ddqn_environment_v2.py")
print("Original agent, environment and checkpoint preserved.")

In [ ]:
import importlib.util
import numpy as np
import torch

def load_version(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

agent_v2_module = load_version(
    "agent_v2_check",
    project / "agents/escalation_ddqn_v2.py"
)
env_v2_module = load_version(
    "env_v2_check",
    project / "ddqn_environment_v2.py"
)

class CheckPredictor:
    def predict(self, features):
        return np.full(len(features), 10.0)

task = {
    "id": "hold_check",
    "arrival_time": 0.0,
    "origin_node": 0,
    "cpu_request": 1.0,
    "mem_request": 1.0,
    "reference_runtime_s": 10.0,
    "data_size_mb": 0.0,
    "deadline": 100.0,
    "label": "can_wait"
}

check_env = env_v2_module.FogEscalationEnvV2(CheckPredictor())
before = check_env.reset([task])
after, reward, done, info = check_env.step(4)

assert before.shape == after.shape == (10,)
assert before[6] == 0.0
assert np.isclose(after[6], 1 / 3)
assert not done
assert info["time"] == 1.0

check_agent = agent_v2_module.EscalationDDQN(seed=42)

with torch.no_grad():
    output = check_agent.online(torch.tensor(after).unsqueeze(0))

assert output.shape == (1, 5)

# Finish the task and verify terminal observation.
terminal, _, done, _ = check_env.step(0)
assert done
assert terminal.shape == (10,)
assert np.all(terminal == 0)

print("Passed: hold count changes the observation.")
print("Passed: 10 state inputs produce 5 action values.")
print("Passed: terminal observation and completion.")

In [ ]:
import joblib
import pandas as pd

bundle = joblib.load(
    project / "models/alibaba_runtime_rf.joblib"
)

rows = []

for policy in ["fixed_routing", "cloud_only"]:
    print("Running:", policy, flush=True)

    env = env_v2_module.FogEscalationEnvV2(bundle["model"])
    state = env.reset(evaluation_tasks)

    done = False
    decisions = 0

    while not done:
        valid = env.valid_actions()

        action = (
            3 if policy == "cloud_only"
            else next(a for a in [0, 1, 2, 3] if a in valid)
        )

        state, reward, done, info = env.step(action)
        decisions += 1

        if decisions % 500 == 0:
            print("Decisions:", decisions, flush=True)

    frame = pd.DataFrame(env.completed)

    assert len(frame) == len(evaluation_tasks)
    assert frame["id"].is_unique

    rows.append({
        "policy": policy,
        "tasks": len(frame),
        "success_pct": 100 * frame["deadline_met"].mean(),
        "mean_response_s": frame["response_s"].mean(),
        "p95_response_s": frame["response_s"].quantile(0.95),
        "fog_tasks": int((frame["route"] != "cloud").sum()),
        "cloud_tasks": int((frame["route"] == "cloud").sum())
    })

    frame.to_csv(
        project / f"results/concurrent_{policy}_diagnostic.csv",
        index=False
    )

concurrency_summary = pd.DataFrame(rows)
print("\n", concurrency_summary.round(3).to_string(index=False))

In [ ]:
import joblib
import numpy as np
import pandas as pd

bundle = joblib.load(
    project / "models/alibaba_runtime_rf.joblib"
)

agent_v2 = agent_v2_module.EscalationDDQN(
    seed=42,
    learning_rate=0.0003,
    epsilon=1.0,
    target_sync_every=200
)

training_env_v2 = env_v2_module.FogEscalationEnvV2(
    bundle["model"]
)

v2_log = []

for index in np.random.default_rng(42).permutation(len(episodes)):
    episode = episodes[int(index)]
    state = training_env_v2.reset(episode["tasks"])

    done = False
    decisions = 0
    total_reward = 0.0
    holds = 0
    losses = []

    print(
        f"Starting episode {episode['episode_id']}"
        f" — {len(episode['tasks'])} tasks",
        flush=True
    )

    while not done:
        action = agent_v2.act(
            state,
            valid_actions=training_env_v2.valid_actions()
        )

        next_state, reward, done, info = training_env_v2.step(action)

        agent_v2.remember(
            state, action, reward, next_state, done,
            next_valid_actions=info["valid_actions"]
        )

        loss = agent_v2.train_step(batch_size=64)
        if loss is not None:
            assert np.isfinite(loss)
            losses.append(loss)

        state = next_state
        total_reward += reward
        decisions += 1
        holds += int(action == 4)
        agent_v2.epsilon = max(0.10, agent_v2.epsilon * 0.9995)

        assert decisions <= 4 * len(episode["tasks"])

        if decisions % 500 == 0:
            print("  Decisions:", decisions, flush=True)

    completed = training_env_v2.completed

    v2_log.append({
        "episode": episode["episode_id"],
        "tasks": len(completed),
        "decisions": decisions,
        "holds": holds,
        "reward": total_reward,
        "success_pct": 100 * np.mean(
            [t["deadline_met"] for t in completed]
        ),
        "cloud_tasks": sum(t["route"] == "cloud" for t in completed),
        "epsilon": agent_v2.epsilon,
        "mean_loss": np.mean(losses) if losses else None,
        "training_steps": agent_v2.training_steps
    })

    agent_v2.save(
        project / "models/escalation_ddqn_v2_initial.pt"
    )
    pd.DataFrame(v2_log).to_csv(
        project / "results/ddqn_v2_initial_training.csv",
        index=False
    )

    print(
        f"Saved episode {episode['episode_id']}"
        f" | success {v2_log[-1]['success_pct']:.1f}%"
        f" | holds {holds}"
        f" | epsilon {agent_v2.epsilon:.3f}",
        flush=True
    )

print("\nVersion 2 initial training finished.")
print("Gradient updates:", agent_v2.training_steps)
print(pd.DataFrame(v2_log).round(3).to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd

evaluated_agent = agent_v2_module.EscalationDDQN(seed=42).load(
    project / "models/escalation_ddqn_v2_initial.pt"
)

rows = []
v2_evaluation_results = {}

for policy in ["ddqn_v2", "fixed_routing", "cloud_only"]:
    print("Evaluating:", policy, flush=True)

    env = env_v2_module.FogEscalationEnvV2(bundle["model"])
    state = env.reset(evaluation_tasks)

    done = False
    decisions = 0
    holds = 0
    total_reward = 0.0

    while not done:
        valid = env.valid_actions()

        if policy == "ddqn_v2":
            action = evaluated_agent.act(
                state, greedy=True, valid_actions=valid
            )
        elif policy == "fixed_routing":
            action = next(a for a in [0, 1, 2, 3] if a in valid)
        else:
            action = 3

        state, reward, done, info = env.step(action)

        decisions += 1
        holds += int(action == 4)
        total_reward += reward

        assert decisions <= 4 * len(evaluation_tasks)

        if decisions % 500 == 0:
            print("  Decisions:", decisions, flush=True)

    frame = pd.DataFrame(env.completed)
    assert len(frame) == len(evaluation_tasks)
    assert frame["id"].is_unique

    v2_evaluation_results[policy] = frame

    rows.append({
        "policy": policy,
        "tasks": len(frame),
        "success_pct": 100 * frame["deadline_met"].mean(),
        "mean_response_s": frame["response_s"].mean(),
        "p95_response_s": frame["response_s"].quantile(0.95),
        "fog_tasks": int((frame["route"] != "cloud").sum()),
        "cloud_tasks": int((frame["route"] == "cloud").sum()),
        "hold_actions": holds,
        "holds_per_task": holds / len(frame),
        "reward": total_reward
    })

    frame.to_csv(
        project / f"results/v2_diagnostic_{policy}.csv",
        index=False
    )

v2_summary = pd.DataFrame(rows)
v2_summary.to_csv(
    project / "results/ddqn_v2_diagnostic_summary.csv",
    index=False
)

print("\n", v2_summary.round(3).to_string(index=False))
print("\nGreedy evaluation complete; no learning updates performed.")

In [ ]:
from pathlib import Path

project = Path("/kaggle/working/something_unique")

for name in [
    "models/escalation_ddqn_v2_initial.pt",
    "agents/escalation_ddqn_v2.py",
    "ddqn_environment_v2.py",
    "ddqn_environment_concurrent.py",
    "ddqn_environment.py",
    "models/alibaba_runtime_rf.joblib"
]:
    path = project / name
    print(f"{'FOUND' if path.exists() else 'MISSING'}: {name}")

In [ ]:
import sys
import time
import joblib
import torch
import pandas as pd
import numpy as np
from pathlib import Path

project = Path("/kaggle/working/something_unique")
sys.path.insert(0, str(project))

torch.set_num_threads(1)

from agents.escalation_ddqn_v2 import EscalationDDQN
from ddqn_environment_v2 import FogEscalationEnvV2

bundle = joblib.load(
    project / "models/alibaba_runtime_rf.joblib"
)
runtime_model = bundle["model"]
runtime_model.set_params(n_jobs=1)

agent = EscalationDDQN(seed=42).load(
    project / "models/escalation_ddqn_v2_initial.pt"
)

# Restore the exact diagnostic tasks from a completed earlier run.
previous = pd.read_csv(
    project / "results/ddqn_evaluation_fixed_routing.csv"
)

fields = [
    "id", "arrival_time", "origin_node",
    "cpu_request", "mem_request", "reference_runtime_s",
    "data_size_mb", "deadline", "label"
]

evaluation_tasks = previous[fields].sort_values(
    ["arrival_time", "id"]
).to_dict("records")

# One batched prediction for the diagnostic workload.
feature_names = ["cpu_request", "mem_request"]
features = pd.DataFrame(evaluation_tasks)[feature_names]

start = time.perf_counter()
predictions = runtime_model.predict(features)

prediction_cache = {
    (float(row.cpu_request), float(row.mem_request)): float(pred)
    for row, pred in zip(
        features.itertuples(index=False), predictions
    )
}

class CachedPredictor:
    def predict(self, features):
        return np.array([
            prediction_cache[
                (float(row.cpu_request), float(row.mem_request))
            ]
            for row in features.itertuples(index=False)
        ])

cached_model = CachedPredictor()

assert np.allclose(
    cached_model.predict(features),
    predictions
)

print("Model reloaded.")
print("Diagnostic tasks restored:", len(evaluation_tasks))
print("Prediction preparation:", round(
    time.perf_counter() - start, 3
), "seconds")
print("Cached predictions match the RF.")

In [ ]:
env = FogEscalationEnvV2(cached_model)
state = env.reset(evaluation_tasks[:20])

done = False
decisions = 0
start = time.perf_counter()

while not done:
    action = agent.act(
        state,
        greedy=True,
        valid_actions=env.valid_actions()
    )
    state, reward, done, info = env.step(action)
    decisions += 1

    assert decisions <= 80

    if decisions % 10 == 0:
        print(
            "Decisions:", decisions,
            "| elapsed:", round(time.perf_counter() - start, 2),
            "seconds",
            flush=True
        )

    if time.perf_counter() - start > 30 and not done:
        raise RuntimeError(
            "Speed check exceeded 30 seconds. Do not run full evaluation."
        )

print("Completed:", len(env.completed))
print("Total elapsed:", round(time.perf_counter() - start, 3), "seconds")

In [ ]:
import time
import pandas as pd

rows = []
fast_evaluation_results = {}

for policy in ["ddqn_v2", "fixed_routing", "cloud_only"]:
    print("Evaluating:", policy, flush=True)

    env = FogEscalationEnvV2(cached_model)
    state = env.reset(evaluation_tasks)

    done = False
    decisions = 0
    holds = 0
    total_reward = 0.0
    start = time.perf_counter()

    while not done:
        valid = env.valid_actions()

        if policy == "ddqn_v2":
            action = agent.act(
                state, greedy=True, valid_actions=valid
            )
        elif policy == "fixed_routing":
            action = next(a for a in [0, 1, 2, 3] if a in valid)
        else:
            action = 3

        state, reward, done, info = env.step(action)
        decisions += 1
        holds += int(action == 4)
        total_reward += reward

        assert decisions <= 4 * len(evaluation_tasks)

        elapsed = time.perf_counter() - start

        if decisions % 250 == 0:
            print(
                f"  {decisions} decisions | {elapsed:.2f}s",
                flush=True
            )

        if elapsed > 60 and not done:
            raise RuntimeError(
                f"{policy} exceeded 60 seconds. "
                "Stopped before reporting an incomplete result."
            )

    frame = pd.DataFrame(env.completed)
    assert len(frame) == len(evaluation_tasks)
    assert frame["id"].is_unique

    fast_evaluation_results[policy] = frame

    rows.append({
        "policy": policy,
        "tasks": len(frame),
        "success_pct": 100 * frame["deadline_met"].mean(),
        "mean_response_s": frame["response_s"].mean(),
        "p95_response_s": frame["response_s"].quantile(0.95),
        "fog_tasks": int((frame["route"] != "cloud").sum()),
        "cloud_tasks": int((frame["route"] == "cloud").sum()),
        "hold_actions": holds,
        "holds_per_task": holds / len(frame),
        "reward": total_reward,
        "evaluation_wall_s": time.perf_counter() - start
    })

    frame.to_csv(
        project / f"results/v2_fast_diagnostic_{policy}.csv",
        index=False
    )

fast_summary = pd.DataFrame(rows)
fast_summary.to_csv(
    project / "results/ddqn_v2_fast_diagnostic_summary.csv",
    index=False
)

print("\n", fast_summary.round(3).to_string(index=False))

In [ ]:
from pathlib import Path
import shutil
import zipfile

project = Path("/kaggle/working/something_unique")

required = [
    "agents/escalation_ddqn_v2.py",
    "models/escalation_ddqn_v2_initial.pt",
    "ddqn_environment.py",
    "ddqn_environment_concurrent.py",
    "ddqn_environment_v2.py",
    "agents/batch_placement.py",
    "replay_alibaba.py",
    "models/alibaba_runtime_rf.joblib"
]

missing = [name for name in required if not (project / name).exists()]
assert not missing, f"Missing files: {missing}"

archive = shutil.make_archive(
    "/kaggle/working/fog_scheduler_meeting_checkpoint",
    "zip",
    root_dir=project.parent,
    base_dir=project.name
)

with zipfile.ZipFile(archive) as saved:
    assert saved.testzip() is None

print("Verified backup:", archive)

print("\nAvailable results:")
for path in sorted((project / "results").glob("*.csv")):
    print(path.name)

In [ ]:
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
import joblib

from scipy.optimize import milp, Bounds, LinearConstraint
from scipy.sparse import lil_matrix, csr_matrix

project = Path("/kaggle/working/something_unique")
out_dir = project / "results"
out_dir.mkdir(exist_ok=True)

bundle = joblib.load(
    project / "models/alibaba_runtime_rf.joblib"
)
runtime_model = bundle["model"]
runtime_model.set_params(n_jobs=1)

raw = pd.read_csv(
    project / "data/alibaba_single_instance_1hour.csv"
)

# Use earlier records, leaving the later evaluation interval out.
pool = raw.loc[
    raw["start_time"].lt(bundle["cutoff"])
    & raw["end_time"].le(bundle["cutoff"])
].copy()

TIME_LIMIT = 3.0


def solve_snapshot(cpu, mem, free_cpu, free_mem, costs):
    """Maximize fog admissions, then minimize their placement cost."""
    m, n = costs.shape

    if m == 0:
        return np.empty(0, dtype=int), 0.0

    matrix = lil_matrix((m + 2 * n, m * n))

    for i in range(m):
        matrix[i, i*n:(i+1)*n] = 1

    for j in range(n):
        columns = np.arange(m) * n + j
        matrix[m + j, columns] = cpu
        matrix[m + n + j, columns] = mem

    upper = np.concatenate([np.ones(m), free_cpu, free_mem])
    constraint = LinearConstraint(
        matrix.tocsr(), np.full(len(upper), -np.inf), upper
    )

    allowed = (
        (cpu[:, None] <= free_cpu[None, :] + 1e-9)
        & (mem[:, None] <= free_mem[None, :] + 1e-9)
    )

    bounds = Bounds(
        np.zeros(m*n), allowed.ravel().astype(float)
    )
    options = {"time_limit": TIME_LIMIT, "mip_rel_gap": 0.0}

    first = milp(
        c=-np.ones(m*n),
        integrality=np.ones(m*n),
        bounds=bounds,
        constraints=constraint,
        options=options
    )

    if first.status != 0:
        raise RuntimeError(
            f"Admission stage status {first.status}: {first.message}"
        )

    admitted = int(np.rint(first.x).sum())

    count_constraint = LinearConstraint(
        csr_matrix(np.ones((1, m*n))), admitted, admitted
    )

    second = milp(
        c=costs.ravel(),
        integrality=np.ones(m*n),
        bounds=bounds,
        constraints=[constraint, count_constraint],
        options=options
    )

    if second.status != 0:
        raise RuntimeError(
            f"Cost stage status {second.status}: {second.message}"
        )

    assignment = np.rint(second.x).astype(int).reshape(m, n)

    assert np.all(assignment.sum(axis=1) <= 1)
    assert not np.any(assignment.astype(bool) & ~allowed)
    assert np.all(assignment.T @ cpu <= free_cpu + 1e-6)
    assert np.all(assignment.T @ mem <= free_mem + 1e-6)

    choices = np.full(m, -1, dtype=int)
    placed = assignment.sum(axis=1) > 0
    choices[placed] = assignment[placed].argmax(axis=1)

    return choices, float((assignment * costs).sum())


def scoped_snapshot(cpu, mem, free_cpu, free_mem, costs, origins):
    """Own cluster, then one paired peer; never searches >3 nodes."""
    n = len(free_cpu)
    groups = n // 3
    spare_cpu = free_cpu.copy()
    spare_mem = free_mem.copy()
    choices = np.full(len(cpu), -1, dtype=int)
    calls = 0

    def place(ids, group):
        nonlocal calls
        if not len(ids):
            return

        nodes = np.arange(group*3, group*3 + 3)
        local, _ = solve_snapshot(
            cpu[ids], mem[ids],
            spare_cpu[nodes], spare_mem[nodes],
            costs[np.ix_(ids, nodes)]
        )
        calls += 1

        for task_id, local_node in zip(ids, local):
            if local_node >= 0:
                node = int(nodes[local_node])
                choices[task_id] = node
                spare_cpu[node] -= cpu[task_id]
                spare_mem[node] -= mem[task_id]

    # Let each cluster serve its own workload first.
    for group in range(groups):
        place(np.flatnonzero(origins // 3 == group), group)

    # Paired controller groups: (0,1), (2,3), ...
    for group in range(groups):
        leftovers = np.flatnonzero(
            (origins // 3 == group) & (choices < 0)
        )
        place(leftovers, group ^ 1)

    placed = choices >= 0
    cost = costs[
        np.flatnonzero(placed), choices[placed]
    ].sum()

    # Verify the complete combined solution.
    used_cpu = np.zeros(n)
    used_mem = np.zeros(n)

    for i, node in enumerate(choices):
        if node >= 0:
            used_cpu[node] += cpu[i]
            used_mem[node] += mem[i]

    assert np.all(used_cpu <= free_cpu + 1e-6)
    assert np.all(used_mem <= free_mem + 1e-6)

    return choices, float(cost), calls


rows = []

for n in [6, 12, 24]:
    for seed in [42, 43, 44]:
        rng = np.random.default_rng(seed)
        m = 4 * n  # Demand grows with network size.

        sample = pool.sample(
            n=m, replace=False, random_state=seed
        )

        cpu = sample["plan_cpu"].to_numpy(float) / 100.0
        mem = sample["plan_mem"].to_numpy(float)

        # Same repeating node types at every size.
        capacity_cpu = np.tile([8., 10., 12.], n // 3)
        capacity_mem = capacity_cpu * 0.8
        bandwidth = np.tile([40., 60., 100.], n // 3)
        speed = np.tile([0.8, 1.0, 1.2], n // 3)

        # Fixed stressed snapshot: 25% of each resource is free.
        free_cpu = capacity_cpu * 0.25
        free_mem = capacity_mem * 0.25

        # 65% of arrivals go to the first half of the network.
        weights = np.concatenate([
            np.full(n // 2, 0.65 / (n // 2)),
            np.full(n // 2, 0.35 / (n // 2))
        ])
        origins = rng.choice(n, size=m, p=weights)

        features = pd.DataFrame({
            "cpu_request": cpu,
            "mem_request": mem
        })

        preparation_start = perf_counter()
        predicted = np.maximum(
            0.2, runtime_model.predict(features)
        )
        costs = predicted[:, None] / speed[None, :]

        # Assumed payload: 40 MB; transfer uses Mbps correctly.
        for i, origin in enumerate(origins):
            for node in range(n):
                if node != origin:
                    if node // 3 == origin // 3:
                        hops = 1
                    elif (node // 3) // 2 == (origin // 3) // 2:
                        hops = 3
                    else:
                        hops = 5

                    costs[i, node] += (
                        8 * 40.0
                        / min(bandwidth[origin], bandwidth[node])
                        * (1 + 0.5 * hops)
                    )

        preparation_ms = (
            perf_counter() - preparation_start
        ) * 1000

        methods = ["flat", "scoped"]
        if seed % 2:
            methods.reverse()

        for method in methods:
            print(
                f"Running {n} nodes, {m} tasks, "
                f"seed {seed}, {method}...",
                flush=True
            )

            begin = perf_counter()
            row = {
                "nodes": n,
                "tasks": m,
                "seed": seed,
                "method": method,
                "common_cost_preparation_ms": preparation_ms,
                "max_nodes_per_solve": n if method == "flat" else 3
            }

            try:
                if method == "flat":
                    choices, cost = solve_snapshot(
                        cpu, mem, free_cpu, free_mem, costs
                    )
                    calls = 1
                else:
                    choices, cost, calls = scoped_snapshot(
                        cpu, mem, free_cpu, free_mem, costs, origins
                    )

                row.update({
                    "status": "optimal_subproblems",
                    "fog_placed": int((choices >= 0).sum()),
                    "cloud_leftovers": int((choices < 0).sum()),
                    "fog_cost": cost,
                    "solver_calls": calls
                })

            except RuntimeError as error:
                row.update({
                    "status": str(error),
                    "fog_placed": np.nan,
                    "cloud_leftovers": np.nan,
                    "fog_cost": np.nan,
                    "solver_calls": np.nan
                })

            row["placement_ms"] = (
                perf_counter() - begin
            ) * 1000
            rows.append(row)

            # Preserve progress after every method.
            pd.DataFrame(rows).to_csv(
                out_dir / "meeting_scaling_raw.csv",
                index=False
            )

scaling_results = pd.DataFrame(rows)

print("\n===== RAW RESULTS =====")
print(scaling_results.round(3).to_string(index=False))
print("\nSaved: results/meeting_scaling_raw.csv")

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

raw_results = pd.read_csv(
    project / "results/meeting_scaling_raw.csv"
)

# Only compare trials where BOTH methods completed optimally.
successful = raw_results.loc[
    raw_results["status"].eq("optimal_subproblems")
].copy()

paired_keys = successful.groupby(
    ["nodes", "seed"]
)["method"].nunique()

paired_keys = paired_keys[paired_keys.eq(2)].reset_index()

paired = successful.merge(
    paired_keys[["nodes", "seed"]],
    on=["nodes", "seed"],
    how="inner"
)

if paired.empty:
    print("No fully solved paired trials. Show solver statuses instead.")
    display(raw_results[["nodes", "seed", "method", "status"]])
else:
    summary = paired.groupby(
        ["nodes", "method"], as_index=False
    ).agg(
        paired_trials=("seed", "count"),
        median_placement_ms=("placement_ms", "median"),
        mean_fog_placed=("fog_placed", "mean"),
        mean_cloud_leftovers=("cloud_leftovers", "mean"),
        mean_solver_calls=("solver_calls", "mean")
    )

    display(summary.round(3))

    summary.to_csv(
        project / "results/meeting_scaling_summary.csv",
        index=False
    )

    fig, ax = plt.subplots(figsize=(7, 4))

    for method, group in summary.groupby("method"):
        ax.plot(
            group["nodes"],
            group["median_placement_ms"],
            marker="o",
            label=method
        )

    ax.set_xlabel("Total leaf fog nodes")
    ax.set_ylabel("Median placement computation time (ms)")
    ax.set_title("Preliminary Alibaba-derived placement benchmark")
    ax.set_xticks([6, 12, 24])
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    fig.savefig(
        project / "results/meeting_scaling_plot.png",
        dpi=180
    )
    plt.show()

failed = raw_results.loc[
    ~raw_results["status"].eq("optimal_subproblems")
]

print("Unsolved/timed-out method runs:", len(failed))
if len(failed):
    display(failed[["nodes", "seed", "method", "status"]])

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

project = Path("/kaggle/working/something_unique")
results_dir = project / "results"

frames = []

for policy in ["ddqn_v2", "fixed_routing", "cloud_only"]:
    path = results_dir / f"v2_fast_diagnostic_{policy}.csv"
    df = pd.read_csv(path)

    df["policy"] = policy
    df["decision_wait_s"] = (
        df["placement_time"] - df["arrival_time"]
    )

    # Cloud starts after transfer; fog may additionally wait in queue.
    df["queue_wait_s"] = (
        df["start_time"]
        - df["placement_time"]
        - df["transfer_s"]
    )

    df["execution_s"] = (
        df["finish_time"] - df["start_time"]
    )

    assert (df["queue_wait_s"] >= -1e-6).all()
    frames.append(df)

all_tasks = pd.concat(frames, ignore_index=True)

breakdown = all_tasks.groupby(
    ["policy", "route"], as_index=False
).agg(
    tasks=("id", "count"),
    mean_response_s=("response_s", "mean"),
    p95_response_s=("response_s", lambda x: x.quantile(0.95)),
    mean_decision_wait_s=("decision_wait_s", "mean"),
    mean_transfer_s=("transfer_s", "mean"),
    mean_queue_wait_s=("queue_wait_s", "mean"),
    mean_execution_s=("execution_s", "mean"),
    success_pct=("deadline_met", lambda x: 100 * x.mean())
)

breakdown.to_csv(
    results_dir / "meeting_response_breakdown.csv",
    index=False
)

display(breakdown.round(3))

# Show the four placement categories without implying extra fog tiers.
route_order = ["local", "sibling", "cross_cluster", "cloud"]

counts = pd.crosstab(
    all_tasks["policy"], all_tasks["route"]
).reindex(columns=route_order, fill_value=0)

display(counts)

ax = counts.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 4),
    color=["#4472C4", "#70AD47", "#FFC000", "#ED7D31"]
)
ax.set_title("Placement destinations — 816-task diagnostic")
ax.set_xlabel("")
ax.set_ylabel("Number of tasks")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Route", bbox_to_anchor=(1.02, 1))
plt.tight_layout()

plt.savefig(
    results_dir / "meeting_placement_routes.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

project = Path("/kaggle/working/something_unique")
results_dir = project / "results"

raw = pd.read_csv(results_dir / "meeting_scaling_raw.csv")

# Keep only seeds where both methods solved successfully.
ok = raw.loc[
    raw["status"].eq("optimal_subproblems")
].copy()

paired_keys = (
    ok.groupby(["nodes", "seed"])["method"]
    .nunique()
    .reset_index(name="methods")
)
paired_keys = paired_keys.loc[paired_keys["methods"].eq(2)]

paired = ok.merge(
    paired_keys[["nodes", "seed"]],
    on=["nodes", "seed"]
)

paired["fog_placement_pct"] = (
    100 * paired["fog_placed"] / paired["tasks"]
)

summary = paired.groupby(
    ["nodes", "method"], as_index=False
).agg(
    trials=("seed", "count"),
    median_placement_ms=("placement_ms", "median"),
    mean_fog_placement_pct=("fog_placement_pct", "mean"),
    mean_cloud_leftovers=("cloud_leftovers", "mean")
)

display(Markdown("### Speed and placement quality — paired trials"))
display(summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for method, group in summary.groupby("method"):
    axes[0].plot(
        group["nodes"], group["median_placement_ms"],
        marker="o", label=method
    )
    axes[1].plot(
        group["nodes"], group["mean_fog_placement_pct"],
        marker="o", label=method
    )

axes[0].set_title("Placement computation time")
axes[0].set_ylabel("Median milliseconds")
axes[0].set_yscale("log")

axes[1].set_title("Fog placement quality")
axes[1].set_ylabel("Tasks placed in fog (%)")
axes[1].set_ylim(0, 105)

for ax in axes:
    ax.set_xlabel("Leaf fog nodes")
    ax.set_xticks([6, 12, 24])
    ax.grid(alpha=0.25)
    ax.legend()

fig.suptitle(
    "Preliminary Alibaba-derived placement benchmark\n"
    "Own cluster + one paired peer; four tasks per leaf"
)
plt.tight_layout()
plt.savefig(
    results_dir / "meeting_speed_quality.png",
    dpi=180, bbox_inches="tight"
)
plt.show()

display(Markdown("### Solver completion status — all runs"))
status = raw.assign(
    solved=raw["status"].eq("optimal_subproblems")
).groupby(["nodes", "method"], as_index=False).agg(
    attempted=("seed", "count"),
    solved=("solved", "sum")
)
status["not_proven_optimal"] = status["attempted"] - status["solved"]
display(status)

summary.to_csv(
    results_dir / "meeting_speed_quality.csv", index=False
)
status.to_csv(
    results_dir / "meeting_solver_status.csv", index=False
)

In [ ]:
from pathlib import Path

dataset_folder = Path(
    "/kaggle/input/datasets/rohitdurbha/alibaba-cluster-data"
)

files = [
    p for p in dataset_folder.rglob("*")
    if p.is_file() and any(
        name in p.name.lower()
        for name in ["machine_usage", "container_usage", "load_traces"]
    )
]

if files:
    for p in files:
        print(p)
else:
    print("No measured utilization files found.")

In [ ]:
from pathlib import Path
import pandas as pd

telemetry_path = Path(
    "/kaggle/input/datasets/rohitdurbha/"
    "alibaba-cluster-data/machine_usage_bigger.csv"
)

preview = pd.read_csv(
    telemetry_path,
    header=None,
    nrows=5
)

print("Columns:", preview.shape[1])
print(preview.to_string(index=False, header=False))

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import sklearn

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from IPython.display import display

project = Path("/kaggle/working/something_unique")
telemetry_path = Path(
    "/kaggle/input/datasets/rohitdurbha/"
    "alibaba-cluster-data/machine_usage_bigger.csv"
)

# Header is present. Read only needed fields and a bounded sample.
telemetry = pd.read_csv(
    telemetry_path,
    usecols=[
        "machine_id", "time_stamp",
        "cpu_util_percent", "mem_util_percent"
    ],
    nrows=200_000
)

for column in [
    "time_stamp", "cpu_util_percent", "mem_util_percent"
]:
    telemetry[column] = pd.to_numeric(
        telemetry[column], errors="coerce"
    )

telemetry = telemetry.dropna()

telemetry = telemetry.loc[
    np.isfinite(telemetry["time_stamp"])
    & telemetry["cpu_util_percent"].between(0, 100)
    & telemetry["mem_util_percent"].between(0, 100)
].copy()

# Diagnostic subset: up to five machines with sufficient observations.
counts = telemetry["machine_id"].value_counts()
machines = counts[counts >= 100].head(5).index

assert len(machines), "Not enough telemetry for forecasting."

telemetry = telemetry.loc[
    telemetry["machine_id"].isin(machines)
].copy()

# One common chronological boundary across selected machines.
cutoff = float(telemetry["time_stamp"].quantile(0.70))

STEP = 30
WINDOW = 6
HORIZON = 4

X, y, persistence, origins, targets, ids = [], [], [], [], [], []

for machine_id, group in telemetry.groupby("machine_id"):
    group = (
        group.sort_values("time_stamp")
        .drop_duplicates("time_stamp", keep="last")
    )

    first = np.ceil(group["time_stamp"].min() / STEP) * STEP
    last = np.floor(group["time_stamp"].max() / STEP) * STEP

    grid = pd.DataFrame({
        "grid_time": np.arange(first, last + STEP, STEP)
    })

    # At each grid point, use only a reading already observed.
    # Reject readings older than 30 seconds; no future interpolation.
    aligned = pd.merge_asof(
        grid,
        group,
        left_on="grid_time",
        right_on="time_stamp",
        direction="backward",
        tolerance=STEP
    )

    values = aligned[
        ["cpu_util_percent", "mem_util_percent"]
    ].to_numpy(float) / 100.0

    times = aligned["grid_time"].to_numpy()

    for i in range(WINDOW - 1, len(values) - HORIZON):
        history = values[i-WINDOW+1:i+1]
        future = values[i+HORIZON]

        if not np.isfinite(history).all():
            continue
        if not np.isfinite(future).all():
            continue

        X.append(history.ravel())
        y.append(future)
        persistence.append(values[i])
        origins.append(times[i])
        targets.append(times[i+HORIZON])
        ids.append(machine_id)

X = np.asarray(X)
y = np.asarray(y)
persistence = np.asarray(persistence)
origins = np.asarray(origins)
targets = np.asarray(targets)
ids = np.asarray(ids)

assert len(X), "No complete forecasting windows."

# Training targets occur before the split.
# Test forecasts are issued after the split.
train_mask = targets < cutoff
test_mask = origins >= cutoff

assert train_mask.sum() >= 50, "Too few training windows."
assert test_mask.sum() >= 20, "Too few test windows."
assert np.all(targets - origins == 120)

forecaster = RandomForestRegressor(
    n_estimators=60,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=1
)

forecaster.fit(X[train_mask], y[train_mask])
predictions = np.clip(
    forecaster.predict(X[test_mask]), 0, 1
)

actual = y[test_mask]
baseline = persistence[test_mask]

rows = []

for name, estimates in [
    ("Persistence", baseline),
    ("Random Forest", predictions)
]:
    for resource, column in [("CPU", 0), ("Memory", 1)]:
        truth = actual[:, column]
        guess = estimates[:, column]

        actual_overload = truth >= 0.85
        predicted_overload = guess >= 0.85
        true_positive = np.sum(actual_overload & predicted_overload)

        rows.append({
            "model": name,
            "resource": resource,
            "MAE_percentage_points":
                100 * mean_absolute_error(truth, guess),
            "RMSE_percentage_points":
                100 * np.sqrt(mean_squared_error(truth, guess)),
            "actual_overloads": int(actual_overload.sum()),
            "predicted_overloads": int(predicted_overload.sum()),
            "overload_precision": (
                true_positive / predicted_overload.sum()
                if predicted_overload.any() else np.nan
            ),
            "overload_recall": (
                true_positive / actual_overload.sum()
                if actual_overload.any() else np.nan
            )
        })

forecast_results = pd.DataFrame(rows)

(project / "models").mkdir(exist_ok=True)
(project / "results").mkdir(exist_ok=True)

joblib.dump({
    "model": forecaster,
    "window": WINDOW,
    "step_seconds": STEP,
    "horizon_seconds": 120,
    "feature_order": "oldest-to-newest CPU,memory pairs; fractions",
    "maximum_reading_age_seconds": STEP,
    "cutoff": cutoff,
    "machines": list(machines),
    "source": str(telemetry_path),
    "sample_limit": 200_000,
    "sklearn_version": sklearn.__version__,
    "status": "standalone prototype; not integrated into DDQN"
}, project / "models/forecasting_rf_120s.joblib")

forecast_results.to_csv(
    project / "results/forecasting_120s_evaluation.csv",
    index=False
)

print("Machines used:", len(machines))
print("Training windows:", int(train_mask.sum()))
print("Test windows:", int(test_mask.sum()))
print("Forecast horizon: 120 seconds on the monitoring grid")
print("Maximum reading age at a grid point: 30 seconds")
display(forecast_results.round(4))
print("Saved: models/forecasting_rf_120s.joblib")
print("Existing DDQN and forecasting.py were not changed.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import sklearn
from sklearn.ensemble import RandomForestRegressor
from IPython.display import display

project = Path("/kaggle/working/something_unique")
(project / "models").mkdir(parents=True, exist_ok=True)
(project / "results").mkdir(parents=True, exist_ok=True)

source = Path(
    "/kaggle/input/datasets/rohitdurbha/"
    "alibaba-cluster-data/machine_usage_bigger.csv"
)

# Settings: six readings, 30 seconds apart; forecast 120 seconds ahead.
STEP = 30
WINDOW = 6
HORIZON = 4
ROW_LIMIT = 200_000

columns = [
    "machine_id", "time_stamp",
    "cpu_util_percent", "mem_util_percent"
]

print("Reading telemetry...", flush=True)
telemetry = pd.read_csv(
    source,
    usecols=columns,
    nrows=ROW_LIMIT
)

for column in columns[1:]:
    telemetry[column] = pd.to_numeric(
        telemetry[column], errors="coerce"
    )

telemetry = telemetry.dropna(subset=columns).copy()

# FIX: both merge timestamps must have the same dtype.
telemetry["time_stamp"] = telemetry["time_stamp"].astype("float64")

telemetry = telemetry[
    np.isfinite(telemetry["time_stamp"])
    & telemetry["cpu_util_percent"].between(0, 100)
    & telemetry["mem_util_percent"].between(0, 100)
].copy()

counts = telemetry.groupby("machine_id").size()
machines = counts[counts >= 100].nlargest(5).index.tolist()

if not machines:
    raise ValueError("Not enough observations per machine in this sample.")

selected = telemetry[
    telemetry["machine_id"].isin(machines)
].copy()

# One chronological cutoff shared across the selected machines.
cutoff = float(selected["time_stamp"].quantile(0.70))

features = []
targets = []
persistence_predictions = []
origin_times = []
target_times = []
machine_ids = []

print("Preparing historical windows...", flush=True)

for machine_id, group in selected.groupby("machine_id"):
    group = (
        group.sort_values("time_stamp")
        .drop_duplicates("time_stamp", keep="last")
        .copy()
    )

    first = np.ceil(group["time_stamp"].min() / STEP) * STEP
    last = np.floor(group["time_stamp"].max() / STEP) * STEP

    grid = pd.DataFrame({
        "grid_time": np.arange(
            first, last + STEP, STEP, dtype="float64"
        )
    })

    group["time_stamp"] = group["time_stamp"].astype("float64")

    # Only use readings already observed, at most 30 seconds old.
    aligned = pd.merge_asof(
        grid.sort_values("grid_time"),
        group.sort_values("time_stamp"),
        left_on="grid_time",
        right_on="time_stamp",
        direction="backward",
        tolerance=float(STEP)
    )

    values = aligned[
        ["cpu_util_percent", "mem_util_percent"]
    ].to_numpy(dtype=float) / 100.0

    times = aligned["grid_time"].to_numpy()

    for i in range(WINDOW - 1, len(values) - HORIZON):
        history = values[i - WINDOW + 1:i + 1]
        future = values[i + HORIZON]

        if not (
            np.isfinite(history).all()
            and np.isfinite(future).all()
        ):
            continue

        features.append(history.reshape(-1))
        targets.append(future)
        persistence_predictions.append(values[i])
        origin_times.append(times[i])
        target_times.append(times[i + HORIZON])
        machine_ids.append(machine_id)

if not features:
    raise ValueError(
        "No valid windows found. Share this output before changing settings."
    )

X = np.asarray(features)
y = np.asarray(targets)
persistence = np.asarray(persistence_predictions)
origin_times = np.asarray(origin_times)
target_times = np.asarray(target_times)

# Training targets occur before the cutoff.
# Test forecasts originate at or after the cutoff.
train = target_times < cutoff
test = origin_times >= cutoff

print("Training windows:", int(train.sum()))
print("Test windows:", int(test.sum()))

if train.sum() < 50 or test.sum() < 20:
    raise ValueError(
        "Too few chronological training/test windows. "
        "Share the counts above."
    )

assert np.allclose(target_times - origin_times, 120)

print("Training forecasting Random Forest...", flush=True)

model = RandomForestRegressor(
    n_estimators=60,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=1
)

model.fit(X[train], y[train])

rf_predictions = np.clip(model.predict(X[test]), 0, 1)
baseline_predictions = persistence[test]
actual = y[test]

rows = []

for name, predictions in [
    ("Persistence baseline", baseline_predictions),
    ("Random Forest", rf_predictions)
]:
    for index, resource in enumerate(["CPU", "Memory"]):
        errors = predictions[:, index] - actual[:, index]

        actual_overload = actual[:, index] >= 0.85
        predicted_overload = predictions[:, index] >= 0.85
        true_positive = int(
            (actual_overload & predicted_overload).sum()
        )

        predicted_count = int(predicted_overload.sum())
        actual_count = int(actual_overload.sum())

        rows.append({
            "model": name,
            "resource": resource,
            "MAE_percentage_points": 100 * np.abs(errors).mean(),
            "RMSE_percentage_points": 100 * np.sqrt(
                np.mean(errors ** 2)
            ),
            "actual_overloads": actual_count,
            "predicted_overloads": predicted_count,
            "overload_precision": (
                true_positive / predicted_count
                if predicted_count else np.nan
            ),
            "overload_recall": (
                true_positive / actual_count
                if actual_count else np.nan
            )
        })

results = pd.DataFrame(rows)

model_path = project / "models/forecasting_rf_120s.joblib"
results_path = project / "results/forecasting_120s_evaluation.csv"

joblib.dump({
    "model": model,
    "window": WINDOW,
    "step_seconds": STEP,
    "horizon_seconds": STEP * HORIZON,
    "feature_order": (
        "oldest to newest; CPU then memory at each time; fractions 0–1"
    ),
    "maximum_reading_age_seconds": STEP,
    "chronological_cutoff": cutoff,
    "machines": machines,
    "source": str(source),
    "sample_row_limit": ROW_LIMIT,
    "sklearn_version": sklearn.__version__,
    "status": "standalone evaluation; not connected to DDQN"
}, model_path)

results.to_csv(results_path, index=False)

print("\n===== FORECASTING EVALUATION =====")
display(results.round(4))

print("\nForecast horizon: 120 seconds")
print("Machines used:", len(machines))
print("Saved model:", model_path)
print("Saved results:", results_path)
print("Status: trained and evaluated separately; not connected to DDQN.")

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil
import zipfile

project = Path("/kaggle/working/something_unique")
assert project.is_dir(), "Project folder not found."

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_base = Path(f"/kaggle/working/fog_scheduler_backup_{stamp}")

backup = shutil.make_archive(
    str(backup_base),
    "zip",
    root_dir=str(project.parent),
    base_dir=project.name
)

with zipfile.ZipFile(backup) as archive:
    assert archive.testzip() is None, "Backup verification failed."
    assert (
        "something_unique/models/forecasting_rf_120s.joblib"
        in archive.namelist()
    ), "Forecasting model missing."

print("Backup verified:", backup)
print(f"Size: {Path(backup).stat().st_size / 1024**2:.2f} MB")

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

project = Path("/kaggle/working/something_unique")
results = project / "results"

def show_table(title, filename):
    display(Markdown(f"### {title}"))
    path = results / filename
    if path.exists():
        display(pd.read_csv(path).round(3))
    else:
        print("File not found:", filename)

def show_image(filename):
    path = results / filename
    if path.exists():
        display(Image(filename=str(path)))

show_table(
    "1. Flat MIP versus scoped placement — scaling",
    "meeting_speed_quality.csv"
)
show_image("meeting_speed_quality.png")

show_table(
    "2. Solver completion status",
    "meeting_solver_status.csv"
)

show_table(
    "3. Dueling Double DQN — 816-task diagnostic",
    "ddqn_v2_fast_diagnostic_summary.csv"
)

show_table(
    "4. Response time by placement destination",
    "meeting_response_breakdown.csv"
)
show_image("meeting_placement_routes.png")

show_table(
    "5. CPU and memory forecasting — 120 seconds ahead",
    "forecasting_120s_evaluation.csv"
)

show_table(
    "6. Alibaba execution-time prediction",
    "alibaba_runtime_evaluation.csv"
)

In [2]:
from pathlib import Path
from datetime import datetime
import shutil
import zipfile

project = Path("/kaggle/working/something_unique")
assert project.is_dir(), "Project folder not found."

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
working = Path("/kaggle/working")

# A. Full project backup: code, models, data and results.
full_zip = shutil.make_archive(
    str(working / f"fog_project_backup_{stamp}"),
    "zip",
    root_dir=str(project.parent),
    base_dir=project.name
)

# B. Results bundle for the professor.
bundle = working / f"fog_results_{stamp}"
bundle.mkdir()

results = project / "results"
assert results.is_dir(), "Results folder not found."
shutil.copytree(results, bundle / "results")

(bundle / "READ_ME.txt").write_text(
    """Preliminary fog scheduling results

Start with:
1. meeting_speed_quality.png / .csv
2. meeting_solver_status.csv
3. ddqn_v2_fast_diagnostic_summary.csv
4. meeting_response_breakdown.csv
5. meeting_placement_routes.png
6. forecasting_120s_evaluation.csv
7. alibaba_runtime_evaluation.csv

Important interpretation:
- The scaling benchmark, DDQN diagnostic and new forecasting evaluation
  are separate experiments, not one fully integrated evaluation.
- Scaling comparisons use identical inputs within each paired trial.
- The 24-node paired summary uses two completed pairs.
  One additional flat-MIP run timed out.
- Cloud leftovers in the scaling benchmark are unassigned fog tasks,
  not measured cloud execution results.
- DDQN v2 fast diagnostic files are the latest diagnostic results.
  Earlier DDQN files are retained for historical reference.
- Deadlines, network delays, cloud speed and fog capacities are modeled.
- The new forecasting model is not yet connected to DDQN.
""",
    encoding="utf-8"
)

results_zip = shutil.make_archive(
    str(bundle),
    "zip",
    root_dir=str(bundle.parent),
    base_dir=bundle.name
)

for filename in [full_zip, results_zip]:
    with zipfile.ZipFile(filename) as archive:
        assert archive.testzip() is None
    size = Path(filename).stat().st_size / 1024**2
    print(f"Verified: {filename}\nSize: {size:.2f} MB\n")

AssertionError: Project folder not found.

In [3]:
from pathlib import Path

for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
    print(f"\n===== {root} =====")

    if not root.exists():
        continue

    print("Top-level contents:")
    for path in sorted(root.iterdir()):
        print(path)

    print("\nPossible project folders:")
    for path in root.rglob("fresh_comparison_tasks.json"):
        print(path.parent.parent)

    print("\nAvailable ZIP checkpoints:")
    for path in root.rglob("*.zip"):
        print(path)


===== /kaggle/working =====
Top-level contents:
/kaggle/working/.virtual_documents

Possible project folders:

Available ZIP checkpoints:

===== /kaggle/input =====
Top-level contents:
/kaggle/input/datasets

Possible project folders:
/kaggle/input/datasets/satvikiatakani/checkpoint-22/Checkpoint 22

Available ZIP checkpoints:


In [4]:
from pathlib import Path
import shutil

source = Path(
    "/kaggle/input/datasets/satvikiatakani/"
    "checkpoint-22/Checkpoint 22"
)
project = Path("/kaggle/working/something_unique")

assert source.is_dir(), f"Source folder not found: {source}"

if project.exists():
    print("Working folder already exists; leaving its files unchanged.")
else:
    shutil.copytree(source, project)
    print("Project copied successfully.")

required = [
    "data/fresh_comparison_tasks.json",
    "models/alibaba_runtime_rf.joblib",
    "finite_cloud.py",
    "ddqn_environment_finite_cloud.py",
]

missing = [
    name for name in required
    if not (project / name).is_file()
]

assert not missing, f"Missing required files: {missing}"

print("Passed: replay data, runtime model and finite-cloud files found.")
print("Working project:", project)

Project copied successfully.
Passed: replay data, runtime model and finite-cloud files found.
Working project: /kaggle/working/something_unique


In [5]:
from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd
import shutil
import zipfile
from IPython.display import display

project = Path("/kaggle/working/something_unique")
audit_dir = project / "results/week1_audit"
audit_dir.mkdir(exist_ok=True)

source = Path(
    "/kaggle/input/datasets/rohitdurbha/"
    "alibaba-cluster-data/machine_meta.csv"
)

columns = [
    "machine_id",
    "time_stamp",
    "failure_domain_1",
    "failure_domain_2",
    "cpu_num",
    "mem_size",
    "status"
]

metadata = pd.read_csv(
    source,
    header=None,
    names=columns
)

for column in ["time_stamp", "cpu_num", "mem_size"]:
    metadata[column] = pd.to_numeric(
        metadata[column], errors="coerce"
    )

metadata = metadata.dropna(
    subset=["machine_id", "time_stamp", "cpu_num", "mem_size"]
)

definition = json.loads(
    (project / "data/fresh_comparison_tasks.json").read_text()
)
replay_start = float(definition["original_start"])

# Use only metadata known when the replay begins.
snapshot = (
    metadata[metadata["time_stamp"] <= replay_start]
    .sort_values("time_stamp", kind="stable")
    .drop_duplicates("machine_id", keep="last")
    .copy()
)

snapshot["status"] = (
    snapshot["status"].astype(str).str.strip().str.upper()
)

usable = snapshot[
    snapshot["status"].eq("USING")
    & snapshot["cpu_num"].gt(0)
    & snapshot["mem_size"].gt(0)
    & snapshot["mem_size"].le(100)
].copy()

assert not usable.empty, (
    "No usable machines found. Send this message before continuing."
)

configurations = (
    usable.groupby(["cpu_num", "mem_size"])
    .size()
    .reset_index(name="machines")
    .sort_values(
        ["machines", "cpu_num", "mem_size"],
        ascending=[False, True, True]
    )
    .reset_index(drop=True)
)

print("Metadata snapshot time:", replay_start)
print("Machines with metadata by that time:", len(snapshot))
print("Usable machines:", len(usable))

print("\n===== MACHINE RESOURCE CONFIGURATIONS =====")
display(configurations)

# Use the most common pair of CPU and memory capacities.
reference = configurations.iloc[0]
cpu_per_machine = float(reference["cpu_num"])
memory_per_machine = float(reference["mem_size"])

# Check that individual replay tasks fit in the main allocation.
tasks = pd.DataFrame(definition["tasks"])

assert tasks["cpu_request"].max() <= cpu_per_machine + 1e-9
assert tasks["mem_request"].max() <= memory_per_machine + 1e-9

capacity_rows = []

for equivalents in [1, 2, 4]:
    capacity_rows.append({
        "machine_equivalents": equivalents,
        "cloud_cpu_capacity": equivalents * cpu_per_machine,
        "cloud_memory_capacity": equivalents * memory_per_machine,
        "role": (
            "main_configuration"
            if equivalents == 1
            else "capacity_sensitivity"
        )
    })

capacity_table = pd.DataFrame(capacity_rows)

config = {
    "resource_model": (
        "Finite pooled cloud allocation for this application"
    ),
    "main_machine_equivalents": 1,
    "reference_cpu_capacity": cpu_per_machine,
    "reference_memory_capacity": memory_per_machine,
    "capacity_scenarios": capacity_rows,
    "cpu_units": (
        "Trace CPU units; task plan_cpu divided by 100"
    ),
    "memory_units": (
        "Normalized Alibaba memory units; task plan_mem retained"
    ),
    "reference_selection": (
        "Most common CPU/memory configuration among USING machines "
        "at or before the replay start"
    ),
    "metadata_snapshot_time": replay_start,
    "capacity_source": str(source),
    "schema_source": (
        "https://github.com/alibaba/clusterdata/blob/master/"
        "cluster-trace-v2018/schema.txt"
    ),
    "resource_model_reference": (
        "Calheiros et al., CloudSim: A Novel Framework for Modeling "
        "and Simulation of Cloud Computing Infrastructures and Services, "
        "https://arxiv.org/abs/0903.2525"
    ),
    "allocation_size_justification": (
        "One trace-machine-equivalent main allocation, with two and "
        "four equivalents predeclared for sensitivity analysis. "
        "Allocation counts are experimental choices, not literature values."
    ),
    "limitations": [
        "Pooled resources do not reproduce per-host fragmentation.",
        "Memory is normalized, not GB.",
        "This does not establish hardware-equivalent execution speed."
    ],
    "status": (
        "Capacity configuration prepared; real-workload evaluation pending"
    )
}

config_path = project / "data/week1_cloud_capacity_config.json"
config_path.write_text(
    json.dumps(config, indent=2),
    encoding="utf-8"
)

configurations.to_csv(
    audit_dir / "machine_capacity_reference.csv", index=False
)
capacity_table.to_csv(
    audit_dir / "planned_cloud_capacity_scenarios.csv", index=False
)

print("\n===== PREDECLARED CLOUD CAPACITIES =====")
display(capacity_table)

print("\nSaved:", config_path)
print("Passed: every individual replay task fits the main cloud allocation.")
print("No deadlines, execution-speed factors or network settings changed.")
print("No workload simulation or training was run.")

# Save a checkpoint containing the configuration and evidence.
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = shutil.make_archive(
    f"/kaggle/working/fog_week1_checkpoint_{stamp}",
    "zip",
    root_dir=str(project.parent),
    base_dir=project.name
)

with zipfile.ZipFile(backup) as archive:
    assert archive.testzip() is None

print("\nVerified checkpoint:", backup)

Metadata snapshot time: 89507.0
Machines with metadata by that time: 3955
Usable machines: 3955

===== MACHINE RESOURCE CONFIGURATIONS =====


,cpu_num,mem_size,machines
0,96,100,3955



===== PREDECLARED CLOUD CAPACITIES =====


,machine_equivalents,cloud_cpu_capacity,cloud_memory_capacity,role
0,1,96.0,100.0,main_configuration
1,2,192.0,200.0,capacity_sensitivity
2,4,384.0,400.0,capacity_sensitivity



Saved: /kaggle/working/something_unique/data/week1_cloud_capacity_config.json
Passed: every individual replay task fits the main cloud allocation.
No deadlines, execution-speed factors or network settings changed.
No workload simulation or training was run.

Verified checkpoint: /kaggle/working/fog_week1_checkpoint_20261103_105850.zip


In [6]:
from pathlib import Path
from datetime import datetime
import sys
import json
import joblib
import numpy as np
import pandas as pd
import shutil
import zipfile
from IPython.display import display

project = Path("/kaggle/working/something_unique")
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from ddqn_environment_finite_cloud import FiniteCloudFogEnv

# ---------- Load the saved configuration and workload ----------

config = json.loads(
    (project / "data/week1_cloud_capacity_config.json").read_text()
)
definition = json.loads(
    (project / "data/fresh_comparison_tasks.json").read_text()
)
tasks = definition["tasks"]

main = next(
    row for row in config["capacity_scenarios"]
    if row["role"] == "main_configuration"
)

bundle = joblib.load(project / "models/alibaba_runtime_rf.joblib")
runtime_model = bundle["model"]
runtime_model.set_params(n_jobs=1)

# Cache predictions to avoid repeatedly querying the RF.
features = (
    pd.DataFrame(tasks)[["cpu_request", "mem_request"]]
    .drop_duplicates()
)
lookup = dict(zip(
    map(tuple, features.to_numpy(dtype=float)),
    runtime_model.predict(features)
))

class CachedRuntime:
    def predict(self, frame):
        return np.array([
            lookup[tuple(row)]
            for row in frame[
                ["cpu_request", "mem_request"]
            ].to_numpy(dtype=float)
        ])

# ---------- Run cloud-only; no training ----------

env = FiniteCloudFogEnv(
    CachedRuntime(),
    cloud_cpu_capacity=main["cloud_cpu_capacity"],
    cloud_memory_capacity=main["cloud_memory_capacity"],
    cloud_runtime_factor=0.3
)

env.reset(tasks)
decisions = 0

print(
    f"Running {len(tasks):,} tasks with "
    f"{main['cloud_cpu_capacity']:g} CPU units and "
    f"{main['cloud_memory_capacity']:g} normalized memory units.",
    flush=True
)

while not env.done:
    env.step(3)  # Cloud action
    decisions += 1

    if decisions % 1000 == 0:
        print(f"Placed {decisions:,}/{len(tasks):,} tasks", flush=True)

completed = pd.DataFrame(env.completed)
history = pd.DataFrame(env.cloud.history)

# ---------- Verify correctness ----------

assert len(completed) == len(tasks)
assert completed["id"].nunique() == len(tasks)
assert completed["route"].eq("cloud").all()
assert decisions == len(tasks)

assert completed["cloud_queue_wait_s"].ge(-1e-6).all()
assert history["used_cpu"].le(
    main["cloud_cpu_capacity"] + 1e-6
).all()
assert history["used_memory"].le(
    main["cloud_memory_capacity"] + 1e-6
).all()

execution = completed["finish_time"] - completed["start_time"]
decision_wait = (
    completed["placement_time"] - completed["arrival_time"]
)

assert np.allclose(
    completed["response_s"],
    decision_wait
    + completed["transfer_s"]
    + completed["cloud_queue_wait_s"]
    + execution
)
assert np.allclose(
    execution,
    0.3 * completed["reference_runtime_s"]
)
assert (
    completed["deadline_met"]
    == (completed["finish_time"] <= completed["deadline"])
).all()

assert not env.cloud.active
assert not env.cloud.queue
assert abs(env.cloud.used_cpu) <= 1e-6
assert abs(env.cloud.used_memory) <= 1e-6

# ---------- Overall and per-class results ----------

rows = []

for label in ["overall", "urgent", "heavy_compute", "can_wait"]:
    group = (
        completed if label == "overall"
        else completed[completed["label"].eq(label)]
    )
    if group.empty:
        continue

    rows.append({
        "class": label,
        "tasks": len(group),
        "mean_response_s": group["response_s"].mean(),
        "p95_response_s": group["response_s"].quantile(0.95),
        "deadline_met_pct": 100 * group["deadline_met"].mean(),
        "deadline_misses": int((~group["deadline_met"]).sum()),
        "mean_cloud_queue_s": group["cloud_queue_wait_s"].mean(),
        "p95_cloud_queue_s": group["cloud_queue_wait_s"].quantile(0.95),
        "queued_tasks_pct": (
            100 * group["cloud_queue_wait_s"].gt(1e-6).mean()
        )
    })

summary = pd.DataFrame(rows)

load = pd.DataFrame([{
    "cloud_tasks": len(completed),
    "wan_upload_mb": completed["data_size_mb"].sum(),
    "cloud_execution_seconds": execution.sum(),
    "cloud_cpu_request_seconds": (
        execution * completed["cpu_request"]
    ).sum(),
    "cloud_memory_request_seconds": (
        execution * completed["mem_request"]
    ).sum(),
    "peak_reserved_cpu": history["used_cpu"].max(),
    "peak_reserved_memory": history["used_memory"].max(),
    "peak_queued_tasks": history["queued_tasks"].max()
}])

print("\n===== FINITE CLOUD-ONLY: RESPONSE AND DEADLINES =====")
display(summary.round(3))

print("\n===== CLOUD LOAD AND WAN TRAFFIC =====")
display(load.round(3))

print("\nPassed: capacity limits, response decomposition and resource release.")
print("All tasks completed; deadline satisfaction is reported separately.")
print("One existing development interval; no training or multi-seed evaluation.")
print("Resource-time values are load proxies, not monetary costs.")

# ---------- Save results and checkpoint ----------

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out = project / "results/week1_audit" / f"finite_cloud_only_{stamp}"
out.mkdir(parents=True, exist_ok=False)

completed.to_csv(out / "tasks.csv", index=False)
history.to_csv(out / "cloud_resource_history.csv", index=False)
summary.to_csv(out / "response_by_class.csv", index=False)
load.to_csv(out / "cloud_load.csv", index=False)

(out / "settings.json").write_text(json.dumps({
    "policy": "cloud_only",
    "capacity": main,
    "cloud_runtime_factor": 0.3,
    "cloud_transfer_formula": "4 + 8 * data_size_mb / 20",
    "deadlines": "unchanged from saved workload",
    "workload": "data/fresh_comparison_tasks.json",
    "original_start": definition["original_start"],
    "tasks": len(tasks),
    "evaluation_intervals": 1,
    "training_run": False
}, indent=2))

print("\nResults saved:", out)

backup = shutil.make_archive(
    f"/kaggle/working/fog_week1_checkpoint_{stamp}",
    "zip",
    root_dir=str(project.parent),
    base_dir=project.name
)
with zipfile.ZipFile(backup) as archive:
    assert archive.testzip() is None

print("Verified checkpoint:", backup)

Running 5,871 tasks with 96 CPU units and 100 normalized memory units.
Placed 1,000/5,871 tasks
Placed 2,000/5,871 tasks
Placed 3,000/5,871 tasks
Placed 4,000/5,871 tasks
Placed 5,000/5,871 tasks

===== FINITE CLOUD-ONLY: RESPONSE AND DEADLINES =====


,class,tasks,mean_response_s,p95_response_s,deadline_met_pct,deadline_misses,mean_cloud_queue_s,p95_cloud_queue_s,queued_tasks_pct
0,overall,5871,2188.153,6696.750,59.854,2357,2106.763,6604.000,79.458
1,urgent,1148,2319.059,6697.285,58.972,471,2193.905,6604.000,79.530
2,heavy_compute,2795,2101.191,6727.870,60.358,1108,2015.957,6688.790,82.182
3,can_wait,1928,2236.275,6623.260,59.647,778,2186.516,6561.595,75.467



===== CLOUD LOAD AND WAN TRAFFIC =====


,cloud_tasks,wan_upload_mb,cloud_execution_seconds,cloud_cpu_request_seconds,cloud_memory_request_seconds,peak_reserved_cpu,peak_reserved_memory,peak_queued_tasks
0,5871,234840.0,360421.2,1010975.235,466444.455,96.0,49.17,1914



Passed: capacity limits, response decomposition and resource release.
All tasks completed; deadline satisfaction is reported separately.
One existing development interval; no training or multi-seed evaluation.
Resource-time values are load proxies, not monetary costs.

Results saved: /kaggle/working/something_unique/results/week1_audit/finite_cloud_only_20261103_110159
Verified checkpoint: /kaggle/working/fog_week1_checkpoint_20261103_110159.zip


In [7]:
from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd
import shutil
import zipfile
from IPython.display import display

# Verify that the preceding cloud-only cell has been run.
required_variables = ["tasks", "main", "CachedRuntime", "FiniteCloudFogEnv"]
missing = [name for name in required_variables if name not in globals()]
assert not missing, (
    f"Missing session variables: {missing}. "
    "Rerun the preceding cloud-only cell first."
)

project = Path("/kaggle/working/something_unique")

# Use the exact cloud-only results you just produced.
cloud_run = (
    project / "results/week1_audit"
    / "finite_cloud_only_20261103_110159"
)
cloud_tasks = pd.read_csv(cloud_run / "tasks.csv")
cloud_summary = pd.read_csv(cloud_run / "response_by_class.csv")
cloud_load = pd.read_csv(cloud_run / "cloud_load.csv")

# Confirm matching input tasks, rather than just matching task counts.
original = pd.DataFrame(tasks).set_index("id").sort_index()
previous = cloud_tasks.set_index("id").sort_index()

assert original.index.equals(previous.index)
for column in [
    "arrival_time", "cpu_request", "mem_request",
    "reference_runtime_s", "data_size_mb", "deadline", "origin_node"
]:
    assert np.allclose(original[column], previous[column])

assert original["label"].equals(previous["label"])

# ---------- Local-first, same finite cloud ----------

env_local = FiniteCloudFogEnv(
    CachedRuntime(),
    cloud_cpu_capacity=main["cloud_cpu_capacity"],
    cloud_memory_capacity=main["cloud_memory_capacity"],
    cloud_runtime_factor=0.3
)

env_local.reset(tasks)
decisions = 0

print(f"Running local-first on {len(tasks):,} identical tasks.", flush=True)

while not env_local.done:
    valid = env_local.valid_actions()

    # 0: origin; 1: siblings; 2: other cluster; 3: cloud.
    action = next(a for a in [0, 1, 2, 3] if a in valid)
    env_local.step(action)
    decisions += 1

    if decisions % 1000 == 0:
        print(f"Placed {decisions:,}/{len(tasks):,} tasks", flush=True)

completed = pd.DataFrame(env_local.completed)
history = pd.DataFrame(env_local.cloud.history)
cloud = completed[completed["route"].eq("cloud")].copy()

# ---------- Correctness checks ----------

assert len(completed) == len(tasks)
assert completed["id"].nunique() == len(tasks)
assert decisions == len(tasks)
assert completed["holds"].eq(0).all()

execution = completed["finish_time"] - completed["start_time"]
queue_wait = (
    completed["start_time"]
    - completed["placement_time"]
    - completed["transfer_s"]
)

assert queue_wait.ge(-1e-6).all()
assert np.allclose(
    completed["response_s"],
    completed["placement_time"] - completed["arrival_time"]
    + completed["transfer_s"] + queue_wait + execution
)
assert np.allclose(
    execution,
    completed["reference_runtime_s"]
    * np.where(completed["route"].eq("cloud"), 0.3, 1.0)
)
assert (
    completed["deadline_met"]
    == (completed["finish_time"] <= completed["deadline"])
).all()

if not history.empty:
    assert history["used_cpu"].le(
        main["cloud_cpu_capacity"] + 1e-6
    ).all()
    assert history["used_memory"].le(
        main["cloud_memory_capacity"] + 1e-6
    ).all()

assert not env_local.cloud.active
assert not env_local.cloud.queue
assert np.allclose(env_local.cpu, 0)
assert np.allclose(env_local.mem, 0)

# ---------- Overall and per-class comparison ----------

rows = []

for label in ["overall", "urgent", "heavy_compute", "can_wait"]:
    group = (
        completed if label == "overall"
        else completed[completed["label"].eq(label)]
    )
    if group.empty:
        continue

    rows.append({
        "class": label,
        "tasks": len(group),
        "mean_response_s": group["response_s"].mean(),
        "p95_response_s": group["response_s"].quantile(0.95),
        "deadline_met_pct": 100 * group["deadline_met"].mean(),
        "deadline_misses": int((~group["deadline_met"]).sum())
    })

local_summary = pd.DataFrame(rows)

columns = [
    "class", "tasks", "mean_response_s",
    "p95_response_s", "deadline_met_pct", "deadline_misses"
]
comparison = pd.concat([
    cloud_summary[columns].assign(policy="cloud_only"),
    local_summary[columns].assign(policy="local_first")
], ignore_index=True)

print("\n===== SAME-WORKLOAD POLICY COMPARISON =====")
display(comparison[
    ["policy"] + columns
].round(3))

# ---------- Cloud demand and traffic ----------

cloud_execution = cloud["finish_time"] - cloud["start_time"]

local_load = pd.DataFrame([{
    "cloud_tasks": len(cloud),
    "wan_upload_mb": cloud["data_size_mb"].sum(),
    "cloud_execution_seconds": cloud_execution.sum(),
    "cloud_cpu_request_seconds": (
        cloud_execution * cloud["cpu_request"]
    ).sum(),
    "cloud_memory_request_seconds": (
        cloud_execution * cloud["mem_request"]
    ).sum(),
    "peak_reserved_cpu": (
        history["used_cpu"].max() if not history.empty else 0
    ),
    "peak_reserved_memory": (
        history["used_memory"].max() if not history.empty else 0
    ),
    "peak_queued_tasks": (
        history["queued_tasks"].max() if not history.empty else 0
    )
}])

load_comparison = pd.concat([
    cloud_load.assign(policy="cloud_only"),
    local_load.assign(policy="local_first")
], ignore_index=True)

print("\n===== CLOUD LOAD AND WAN COMPARISON =====")
display(load_comparison.round(3))

routes = (
    completed["route"].value_counts()
    .rename_axis("route").reset_index(name="tasks")
)
print("\n===== LOCAL-FIRST ROUTES =====")
display(routes)

print("\nPassed: identical inputs, completion counts and response checks.")
print("Local-first used zero holds.")
print("One development interval; no DDQN training performed.")

# ---------- Save results and checkpoint ----------

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out = project / "results/week1_audit" / f"finite_local_first_{stamp}"
out.mkdir(parents=True, exist_ok=False)

completed.to_csv(out / "tasks.csv", index=False)
history.to_csv(out / "cloud_resource_history.csv", index=False)
local_summary.to_csv(out / "response_by_class.csv", index=False)
local_load.to_csv(out / "cloud_load.csv", index=False)
comparison.to_csv(out / "policy_comparison.csv", index=False)
load_comparison.to_csv(out / "load_comparison.csv", index=False)
routes.to_csv(out / "routes.csv", index=False)

(out / "settings.json").write_text(json.dumps({
    "policy": "local_first",
    "action_priority": [0, 1, 2, 3],
    "capacity": main,
    "cloud_runtime_factor": 0.3,
    "cloud_transfer_formula": "4 + 8 * data_size_mb / 20",
    "deadlines": "unchanged",
    "cloud_only_comparison_directory": str(cloud_run),
    "evaluation_intervals": 1,
    "training_run": False
}, indent=2))

backup = shutil.make_archive(
    f"/kaggle/working/fog_week1_checkpoint_{stamp}",
    "zip",
    root_dir=str(project.parent),
    base_dir=project.name
)
with zipfile.ZipFile(backup) as archive:
    assert archive.testzip() is None

print("\nResults saved:", out)
print("Verified checkpoint:", backup)

Running local-first on 5,871 identical tasks.
Placed 1,000/5,871 tasks
Placed 2,000/5,871 tasks
Placed 3,000/5,871 tasks
Placed 4,000/5,871 tasks
Placed 5,000/5,871 tasks

===== SAME-WORKLOAD POLICY COMPARISON =====


,policy,class,tasks,mean_response_s,p95_response_s,deadline_met_pct,deadline_misses
0,cloud_only,overall,5871,2188.153,6696.750,59.854,2357
1,cloud_only,urgent,1148,2319.059,6697.285,58.972,471
2,cloud_only,heavy_compute,2795,2101.191,6727.870,60.358,1108
3,cloud_only,can_wait,1928,2236.275,6623.260,59.647,778
4,local_first,overall,5871,2045.774,6606.450,69.784,1774
5,local_first,urgent,1148,2135.950,6599.785,69.077,355
6,local_first,heavy_compute,2795,2057.692,6658.400,69.553,851
7,local_first,can_wait,1928,1974.804,6530.915,70.539,568



===== CLOUD LOAD AND WAN COMPARISON =====


,cloud_tasks,wan_upload_mb,cloud_execution_seconds,cloud_cpu_request_seconds,cloud_memory_request_seconds,peak_reserved_cpu,peak_reserved_memory,peak_queued_tasks,policy
0,5871,234840.0,360421.2,1010975.235,466444.455,96.0,49.17,1914,cloud_only
1,5041,201640.0,315276.0,1000297.125,461193.789,96.0,45.10,1770,local_first



===== LOCAL-FIRST ROUTES =====


,route,tasks
0,cloud,5041
1,local,400
2,sibling,228
3,cross_cluster,202



Passed: identical inputs, completion counts and response checks.
Local-first used zero holds.
One development interval; no DDQN training performed.

Results saved: /kaggle/working/something_unique/results/week1_audit/finite_local_first_20261103_110622
Verified checkpoint: /kaggle/working/fog_week1_checkpoint_20261103_110622.zip


In [8]:
from pathlib import Path
import pandas as pd
from IPython.display import display

run = Path(
    "/kaggle/working/something_unique/results/week1_audit/"
    "finite_local_first_20261103_110622"
)

print("===== RESPONSE TIME AND DEADLINE COMPARISON =====")
display(pd.read_csv(run / "policy_comparison.csv").round(3))

===== RESPONSE TIME AND DEADLINE COMPARISON =====


,class,tasks,mean_response_s,p95_response_s,deadline_met_pct,deadline_misses,policy
0,overall,5871,2188.153,6696.750,59.854,2357,cloud_only
1,urgent,1148,2319.059,6697.285,58.972,471,cloud_only
2,heavy_compute,2795,2101.191,6727.870,60.358,1108,cloud_only
3,can_wait,1928,2236.275,6623.260,59.647,778,cloud_only
4,overall,5871,2045.774,6606.450,69.784,1774,local_first
5,urgent,1148,2135.950,6599.785,69.077,355,local_first
6,heavy_compute,2795,2057.692,6658.400,69.553,851,local_first
7,can_wait,1928,1974.804,6530.915,70.539,568,local_first


In [9]:
from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd
import shutil
import zipfile
from IPython.display import display

project = Path("/kaggle/working/something_unique")

required = ["CachedRuntime", "FiniteCloudFogEnv", "lookup"]
assert all(name in globals() for name in required), (
    "Session variables are missing. Stop here and send this message."
)

config = json.loads(
    (project / "data/week1_cloud_capacity_config.json").read_text()
)
tasks = json.loads(
    (project / "data/fresh_comparison_tasks.json").read_text()
)["tasks"]

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out = (
    project / "results/week1_audit"
    / f"finite_capacity_sensitivity_{stamp}"
)
out.mkdir(parents=True, exist_ok=False)

rows = []

for capacity in config["capacity_scenarios"]:
    equivalents = capacity["machine_equivalents"]
    if equivalents == 1:
        continue  # Already evaluated and saved.

    for policy in ["cloud_only", "local_first"]:
        print(
            f"\nRunning {policy}: {equivalents} machine equivalents",
            flush=True
        )

        env = FiniteCloudFogEnv(
            CachedRuntime(),
            cloud_cpu_capacity=capacity["cloud_cpu_capacity"],
            cloud_memory_capacity=capacity["cloud_memory_capacity"],
            cloud_runtime_factor=0.3
        )
        env.reset(tasks)

        decisions = 0
        while not env.done:
            action = (
                3 if policy == "cloud_only"
                else next(
                    a for a in [0, 1, 2, 3]
                    if a in env.valid_actions()
                )
            )
            env.step(action)
            decisions += 1

        completed = pd.DataFrame(env.completed)
        history = pd.DataFrame(env.cloud.history)

        assert len(completed) == len(tasks)
        assert completed["id"].nunique() == len(tasks)
        assert decisions == len(tasks)
        assert completed["holds"].eq(0).all()

        execution = completed["finish_time"] - completed["start_time"]
        waiting = (
            completed["start_time"]
            - completed["placement_time"]
            - completed["transfer_s"]
        )
        assert waiting.ge(-1e-6).all()
        assert np.allclose(
            completed["response_s"],
            completed["placement_time"] - completed["arrival_time"]
            + completed["transfer_s"] + waiting + execution
        )
        assert np.allclose(
            execution,
            completed["reference_runtime_s"]
            * np.where(completed["route"].eq("cloud"), 0.3, 1.0)
        )
        assert (
            completed["deadline_met"]
            == (completed["finish_time"] <= completed["deadline"])
        ).all()

        if not history.empty:
            assert history["used_cpu"].le(
                capacity["cloud_cpu_capacity"] + 1e-6
            ).all()
            assert history["used_memory"].le(
                capacity["cloud_memory_capacity"] + 1e-6
            ).all()

        assert not env.cloud.active
        assert not env.cloud.queue
        assert abs(env.cloud.used_cpu) <= 1e-6
        assert abs(env.cloud.used_memory) <= 1e-6
        assert np.allclose(env.cpu, 0)
        assert np.allclose(env.mem, 0)

        for label in ["overall", "urgent", "heavy_compute", "can_wait"]:
            group = (
                completed if label == "overall"
                else completed[completed["label"].eq(label)]
            )
            if group.empty:
                continue

            cloud = group[group["route"].eq("cloud")]
            cloud_execution = cloud["finish_time"] - cloud["start_time"]

            rows.append({
                "machine_equivalents": equivalents,
                "policy": policy,
                "class": label,
                "tasks": len(group),
                "mean_response_s": group["response_s"].mean(),
                "p95_response_s": group["response_s"].quantile(0.95),
                "deadline_met_pct": 100 * group["deadline_met"].mean(),
                "deadline_misses": int((~group["deadline_met"]).sum()),
                "fog_tasks": int(group["route"].ne("cloud").sum()),
                "cloud_tasks": len(cloud),
                "wan_upload_mb": cloud["data_size_mb"].sum(),
                "cloud_cpu_request_seconds": (
                    cloud_execution * cloud["cpu_request"]
                ).sum(),
                "mean_cloud_queue_s": (
                    cloud["cloud_queue_wait_s"].mean()
                    if len(cloud) else 0.0
                )
            })

        prefix = f"{equivalents}x_{policy}"
        completed.to_csv(out / f"{prefix}_tasks.csv", index=False)
        history.to_csv(out / f"{prefix}_cloud_history.csv", index=False)

        # Save after each run so completed work is retained.
        pd.DataFrame(rows).to_csv(out / "summary.csv", index=False)
        print("Passed checks and saved:", prefix, flush=True)

summary = pd.DataFrame(rows)

print("\n===== CAPACITY SENSITIVITY: OVERALL AND URGENT =====")
display(summary[
    summary["class"].isin(["overall", "urgent"])
].round(3))

(out / "settings.json").write_text(json.dumps({
    "capacity_scenarios": config["capacity_scenarios"],
    "policies": ["cloud_only", "local_first"],
    "cloud_runtime_factor": 0.3,
    "network_and_deadlines": "unchanged",
    "evaluation_intervals": 1,
    "training_run": False
}, indent=2))

backup = shutil.make_archive(
    f"/kaggle/working/fog_week1_checkpoint_{stamp}",
    "zip",
    root_dir=str(project.parent),
    base_dir=project.name
)
with zipfile.ZipFile(backup) as archive:
    assert archive.testzip() is None

print("\nResults saved:", out)
print("Verified checkpoint:", backup)


Running cloud_only: 2 machine equivalents
Passed checks and saved: 2x_cloud_only

Running local_first: 2 machine equivalents
Passed checks and saved: 2x_local_first

Running cloud_only: 4 machine equivalents
Passed checks and saved: 4x_cloud_only

Running local_first: 4 machine equivalents
Passed checks and saved: 4x_local_first

===== CAPACITY SENSITIVITY: OVERALL AND URGENT =====


,machine_equivalents,policy,class,tasks,mean_response_s,p95_response_s,deadline_met_pct,deadline_misses,fog_tasks,cloud_tasks,wan_upload_mb,cloud_cpu_request_seconds,mean_cloud_queue_s
0,2,cloud_only,overall,5871,129.823,334.250,72.253,1629,0,5871,234840.0,1010975.235,48.433
1,2,cloud_only,urgent,1148,177.285,325.655,71.864,323,0,1148,45920.0,293637.690,52.131
4,2,local_first,overall,5871,131.289,373.000,80.106,1168,830,5041,201640.0,1000297.125,39.447
5,2,local_first,urgent,1148,175.574,373.000,78.833,243,163,985,39400.0,291842.685,41.576
8,4,cloud_only,overall,5871,83.672,133.400,98.177,107,0,5871,234840.0,1010975.235,2.282
9,4,cloud_only,urgent,1148,127.593,134.365,97.909,24,0,1148,45920.0,293637.690,2.438
12,4,local_first,overall,5871,99.183,317.450,99.029,57,830,5041,201640.0,1000297.125,2.054
13,4,local_first,urgent,1148,141.696,372.650,99.303,8,163,985,39400.0,291842.685,2.091



Results saved: /kaggle/working/something_unique/results/week1_audit/finite_capacity_sensitivity_20261103_110944
Verified checkpoint: /kaggle/working/fog_week1_checkpoint_20261103_110944.zip


In [10]:
import inspect

assert "FiniteCloudFogEnv" in globals(), (
    "FiniteCloudFogEnv is not loaded. Send this message."
)
assert "CachedRuntime" in globals(), (
    "CachedRuntime is not loaded. Send this message."
)

check_env = FiniteCloudFogEnv(
    CachedRuntime(),
    cloud_cpu_capacity=96.0,
    cloud_memory_capacity=100.0,
    cloud_runtime_factor=0.3
)

print("===== CURRENT HOLD SETTINGS =====")
print("Maximum holds per task:", check_env.max_holds)
print("Seconds per hold:", check_env.hold_seconds)

print("\n===== COMPLETION REWARD =====")
print(inspect.getsource(type(check_env)._record))

print("\n===== ACTION REWARD AND HOLD CODE =====")

# Inspect each implementation in the inheritance chain.
for cls in type(check_env).__mro__:
    if "step" not in cls.__dict__:
        continue

    print(f"\n--- {cls.__module__}.{cls.__name__}.step ---")
    lines = inspect.getsource(cls.__dict__["step"]).splitlines()

    selected = set()
    for i, line in enumerate(lines):
        if any(term in line for term in [
            "reward", "holds", "hold_seconds", "action == 4"
        ]):
            selected.update(
                range(max(0, i - 1), min(len(lines), i + 3))
            )

    for i in sorted(selected):
        print(lines[i])

===== CURRENT HOLD SETTINGS =====
Maximum holds per task: 3
Seconds per hold: 1.0

===== COMPLETION REWARD =====
    def _record(self, task):
        task["finish_time"] = self.now
        task["response_s"] = self.now - task["arrival_time"]
        task["deadline_met"] = self.now <= task["deadline"]

        weight = {
            "urgent": 3.0, "heavy_compute": 1.5, "can_wait": 1.0
        }.get(task["label"], 1.0)

        # Reward is based on actual simulated completion.
        self.reward += weight if task["deadline_met"] else -weight
        self.completed.append(task)


===== ACTION REWARD AND HOLD CODE =====

--- ddqn_environment_finite_cloud.FiniteCloudFogEnv.step ---

        self.reward = 0.0
        task = self.current


        # Existing reward, unchanged.
        self.reward -= 0.2 + task["transfer_s"] / 100.0

        self._advance()

        return self._state(), float(self.reward), self.done, {
            "time": self.now,
            "completed": len(self.completed

In [11]:
from pathlib import Path
from datetime import datetime
import importlib
import sys
import json
import numpy as np
import shutil
import zipfile

project = Path("/kaggle/working/something_unique")
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

code = '''
"""Finite-cloud environment with a stronger hold penalty."""

import math
from ddqn_environment_finite_cloud import FiniteCloudFogEnv


class FiniteCloudHoldPenaltyEnv(FiniteCloudFogEnv):
    def __init__(self, runtime_model, hold_penalty=0.10, **kwargs):
        super().__init__(runtime_model, **kwargs)

        if not math.isfinite(hold_penalty) or hold_penalty < 0:
            raise ValueError("Hold penalty must be finite and nonnegative.")

        self.hold_penalty = float(hold_penalty)

    def step(self, action):
        state, reward, done, info = super().step(action)

        if action == 4:
            # Parent already deducted 0.01.
            # Replace that deduction while retaining completion rewards.
            reward -= self.hold_penalty - 0.01
            self.reward = float(reward)

        return state, float(reward), done, info
'''

path = project / "ddqn_environment_finite_cloud_hold_penalty.py"

if path.exists() and path.read_text() != code:
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    shutil.copy2(path, path.with_name(f"{path.stem}_before_{stamp}.py"))

compile(code, str(path), "exec")
path.write_text(code, encoding="utf-8")

importlib.invalidate_caches()
module = importlib.import_module(
    "ddqn_environment_finite_cloud_hold_penalty"
)
module = importlib.reload(module)

from ddqn_environment_finite_cloud import FiniteCloudFogEnv
FiniteCloudHoldPenaltyEnv = module.FiniteCloudHoldPenaltyEnv


# ---------- Controlled tests ----------

class TestRuntime:
    def predict(self, frame):
        return np.full(len(frame), 10.0)


def run_test(environment_class, hold_count):
    env = environment_class(
        TestRuntime(),
        cloud_cpu_capacity=96.0,
        cloud_memory_capacity=100.0,
        cloud_runtime_factor=0.3,
        max_holds=3,
        hold_seconds=1.0
    )

    env.reset([{
        "id": "test",
        "arrival_time": 0.0,
        "origin_node": 0,
        "cpu_request": 2.0,
        "mem_request": 0.5,
        "reference_runtime_s": 10.0,
        "data_size_mb": 40.0,
        "deadline": 100.0,
        "label": "can_wait"
    }])

    rewards = []

    for _ in range(hold_count):
        assert 4 in env.valid_actions()
        _, reward, _, _ = env.step(4)
        rewards.append(reward)

    if hold_count == 3:
        assert 4 not in env.valid_actions()

    _, reward, done, _ = env.step(3)
    rewards.append(reward)

    assert done
    assert len(env.completed) == 1
    assert not env.cloud.active
    assert not env.cloud.queue

    return env.completed[0], np.array(rewards)


# No holds: reward and execution must remain identical.
old_task, old_rewards = run_test(FiniteCloudFogEnv, 0)
new_task, new_rewards = run_test(FiniteCloudHoldPenaltyEnv, 0)

assert np.allclose(old_rewards, new_rewards)
assert old_task["finish_time"] == new_task["finish_time"]

print("Passed: non-hold rewards and execution remain unchanged.")


# Three holds: each costs exactly 0.09 more.
old_task, old_rewards = run_test(FiniteCloudFogEnv, 3)
new_task, new_rewards = run_test(FiniteCloudHoldPenaltyEnv, 3)

assert np.allclose(old_rewards[:3] - new_rewards[:3], 0.09)
assert np.isclose(old_rewards[-1], new_rewards[-1])
assert np.isclose(old_rewards.sum() - new_rewards.sum(), 0.27)
assert old_task["finish_time"] == new_task["finish_time"]
assert new_task["holds"] == 3

# One successful can-wait completion, one cloud placement,
# 20 seconds transfer, three holds:
expected_reward = 1.0 - 0.2 - 20.0 / 100.0 - 3 * 0.10
assert np.isclose(new_rewards.sum(), expected_reward)

print("Passed: three holds incur exactly 0.27 additional penalty.")
print("Passed: completion reward is retained.")
print("Passed: the three-hold limit remains enforced.")
print("Passed: total reward matches the written equation.")


# ---------- Save the treatment and reward explanation ----------

audit_dir = project / "results/week1_audit"
audit_dir.mkdir(parents=True, exist_ok=True)

treatment = {
    "environment": "FiniteCloudHoldPenaltyEnv",
    "old_hold_penalty": 0.01,
    "new_hold_penalty": 0.10,
    "max_holds": 3,
    "hold_seconds": 1.0,
    "justification": (
        "One predeclared stronger-penalty experiment; "
        "not a literature-derived optimum."
    ),
    "unchanged": [
        "completion weights",
        "cloud placement penalty",
        "transfer-time penalty",
        "deadlines",
        "network formula",
        "cloud runtime factor",
        "DDQN state features"
    ],
    "status": "Synthetic tests passed; retraining pending"
}

(project / "data/week1_hold_treatment.json").write_text(
    json.dumps(treatment, indent=2),
    encoding="utf-8"
)

reward_note = """# Verified reward and hold treatment

Step reward:
sum of completion rewards
- 0.2 per cloud placement
- placement transfer_seconds / 100
- 0.10 per hold.

Completion reward is +weight for meeting the deadline and -weight
for missing it. Weights: urgent=3, heavy_compute=1.5, can_wait=1.

The previous hold penalty was 0.01. Hold duration remains 1 second,
with at most 3 holds per task. The new penalty is an experimental
algorithm setting, not a measured physical quantity.

Completion feedback may concern tasks placed at earlier decisions.
It is shared system feedback, not immediate credit for only the
current task.

Required reward-related results:
- Deadline satisfaction by class and weighted completion score.
- Cloud placement count.
- Total placement transfer seconds.
- Hold count, holds/all decisions, and fraction of tasks held.
- Total episode reward, reconciled against the above terms.

Report WAN upload and CPU/memory resource-time separately.
They are useful outcomes but are not explicit reward terms.
No monetary cost model is implemented.

Retraining under this treatment is still pending.
"""

(audit_dir / "verified_reward_and_hold_treatment.md").write_text(
    reward_note, encoding="utf-8"
)

# ---------- Save checkpoint ----------

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = shutil.make_archive(
    f"/kaggle/working/fog_week1_checkpoint_{stamp}",
    "zip",
    root_dir=str(project.parent),
    base_dir=project.name
)

with zipfile.ZipFile(backup) as archive:
    assert archive.testzip() is None

print("\nCreated:", path)
print("Reward explanation and treatment settings saved.")
print("No training has run yet.")
print("Verified checkpoint:", backup)

Passed: non-hold rewards and execution remain unchanged.
Passed: three holds incur exactly 0.27 additional penalty.
Passed: completion reward is retained.
Passed: the three-hold limit remains enforced.
Passed: total reward matches the written equation.

Created: /kaggle/working/something_unique/ddqn_environment_finite_cloud_hold_penalty.py
Reward explanation and treatment settings saved.
No training has run yet.
Verified checkpoint: /kaggle/working/fog_week1_checkpoint_20261103_111258.zip


In [12]:
from pathlib import Path
from datetime import datetime
from time import perf_counter
import sys
import json
import random
import hashlib
import numpy as np
import pandas as pd
import torch
import joblib
import sklearn
import shutil
import zipfile
from IPython.display import display

project = Path("/kaggle/working/something_unique")
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from agents.escalation_ddqn_v2 import EscalationDDQN
from ddqn_environment_finite_cloud_hold_penalty import (
    FiniteCloudHoldPenaltyEnv
)

torch.set_num_threads(1)

# ---------- Load existing training data ----------

training_path = project / "data/ddqn_training_episodes.json"
payload = json.loads(training_path.read_text())

if isinstance(payload, list):
    episodes = payload
elif "episodes" in payload:
    episodes = payload["episodes"]
elif "train" in payload:
    episodes = payload["train"]
else:
    raise ValueError("Unrecognized training episode structure.")

episode_tasks = [
    episode["tasks"] if isinstance(episode, dict) else episode
    for episode in episodes
]

assert episode_tasks
assert all(isinstance(tasks, list) and tasks for tasks in episode_tasks)

comparison = json.loads(
    (project / "data/fresh_comparison_tasks.json").read_text()
)
comparison_ids = {task["id"] for task in comparison["tasks"]}

for tasks_in_episode in episode_tasks:
    ids = [task["id"] for task in tasks_in_episode]
    assert len(ids) == len(set(ids)), "Duplicate IDs within an episode."
    assert not set(ids) & comparison_ids, (
        "Training and comparison task IDs overlap."
    )

# ---------- Load the frozen runtime predictor ----------

runtime_bundle = joblib.load(
    project / "models/alibaba_runtime_rf.joblib"
)

saved_version = runtime_bundle.get("sklearn_version")
if saved_version and saved_version != sklearn.__version__:
    raise RuntimeError(
        f"Runtime model was saved with scikit-learn {saved_version}, "
        f"but this session has {sklearn.__version__}. "
        "Send this message before starting training."
    )

runtime_model = runtime_bundle["model"]
runtime_model.set_params(n_jobs=1)

# Check chronological separation when episode metadata is available.
for episode, tasks_in_episode in zip(episodes, episode_tasks):
    if isinstance(episode, dict) and "trace_start" in episode:
        start = float(episode["trace_start"])
        latest_arrival = start + max(
            task["arrival_time"] for task in tasks_in_episode
        )
        assert latest_arrival < float(comparison["original_start"]), (
            "Training arrivals overlap the comparison interval."
        )

features = pd.DataFrame([
    task
    for tasks_in_episode in episode_tasks
    for task in tasks_in_episode
])[["cpu_request", "mem_request"]].drop_duplicates()

training_lookup = dict(zip(
    map(tuple, features.to_numpy(dtype=float)),
    runtime_model.predict(features)
))

class TrainingRuntime:
    def predict(self, frame):
        return np.array([
            training_lookup[tuple(row)]
            for row in frame[
                ["cpu_request", "mem_request"]
            ].to_numpy(dtype=float)
        ])

# ---------- Load the predeclared settings ----------

capacity_config = json.loads(
    (project / "data/week1_cloud_capacity_config.json").read_text()
)
capacity = next(
    row for row in capacity_config["capacity_scenarios"]
    if row["role"] == "main_configuration"
)

treatment = json.loads(
    (project / "data/week1_hold_treatment.json").read_text()
)

settings = {
    "seed": 42,
    "passes": 1,
    "initialization": "fresh weights",
    "state_features": 10,
    "forecast_features": False,
    "cloud_cpu_capacity": capacity["cloud_cpu_capacity"],
    "cloud_memory_capacity": capacity["cloud_memory_capacity"],
    "cloud_runtime_factor": 0.3,
    "hold_penalty": treatment["new_hold_penalty"],
    "max_holds": treatment["max_holds"],
    "hold_seconds": treatment["hold_seconds"],
    "initial_epsilon": 0.8,
    "minimum_epsilon": 0.1,
    "epsilon_decay_per_decision": 0.9995,
    "batch_size": 64,
    "learning_rate": 0.001,
    "gamma": 0.99,
    "target_sync_every": 200,
    "training_episodes": len(episode_tasks),
    "training_tasks": sum(map(len, episode_tasks)),
    "training_data_sha256": hashlib.sha256(
        training_path.read_bytes()
    ).hexdigest(),
    "sklearn_version": sklearn.__version__,
    "torch_version": str(torch.__version__),
    "note": (
        "One training pass under finite cloud and stronger hold penalty. "
        "Not a convergence claim or isolated penalty-effect experiment."
    )
}

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run = project / "results" / f"finite_cloud_hold_training_{stamp}"
run.mkdir(parents=True, exist_ok=False)
(run / "settings.json").write_text(json.dumps(settings, indent=2))

agent = EscalationDDQN(
    seed=settings["seed"],
    epsilon=settings["initial_epsilon"],
    learning_rate=settings["learning_rate"],
    gamma=settings["gamma"],
    target_sync_every=settings["target_sync_every"]
)

order = list(range(len(episode_tasks)))
random.Random(settings["seed"]).shuffle(order)

logs = []
started = perf_counter()

print("Training directory:", run)
print("Episodes:", len(episode_tasks))
print("Tasks across episodes:", settings["training_tasks"])
print("One pass only; no comparison-workload evaluation yet.", flush=True)

# ---------- Single training pass ----------

for episode_index in order:
    tasks_in_episode = episode_tasks[episode_index]

    env = FiniteCloudHoldPenaltyEnv(
        TrainingRuntime(),
        cloud_cpu_capacity=settings["cloud_cpu_capacity"],
        cloud_memory_capacity=settings["cloud_memory_capacity"],
        cloud_runtime_factor=settings["cloud_runtime_factor"],
        hold_penalty=settings["hold_penalty"],
        max_holds=settings["max_holds"],
        hold_seconds=settings["hold_seconds"]
    )

    state = env.reset(tasks_in_episode)
    assert state.shape == (10,)

    decisions = 0
    holds = 0
    reward_total = 0.0
    losses = []

    print(
        f"\nEpisode {episode_index}: {len(tasks_in_episode):,} tasks",
        flush=True
    )

    while not env.done:
        action = agent.act(
            state, valid_actions=env.valid_actions()
        )

        next_state, reward, done, info = env.step(action)

        agent.remember(
            state, action, reward, next_state, done,
            next_valid_actions=info["valid_actions"]
        )

        loss = agent.train_step(batch_size=settings["batch_size"])
        if loss is not None:
            losses.append(loss)

        agent.epsilon = max(
            settings["minimum_epsilon"],
            agent.epsilon * settings["epsilon_decay_per_decision"]
        )

        state = next_state
        decisions += 1
        holds += int(action == 4)
        reward_total += reward

        assert decisions <= (
            settings["max_holds"] + 1
        ) * len(tasks_in_episode)

        if decisions % 1000 == 0:
            print(
                f"  {decisions:,} decisions; "
                f"elapsed {perf_counter() - started:.1f} s",
                flush=True
            )

    completed = pd.DataFrame(env.completed)
    assert len(completed) == len(tasks_in_episode)
    assert completed["id"].nunique() == len(tasks_in_episode)
    assert completed["holds"].sum() == holds
    assert completed["holds"].le(settings["max_holds"]).all()

    # Reconcile every term of the total episode reward.
    weights = completed["label"].map({
        "urgent": 3.0,
        "heavy_compute": 1.5,
        "can_wait": 1.0
    }).fillna(1.0)

    weighted_completion = (
        weights * np.where(completed["deadline_met"], 1.0, -1.0)
    ).sum()

    cloud_count = int(completed["route"].eq("cloud").sum())
    transfer_total = completed["transfer_s"].sum()

    reconstructed_reward = (
        weighted_completion
        - 0.2 * cloud_count
        - transfer_total / 100.0
        - settings["hold_penalty"] * holds
    )
    assert np.isclose(
        reward_total, reconstructed_reward, atol=1e-5
    ), "Episode reward does not match its recorded components."

    logs.append({
        "episode": episode_index,
        "tasks": len(completed),
        "decisions": decisions,
        "holds": holds,
        "hold_rate_pct": 100 * holds / decisions,
        "held_tasks_pct": 100 * completed["holds"].gt(0).mean(),
        "cloud_tasks": cloud_count,
        "deadline_met_pct": 100 * completed["deadline_met"].mean(),
        "weighted_completion_score": weighted_completion,
        "total_transfer_s": transfer_total,
        "episode_reward": reward_total,
        "epsilon": agent.epsilon,
        "mean_loss": np.mean(losses) if losses else np.nan,
        "gradient_updates": agent.training_steps
    })

    # Save progress after each complete episode.
    agent.save(run / "ddqn_latest.pt")
    completed.to_csv(
        run / f"training_episode_{episode_index}_tasks.csv",
        index=False
    )
    pd.DataFrame(logs).to_csv(run / "training_log.csv", index=False)

    print(
        f"Saved episode {episode_index}; "
        f"hold rate {100 * holds / decisions:.2f}%; "
        f"updates {agent.training_steps}",
        flush=True
    )

assert agent.training_steps > 0

(run / "TRAINING_COMPLETE.json").write_text(json.dumps({
    "completed_episodes": len(logs),
    "passes": 1,
    "gradient_updates": agent.training_steps,
    "elapsed_seconds": perf_counter() - started,
    "evaluation_pending": True
}, indent=2))

(project / "results/latest_finite_cloud_hold_training.txt").write_text(
    str(run), encoding="utf-8"
)

print("\n===== SINGLE-PASS TRAINING COMPLETE =====")
display(pd.DataFrame(logs).round(3))
print("Saved model:", run / "ddqn_latest.pt")
print("These are exploratory training results, not final evaluation.")

# ---------- Save checkpoint ----------

backup_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = shutil.make_archive(
    f"/kaggle/working/fog_week1_checkpoint_{backup_stamp}",
    "zip",
    root_dir=str(project.parent),
    base_dir=project.name
)
with zipfile.ZipFile(backup) as archive:
    assert archive.testzip() is None

print("Verified checkpoint:", backup)

Training directory: /kaggle/working/something_unique/results/finite_cloud_hold_training_20261103_111442
Episodes: 5
Tasks across episodes: 1918
One pass only; no comparison-workload evaluation yet.

Episode 3: 1,004 tasks
  1,000 decisions; elapsed 5.9 s
  2,000 decisions; elapsed 13.0 s
Saved episode 3; hold rate 56.85%; updates 2264

Episode 1: 70 tasks
Saved episode 1; hold rate 27.08%; updates 2360

Episode 2: 192 tasks
Saved episode 2; hold rate 54.18%; updates 2779

Episode 4: 549 tasks
  1,000 decisions; elapsed 25.4 s
Saved episode 4; hold rate 56.97%; updates 4055

Episode 0: 103 tasks
Saved episode 0; hold rate 33.12%; updates 4209

===== SINGLE-PASS TRAINING COMPLETE =====


,episode,tasks,decisions,holds,hold_rate_pct,held_tasks_pct,cloud_tasks,deadline_met_pct,weighted_completion_score,total_transfer_s,episode_reward,epsilon,mean_loss,gradient_updates
0,3,1004,2327,1323,56.854,59.064,730,100.0,1150.0,16414.667,707.553,0.250,0.114,2264
1,1,70,96,26,27.083,18.571,19,100.0,84.0,526.667,72.333,0.238,0.497,2360
2,2,192,419,227,54.177,53.125,78,100.0,228.0,2356.000,166.140,0.193,0.524,2779
3,4,549,1276,727,56.975,53.916,226,100.0,633.0,6066.667,454.433,0.102,0.438,4055
4,0,103,154,51,33.117,24.272,23,100.0,121.0,594.667,105.353,0.100,0.430,4209


Saved model: /kaggle/working/something_unique/results/finite_cloud_hold_training_20261103_111442/ddqn_latest.pt
These are exploratory training results, not final evaluation.
Verified checkpoint: /kaggle/working/fog_week1_checkpoint_20261103_111514.zip


In [13]:
from pathlib import Path
from datetime import datetime
import sys
import json
import joblib
import numpy as np
import pandas as pd
import torch
import sklearn
import shutil
import zipfile
from IPython.display import display

project = Path("/kaggle/working/something_unique")
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from agents.escalation_ddqn_v2 import EscalationDDQN
from ddqn_environment_finite_cloud_hold_penalty import (
    FiniteCloudHoldPenaltyEnv
)

torch.set_num_threads(1)

training_run = (
    project / "results"
    / "finite_cloud_hold_training_20261103_111442"
)
assert (training_run / "TRAINING_COMPLETE.json").is_file()

settings = json.loads((training_run / "settings.json").read_text())
definition = json.loads(
    (project / "data/fresh_comparison_tasks.json").read_text()
)
tasks = definition["tasks"]

baseline_runs = {
    "cloud_only": project / "results/week1_audit"
        / "finite_cloud_only_20261103_110159",
    "local_first": project / "results/week1_audit"
        / "finite_local_first_20261103_110622"
}

# ---------- Verify identical workload and cloud settings ----------

original = pd.DataFrame(tasks).set_index("id").sort_index()

for policy, folder in baseline_runs.items():
    previous = (
        pd.read_csv(folder / "tasks.csv")
        .set_index("id").sort_index()
    )
    assert original.index.equals(previous.index)

    for column in [
        "arrival_time", "origin_node", "cpu_request", "mem_request",
        "reference_runtime_s", "data_size_mb", "deadline"
    ]:
        assert np.allclose(original[column], previous[column])

    assert original["label"].equals(previous["label"])

    baseline_settings = json.loads(
        (folder / "settings.json").read_text()
    )
    for key in ["cloud_cpu_capacity", "cloud_memory_capacity"]:
        assert np.isclose(
            baseline_settings["capacity"][key], settings[key]
        )
    assert np.isclose(
        baseline_settings["cloud_runtime_factor"],
        settings["cloud_runtime_factor"]
    )

# ---------- Cache runtime predictions for evaluation tasks ----------

bundle = joblib.load(project / "models/alibaba_runtime_rf.joblib")
saved_version = bundle.get("sklearn_version")
assert not saved_version or saved_version == sklearn.__version__, (
    "scikit-learn version differs from the saved runtime model."
)

model = bundle["model"]
model.set_params(n_jobs=1)

features = original[["cpu_request", "mem_request"]].drop_duplicates()
evaluation_lookup = dict(zip(
    map(tuple, features.to_numpy(dtype=float)),
    model.predict(features)
))

class EvaluationRuntime:
    def predict(self, frame):
        return np.array([
            evaluation_lookup[tuple(row)]
            for row in frame[
                ["cpu_request", "mem_request"]
            ].to_numpy(dtype=float)
        ])

# ---------- Load the trained policy; disable exploration ----------

agent = EscalationDDQN(seed=settings["seed"])
agent.load(training_run / "ddqn_latest.pt")
agent.epsilon = 0.0
agent.online.eval()

updates_before = agent.training_steps

env = FiniteCloudHoldPenaltyEnv(
    EvaluationRuntime(),
    cloud_cpu_capacity=settings["cloud_cpu_capacity"],
    cloud_memory_capacity=settings["cloud_memory_capacity"],
    cloud_runtime_factor=settings["cloud_runtime_factor"],
    hold_penalty=settings["hold_penalty"],
    max_holds=settings["max_holds"],
    hold_seconds=settings["hold_seconds"]
)

state = env.reset(tasks)
decision_rows = []
reward_total = 0.0

print(f"Evaluating saved DDQN on {len(tasks):,} tasks.", flush=True)

while not env.done:
    task_id = env.current["id"]
    decision_time = env.now

    action = agent.act(
        state,
        greedy=True,
        valid_actions=env.valid_actions()
    )
    state, reward, done, info = env.step(action)

    decision_rows.append({
        "task_id": task_id,
        "time": decision_time,
        "action": action,
        "reward": reward
    })
    reward_total += reward

    assert len(decision_rows) <= (
        settings["max_holds"] + 1
    ) * len(tasks)

    if len(decision_rows) % 2000 == 0:
        print(
            f"{len(decision_rows):,} decisions; "
            f"{len(env.completed):,} completed tasks",
            flush=True
        )

# ---------- Verify completion and reward accounting ----------

completed = pd.DataFrame(env.completed)
decisions = pd.DataFrame(decision_rows)
history = pd.DataFrame(env.cloud.history)
holds = int(decisions["action"].eq(4).sum())

assert agent.training_steps == updates_before
assert len(completed) == len(tasks)
assert completed["id"].nunique() == len(tasks)
assert completed["holds"].sum() == holds
assert completed["holds"].le(settings["max_holds"]).all()
assert len(decisions) == len(tasks) + holds

execution = completed["finish_time"] - completed["start_time"]
waiting = (
    completed["start_time"]
    - completed["placement_time"]
    - completed["transfer_s"]
)

assert waiting.ge(-1e-6).all()
assert np.allclose(
    completed["response_s"],
    completed["placement_time"] - completed["arrival_time"]
    + completed["transfer_s"] + waiting + execution
)
assert np.allclose(
    execution,
    completed["reference_runtime_s"] * np.where(
        completed["route"].eq("cloud"),
        settings["cloud_runtime_factor"], 1.0
    )
)
assert (
    completed["deadline_met"]
    == (completed["finish_time"] <= completed["deadline"])
).all()

weights = completed["label"].map({
    "urgent": 3.0, "heavy_compute": 1.5, "can_wait": 1.0
}).fillna(1.0)

weighted_score = (
    weights * np.where(completed["deadline_met"], 1.0, -1.0)
).sum()

cloud = completed[completed["route"].eq("cloud")]
expected_reward = (
    weighted_score
    - 0.2 * len(cloud)
    - completed["transfer_s"].sum() / 100
    - settings["hold_penalty"] * holds
)
assert np.isclose(reward_total, expected_reward, atol=1e-5)

if not history.empty:
    assert history["used_cpu"].le(
        settings["cloud_cpu_capacity"] + 1e-6
    ).all()
    assert history["used_memory"].le(
        settings["cloud_memory_capacity"] + 1e-6
    ).all()

assert not env.cloud.active
assert not env.cloud.queue
assert abs(env.cloud.used_cpu) <= 1e-6
assert abs(env.cloud.used_memory) <= 1e-6
assert np.allclose(env.cpu, 0)
assert np.allclose(env.mem, 0)

# ---------- Three-policy response comparison ----------

response_rows = []
for label in ["overall", "urgent", "heavy_compute", "can_wait"]:
    group = (
        completed if label == "overall"
        else completed[completed["label"].eq(label)]
    )
    if group.empty:
        continue

    response_rows.append({
        "class": label,
        "tasks": len(group),
        "mean_response_s": group["response_s"].mean(),
        "p95_response_s": group["response_s"].quantile(0.95),
        "deadline_met_pct": 100 * group["deadline_met"].mean(),
        "deadline_misses": int((~group["deadline_met"]).sum())
    })

ddqn_summary = pd.DataFrame(response_rows)
columns = list(ddqn_summary.columns)

comparison = pd.concat([
    *[
        pd.read_csv(folder / "response_by_class.csv")[columns]
        .assign(policy=policy)
        for policy, folder in baseline_runs.items()
    ],
    ddqn_summary.assign(policy="ddqn_retrained")
], ignore_index=True)

print("\n===== THREE-POLICY RESPONSE AND DEADLINES =====")
display(comparison[["policy"] + columns].round(3))

# ---------- Cloud load comparison ----------

cloud_execution = cloud["finish_time"] - cloud["start_time"]
ddqn_load = pd.DataFrame([{
    "cloud_tasks": len(cloud),
    "wan_upload_mb": cloud["data_size_mb"].sum(),
    "cloud_execution_seconds": cloud_execution.sum(),
    "cloud_cpu_request_seconds": (
        cloud_execution * cloud["cpu_request"]
    ).sum(),
    "cloud_memory_request_seconds": (
        cloud_execution * cloud["mem_request"]
    ).sum(),
    "peak_reserved_cpu": (
        history["used_cpu"].max() if not history.empty else 0
    ),
    "peak_reserved_memory": (
        history["used_memory"].max() if not history.empty else 0
    ),
    "peak_queued_tasks": (
        history["queued_tasks"].max() if not history.empty else 0
    )
}])

load_comparison = pd.concat([
    *[
        pd.read_csv(folder / "cloud_load.csv").assign(policy=policy)
        for policy, folder in baseline_runs.items()
    ],
    ddqn_load.assign(policy="ddqn_retrained")
], ignore_index=True)

print("\n===== THREE-POLICY CLOUD LOAD AND WAN =====")
display(load_comparison.round(3))

hold_summary = pd.DataFrame([{
    "tasks": len(completed),
    "decisions": len(decisions),
    "holds": holds,
    "hold_rate_pct": 100 * holds / len(decisions),
    "held_tasks_pct": 100 * completed["holds"].gt(0).mean(),
    "weighted_completion_score": weighted_score,
    "total_transfer_s": completed["transfer_s"].sum(),
    "total_reward": reward_total
}])

print("\n===== DDQN HOLD RATE AND REWARD =====")
display(hold_summary.round(3))

routes = (
    completed["route"].value_counts()
    .rename_axis("route").reset_index(name="tasks")
)
print("\n===== DDQN ROUTES =====")
display(routes)

# ---------- Save evaluation and checkpoint ----------

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out = (
    project / "results/week1_audit"
    / f"finite_ddqn_evaluation_{stamp}"
)
out.mkdir(parents=True, exist_ok=False)

for filename, frame in {
    "tasks": completed,
    "decisions": decisions,
    "cloud_resource_history": history,
    "response_by_class": ddqn_summary,
    "cloud_load": ddqn_load,
    "policy_comparison": comparison,
    "load_comparison": load_comparison,
    "hold_summary": hold_summary,
    "routes": routes
}.items():
    frame.to_csv(out / f"{filename}.csv", index=False)

(out / "settings.json").write_text(json.dumps({
    "training_directory": str(training_run),
    "training_settings": settings,
    "greedy_evaluation": True,
    "evaluation_intervals": 1,
    "training_seeds": 1,
    "workload": "data/fresh_comparison_tasks.json",
    "status": "Development evaluation; not final held-out results"
}, indent=2))

print("\nPassed: no training updates during evaluation.")
print("Passed: completion, capacity and reward checks.")
print("One training seed and one development evaluation interval.")
print("Results saved:", out)

backup = shutil.make_archive(
    f"/kaggle/working/fog_week1_checkpoint_{stamp}",
    "zip",
    root_dir=str(project.parent),
    base_dir=project.name
)
with zipfile.ZipFile(backup) as archive:
    assert archive.testzip() is None

print("Verified checkpoint:", backup)

Evaluating saved DDQN on 5,871 tasks.
2,000 decisions; 335 completed tasks
4,000 decisions; 700 completed tasks
6,000 decisions; 1,033 completed tasks
8,000 decisions; 1,536 completed tasks
10,000 decisions; 2,217 completed tasks
12,000 decisions; 2,917 completed tasks
14,000 decisions; 3,405 completed tasks
16,000 decisions; 3,664 completed tasks
18,000 decisions; 3,721 completed tasks
20,000 decisions; 3,782 completed tasks

===== THREE-POLICY RESPONSE AND DEADLINES =====


,policy,class,tasks,mean_response_s,p95_response_s,deadline_met_pct,deadline_misses
0,cloud_only,overall,5871,2188.153,6696.750,59.854,2357
1,cloud_only,urgent,1148,2319.059,6697.285,58.972,471
2,cloud_only,heavy_compute,2795,2101.191,6727.870,60.358,1108
3,cloud_only,can_wait,1928,2236.275,6623.260,59.647,778
4,local_first,overall,5871,2045.774,6606.450,69.784,1774
5,local_first,urgent,1148,2135.950,6599.785,69.077,355
6,local_first,heavy_compute,2795,2057.692,6658.400,69.553,851
7,local_first,can_wait,1928,1974.804,6530.915,70.539,568
8,ddqn_retrained,overall,5871,2013.070,6612.850,70.295,1744
9,ddqn_retrained,urgent,1148,2137.779,6619.865,69.077,355



===== THREE-POLICY CLOUD LOAD AND WAN =====


,cloud_tasks,wan_upload_mb,cloud_execution_seconds,cloud_cpu_request_seconds,cloud_memory_request_seconds,peak_reserved_cpu,peak_reserved_memory,peak_queued_tasks,policy
0,5871,234840.0,360421.2,1010975.235,466444.455,96.0,49.17,1914,cloud_only
1,5041,201640.0,315276.0,1000297.125,461193.789,96.0,45.10,1770,local_first
2,4840,193600.0,319892.1,1002473.010,462748.359,96.0,45.35,1740,ddqn_retrained



===== DDQN HOLD RATE AND REWARD =====


,tasks,decisions,holds,hold_rate_pct,held_tasks_pct,weighted_completion_score,total_transfer_s,total_reward
0,5871,20900,15029,71.909,87.719,3811.5,99562.667,344.973



===== DDQN ROUTES =====


,route,tasks
0,cloud,4840
1,local,786
2,sibling,199
3,cross_cluster,46



Passed: no training updates during evaluation.
Passed: completion, capacity and reward checks.
One training seed and one development evaluation interval.
Results saved: /kaggle/working/something_unique/results/week1_audit/finite_ddqn_evaluation_20261103_111945
Verified checkpoint: /kaggle/working/fog_week1_checkpoint_20261103_111945.zip


In [14]:
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import shutil
import zipfile

project = Path("/kaggle/working/something_unique")
audit = project / "results/week1_audit"

evaluation = audit / "finite_ddqn_evaluation_20261103_111945"
sensitivity = audit / "finite_capacity_sensitivity_20261103_110944"

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out = audit / f"week1_summary_{stamp}"
out.mkdir(parents=True, exist_ok=False)

# Read saved results; do not manually retype experimental numbers.
response = pd.read_csv(evaluation / "policy_comparison.csv")
load = pd.read_csv(evaluation / "load_comparison.csv")
holds = pd.read_csv(evaluation / "hold_summary.csv")
capacity_results = pd.read_csv(sensitivity / "summary.csv")

assert set(response["policy"]) == {
    "cloud_only", "local_first", "ddqn_retrained"
}
assert len(response[response["class"].eq("overall")]) == 3

# Make Markdown tables without requiring the tabulate package.
def markdown_table(frame):
    frame = frame.round(3).fillna("")
    columns = list(frame.columns)
    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join(["---"] * len(columns)) + " |"
    ]
    for row in frame.itertuples(index=False, name=None):
        lines.append("| " + " | ".join(map(str, row)) + " |")
    return "\n".join(lines)

parameters = pd.DataFrame([
    {
        "parameter": "Cloud resource capacity",
        "old": "Unlimited",
        "current": "96 CPU / 100 normalized memory; 2x and 4x sensitivity",
        "basis": "Modal configuration of 3955 usable Alibaba machines",
        "status": "Trace-derived capacities; allocation size is experimental"
    },
    {
        "parameter": "Cloud admission",
        "old": "Immediate execution after upload",
        "current": "FCFS queue; reserve both CPU and memory",
        "basis": "Finite-resource modeling; CloudSim reference",
        "status": "Implemented and correctness-tested"
    },
    {
        "parameter": "Cloud runtime",
        "old": "0.3 x recorded task duration",
        "current": "Unchanged",
        "basis": "Legacy simulation assumption",
        "status": "External justification/calibration pending"
    },
    {
        "parameter": "Cloud transfer delay",
        "old": "4 + 8 x payload_MB / 20 seconds",
        "current": "Unchanged",
        "basis": "Serialization formula; assumed 4 s latency and 20 Mbps",
        "status": "Formula tested; numerical calibration pending"
    },
    {
        "parameter": "Task payload",
        "old": "40 MB per task",
        "current": "Unchanged",
        "basis": "Synthetic workload assumption",
        "status": "No empirical payload distribution established"
    },
    {
        "parameter": "Deadline",
        "old": "arrival + 2 x recorded duration + 30 seconds",
        "current": "Unchanged",
        "basis": "Synthetic deadline rule",
        "status": "Differentiates outcomes with finite cloud; not a trace SLA"
    },
    {
        "parameter": "Hold penalty",
        "old": "0.01 per hold",
        "current": "0.10 per hold",
        "basis": "One predeclared stronger-penalty experiment",
        "status": "Tested; one retraining completed"
    },
    {
        "parameter": "Hold limit and duration",
        "old": "3 holds; 1 second each",
        "current": "Unchanged",
        "basis": "Existing algorithm settings",
        "status": "Limit verified"
    }
])

report = f"""# Week 1: simulation and DDQN

## Experimental scope

One training seed (42), one training pass across five episodes containing
1918 tasks, and one development evaluation interval containing 5871 tasks.
The evaluation interval was already inspected and is not a final held-out test.

All three policies use identical evaluation tasks, deadlines, network settings
and the main finite-cloud allocation. DDQN evaluation disables exploration.
The original ten-feature state is retained; forecast integration is pending.

## Overall and per-class response and deadlines

{markdown_table(response)}

## Cloud load and WAN traffic

{markdown_table(load)}

Cloud execution seconds and CPU/memory request-seconds are load proxies.
They are not monetary costs or measured hardware utilization.

## Holding and reward

{markdown_table(holds)}

Reward per step:
sum of completion rewards
- 0.2 per cloud placement
- placement transfer_seconds / 100
- 0.10 per hold.

Completion reward is +w for meeting the deadline and -w for missing it,
where urgent=3, heavy_compute=1.5 and can_wait=1.
Completion feedback can concern tasks placed at earlier decisions.

The hold penalty increased from 0.01 to 0.10; the three-hold limit remained.
After one retraining, evaluation still produced 15029 holds in 20900 decisions
(71.909%). We report this outcome without further hold-penalty tuning.

This experiment does not isolate the effect of the penalty because the
training environment also changed.

## Capacity sensitivity: baselines only

{markdown_table(capacity_results)}

The main 1x results appear in the three-policy comparison above.
The 2x and 4x experiments evaluated cloud-only and local-first.
DDQN capacity sensitivity has not been run.

## Interpretation

At the main allocation, DDQN meets 613 more deadlines than cloud-only
and sends 41240 MB less data over the WAN.

Relative to local-first, DDQN meets 30 additional deadlines and sends
8040 MB less WAN data. Urgent deadline satisfaction is unchanged.
DDQN has a lower overall mean response, but slightly worse P95.

DDQN cloud CPU-request seconds are slightly higher than local-first.
Fewer cloud placements therefore do not establish lower cloud computation.

With larger cloud allocations, cloud-only has better overall mean and P95
response than local-first, while local-first retains a deadline-satisfaction
and WAN-traffic advantage in this interval.

These are descriptive development results, not evidence of statistical
significance or general superiority.

## Parameter changes and evidence

{markdown_table(parameters)}

References:
- Alibaba schema:
  https://github.com/alibaba/clusterdata/blob/master/cluster-trace-v2018/schema.txt
- CloudSim:
  https://arxiv.org/abs/0903.2525

CloudSim supports finite-resource modeling; it does not establish the chosen
cloud runtime factor, latency, bandwidth or number of machine equivalents.

## Remaining work

Week 1: resolve or explicitly document the outstanding parameter justification.
The main implementation, one retraining and development evaluation are complete.

Week 2: connect existing RF forecasts to the state and run a matched
with/without-forecast ablation.

Week 3: five seeds/intervals, an untouched final interval, and 48-node scaling
with a first-fit baseline.
"""

(out / "WEEK1_SUMMARY.md").write_text(report, encoding="utf-8")
response.to_csv(out / "response_and_deadlines.csv", index=False)
load.to_csv(out / "cloud_load_and_wan.csv", index=False)
holds.to_csv(out / "holding_and_reward.csv", index=False)
capacity_results.to_csv(out / "capacity_sensitivity.csv", index=False)
parameters.to_csv(out / "parameter_justification.csv", index=False)

print("Saved Week 1 summary:", out / "WEEK1_SUMMARY.md")
print("Saved five supporting CSV tables.")
print("Parameter justification remains explicitly marked where unresolved.")

backup = shutil.make_archive(
    f"/kaggle/working/fog_week1_checkpoint_{stamp}",
    "zip",
    root_dir=str(project.parent),
    base_dir=project.name
)
with zipfile.ZipFile(backup) as archive:
    assert archive.testzip() is None

print("Verified checkpoint:", backup)

Saved Week 1 summary: /kaggle/working/something_unique/results/week1_audit/week1_summary_20261103_112218/WEEK1_SUMMARY.md
Saved five supporting CSV tables.
Parameter justification remains explicitly marked where unresolved.
Verified checkpoint: /kaggle/working/fog_week1_checkpoint_20261103_112218.zip


In [15]:
from pathlib import Path
import pandas as pd

folder = Path(
    "/kaggle/working/something_unique/results/week1_audit/"
    "week1_summary_20261103_112218"
)

source = "https://doi.org/10.3390/electronics14020380"

note = f"""
## Additional network-parameter evidence

The paper "Cooperative Service Caching and Task Offloading in Mobile
Edge Computing: A Novel Hierarchical Reinforcement Learning Approach"
uses a 20-Mbps transmission rate between its MEC server and cloud centre
in Table 1.

Source: {source}

This supplies a published precedent for the selected 20-Mbps rate.
It does not calibrate our deployment or validate our complete workload:
the paper uses 1–2 MB tasks, whereas this experiment uses 40 MB.

The fixed 4-second latency and 0.3 cloud-runtime multiplier remain
uncalibrated experimental assumptions. No simulation parameters,
models or results were changed by this documentation update.
"""

summary = folder / "WEEK1_SUMMARY.md"
text = summary.read_text(encoding="utf-8")
if source not in text:
    summary.write_text(text + "\n" + note, encoding="utf-8")

table_path = folder / "parameter_justification.csv"
table = pd.read_csv(table_path)

row = table["parameter"].eq("Cloud transfer delay")
assert row.sum() == 1

table.loc[row, "basis"] = (
    "20 Mbps: published MEC-to-cloud precedent, "
    "Table 1, doi:10.3390/electronics14020380. "
    "The 4-second fixed latency remains an assumption."
)
table.loc[row, "status"] = (
    "Rate has literature precedent; latency calibration pending. "
    "Paper task sizes differ from this workload."
)
table.to_csv(table_path, index=False)

print("Updated summary and parameter table.")
print("Simulation settings and results unchanged.")

Updated summary and parameter table.
Simulation settings and results unchanged.


In [16]:
from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd
import shutil
import zipfile
from IPython.display import display

project = Path("/kaggle/working/something_unique")

assert "EvaluationRuntime" in globals(), (
    "EvaluationRuntime is missing. Send this message before continuing."
)

from agents.escalation_ddqn_v2 import EscalationDDQN
from ddqn_environment_finite_cloud_hold_penalty import (
    FiniteCloudHoldPenaltyEnv
)

# Separate experimental class; original environment files stay unchanged.
class AssumptionCheckEnv(FiniteCloudHoldPenaltyEnv):
    def __init__(self, runtime_model, fixed_latency_s=4.0, **kwargs):
        super().__init__(runtime_model, **kwargs)
        if not np.isfinite(fixed_latency_s) or fixed_latency_s < 0:
            raise ValueError("Latency must be finite and nonnegative.")
        self.fixed_latency_s = float(fixed_latency_s)

    def step(self, action):
        if action != 3:
            return super().step(action)

        if self.done:
            raise RuntimeError("Episode is finished.")
        if action not in self.valid_actions():
            raise ValueError("Invalid cloud action.")

        self.reward = 0.0
        task = self.current
        task["route"] = "cloud"
        task["destination"] = "cloud"
        task["placement_time"] = self.now

        # Keep payload-dependent serialization and 20-Mbps rate.
        task["transfer_s"] = (
            self.fixed_latency_s
            + 8 * task["data_size_mb"] / 20.0
        )
        task["expected_ready_time"] = self.now + task["transfer_s"]

        self._push(
            task["expected_ready_time"], "finite_cloud_ready", task
        )
        self.reward -= 0.2 + task["transfer_s"] / 100.0
        self._advance()

        return self._state(), float(self.reward), self.done, {
            "time": self.now,
            "completed": len(self.completed),
            "valid_actions": self.valid_actions()
        }


# ---------- Check the configurable timing with one artificial task ----------

class TestRuntime:
    def predict(self, frame):
        return np.full(len(frame), 10.0)

def timing_check(latency, runtime_factor):
    env = AssumptionCheckEnv(
        TestRuntime(),
        fixed_latency_s=latency,
        cloud_cpu_capacity=96,
        cloud_memory_capacity=100,
        cloud_runtime_factor=runtime_factor,
        hold_penalty=0.10
    )
    env.reset([{
        "id": "timing_test", "arrival_time": 0.0,
        "origin_node": 0, "cpu_request": 1.0, "mem_request": 0.5,
        "reference_runtime_s": 10.0, "data_size_mb": 40.0,
        "deadline": 100.0, "label": "can_wait"
    }])
    env.step(3)
    task = env.completed[0]
    assert np.isclose(task["transfer_s"], latency + 16.0)
    assert np.isclose(
        task["response_s"], latency + 16.0 + runtime_factor * 10.0
    )

timing_check(4.0, 0.3)
timing_check(0.0, 0.3)
timing_check(4.0, 1.0)
print("Passed: original timing and both alternative timing checks.")


# ---------- Freeze the two diagnostic scenarios ----------

training_run = (
    project / "results/finite_cloud_hold_training_20261103_111442"
)
settings = json.loads((training_run / "settings.json").read_text())
tasks = json.loads(
    (project / "data/fresh_comparison_tasks.json").read_text()
)["tasks"]

scenarios = [
    {
        "name": "zero_fixed_latency",
        "fixed_latency_s": 0.0,
        "cloud_runtime_factor": 0.3
    },
    {
        "name": "equal_execution_speed",
        "fixed_latency_s": 4.0,
        "cloud_runtime_factor": 1.0
    }
]

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out = (
    project / "results/week1_audit"
    / f"assumption_checks_{stamp}"
)
out.mkdir(parents=True, exist_ok=False)

(out / "settings.json").write_text(json.dumps({
    "scenarios": scenarios,
    "cloud_cpu_capacity": settings["cloud_cpu_capacity"],
    "cloud_memory_capacity": settings["cloud_memory_capacity"],
    "bandwidth_mbps": 20,
    "training_directory": str(training_run),
    "training_run": False,
    "evaluation_intervals": 1,
    "interpretation": (
        "One-factor diagnostic scenarios with frozen policies; "
        "not calibrated deployments or policies retrained for each scenario."
    )
}, indent=2))

policy = EscalationDDQN(seed=settings["seed"])
policy.load(training_run / "ddqn_latest.pt")
policy.epsilon = 0.0
policy.online.eval()
updates_before = policy.training_steps

rows = []

# ---------- Six evaluations; no learning ----------

for scenario in scenarios:
    for policy_name in ["cloud_only", "local_first", "ddqn_retrained"]:
        print(
            f"\n{scenario['name']} | {policy_name}",
            flush=True
        )

        env = AssumptionCheckEnv(
            EvaluationRuntime(),
            fixed_latency_s=scenario["fixed_latency_s"],
            cloud_runtime_factor=scenario["cloud_runtime_factor"],
            cloud_cpu_capacity=settings["cloud_cpu_capacity"],
            cloud_memory_capacity=settings["cloud_memory_capacity"],
            hold_penalty=settings["hold_penalty"],
            max_holds=settings["max_holds"],
            hold_seconds=settings["hold_seconds"]
        )
        state = env.reset(tasks)
        decisions = 0
        holds = 0
        reward_total = 0.0

        while not env.done:
            valid = env.valid_actions()

            if policy_name == "cloud_only":
                action = 3
            elif policy_name == "local_first":
                action = next(a for a in [0, 1, 2, 3] if a in valid)
            else:
                action = policy.act(
                    state, greedy=True, valid_actions=valid
                )

            state, reward, _, _ = env.step(action)
            decisions += 1
            holds += int(action == 4)
            reward_total += reward

            assert decisions <= (
                settings["max_holds"] + 1
            ) * len(tasks)

        completed = pd.DataFrame(env.completed)
        history = pd.DataFrame(env.cloud.history)

        assert len(completed) == len(tasks)
        assert completed["id"].nunique() == len(tasks)
        assert decisions == len(tasks) + holds
        assert completed["holds"].sum() == holds
        assert completed["holds"].le(settings["max_holds"]).all()

        is_cloud = completed["route"].eq("cloud")
        execution = completed["finish_time"] - completed["start_time"]
        waiting = (
            completed["start_time"]
            - completed["placement_time"]
            - completed["transfer_s"]
        )

        assert waiting.ge(-1e-6).all()
        assert np.allclose(
            execution,
            completed["reference_runtime_s"] * np.where(
                is_cloud, scenario["cloud_runtime_factor"], 1.0
            )
        )
        assert np.allclose(
            completed.loc[is_cloud, "transfer_s"],
            scenario["fixed_latency_s"]
            + 8 * completed.loc[is_cloud, "data_size_mb"] / 20.0
        )
        assert np.allclose(
            completed["response_s"],
            completed["placement_time"] - completed["arrival_time"]
            + completed["transfer_s"] + waiting + execution
        )
        assert (
            completed["deadline_met"]
            == (completed["finish_time"] <= completed["deadline"])
        ).all()

        weights = completed["label"].map({
            "urgent": 3.0, "heavy_compute": 1.5, "can_wait": 1.0
        }).fillna(1.0)
        expected_reward = (
            (weights * np.where(completed["deadline_met"], 1, -1)).sum()
            - 0.2 * is_cloud.sum()
            - completed["transfer_s"].sum() / 100.0
            - settings["hold_penalty"] * holds
        )
        assert np.isclose(reward_total, expected_reward, atol=1e-5)

        if not history.empty:
            assert history["used_cpu"].le(
                settings["cloud_cpu_capacity"] + 1e-6
            ).all()
            assert history["used_memory"].le(
                settings["cloud_memory_capacity"] + 1e-6
            ).all()

        assert not env.cloud.active and not env.cloud.queue
        assert abs(env.cloud.used_cpu) <= 1e-6
        assert abs(env.cloud.used_memory) <= 1e-6
        assert np.allclose(env.cpu, 0)
        assert np.allclose(env.mem, 0)

        for label in ["overall", "urgent", "heavy_compute", "can_wait"]:
            group = (
                completed if label == "overall"
                else completed[completed["label"].eq(label)]
            )
            if group.empty:
                continue

            cloud = group[group["route"].eq("cloud")]
            group_holds = int(group["holds"].sum())

            rows.append({
                "scenario": scenario["name"],
                "policy": policy_name,
                "class": label,
                "tasks": len(group),
                "mean_response_s": group["response_s"].mean(),
                "p95_response_s": group["response_s"].quantile(0.95),
                "deadline_met_pct": 100 * group["deadline_met"].mean(),
                "cloud_tasks": len(cloud),
                "wan_upload_mb": cloud["data_size_mb"].sum(),
                "cloud_cpu_request_seconds": (
                    (cloud["finish_time"] - cloud["start_time"])
                    * cloud["cpu_request"]
                ).sum(),
                "hold_rate_pct": (
                    100 * group_holds / (len(group) + group_holds)
                )
            })

        prefix = f"{scenario['name']}_{policy_name}"
        completed.to_csv(out / f"{prefix}_tasks.csv", index=False)
        history.to_csv(out / f"{prefix}_cloud_history.csv", index=False)
        pd.DataFrame(rows).to_csv(out / "summary.csv", index=False)
        print("Passed checks and saved.", flush=True)

assert policy.training_steps == updates_before

summary = pd.DataFrame(rows)
print("\n===== ASSUMPTION CHECKS: OVERALL AND URGENT =====")
display(
    summary[summary["class"].isin(["overall", "urgent"])].round(3)
)

backup = shutil.make_archive(
    f"/kaggle/working/fog_week1_checkpoint_{stamp}",
    "zip",
    root_dir=str(project.parent),
    base_dir=project.name
)
with zipfile.ZipFile(backup) as archive:
    assert archive.testzip() is None

print("\nNo retraining occurred.")
print("Results saved:", out)
print("Verified checkpoint:", backup)

Passed: original timing and both alternative timing checks.

zero_fixed_latency | cloud_only
Passed checks and saved.

zero_fixed_latency | local_first
Passed checks and saved.

zero_fixed_latency | ddqn_retrained
Passed checks and saved.

equal_execution_speed | cloud_only
Passed checks and saved.

equal_execution_speed | local_first
Passed checks and saved.

equal_execution_speed | ddqn_retrained
Passed checks and saved.

===== ASSUMPTION CHECKS: OVERALL AND URGENT =====


,scenario,policy,class,tasks,mean_response_s,p95_response_s,deadline_met_pct,cloud_tasks,wan_upload_mb,cloud_cpu_request_seconds,hold_rate_pct
0,zero_fixed_latency,cloud_only,overall,5871,2184.153,6692.750,65.236,5871,234840.0,1010975.235,0.000
1,zero_fixed_latency,cloud_only,urgent,1148,2315.059,6693.285,64.286,1148,45920.0,293637.690,0.000
4,zero_fixed_latency,local_first,overall,5871,2042.340,6602.450,69.784,5041,201640.0,1000297.125,0.000
5,zero_fixed_latency,local_first,urgent,1148,2132.518,6595.785,69.077,985,39400.0,291842.685,0.000
8,zero_fixed_latency,ddqn_retrained,overall,5871,2009.772,6608.850,70.295,4840,193600.0,1002473.010,71.909
9,zero_fixed_latency,ddqn_retrained,urgent,1148,2134.487,6615.865,69.077,945,37800.0,292083.780,71.870
12,equal_execution_speed,cloud_only,overall,5871,7480.979,22791.000,21.955,5871,234840.0,3369917.450,0.000
13,equal_execution_speed,cloud_only,urgent,1148,7924.648,22775.850,22.125,1148,45920.0,978792.300,0.000
16,equal_execution_speed,local_first,overall,5871,6872.667,22432.500,33.197,5041,201640.0,3334323.750,0.000
17,equal_execution_speed,local_first,urgent,1148,7180.434,22385.300,33.537,985,39400.0,972808.950,0.000



No retraining occurred.
Results saved: /kaggle/working/something_unique/results/week1_audit/assumption_checks_20261103_112527
Verified checkpoint: /kaggle/working/fog_week1_checkpoint_20261103_112527.zip


In [17]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import shutil
import zipfile

project = Path("/kaggle/working/something_unique")
audit = project / "results/week1_audit"
summary_folder = audit / "week1_summary_20261103_112218"
checks_folder = audit / "assumption_checks_20261103_112527"

results = pd.read_csv(checks_folder / "summary.csv")
overall = results[results["class"].eq("overall")].copy()

assert len(overall) == 6
assert set(overall["scenario"]) == {
    "zero_fixed_latency", "equal_execution_speed"
}

results.to_csv(
    summary_folder / "assumption_sensitivity.csv",
    index=False
)

columns = [
    "scenario", "policy", "mean_response_s",
    "p95_response_s", "deadline_met_pct", "hold_rate_pct"
]
table = overall[columns].round(3)

lines = [
    "| " + " | ".join(columns) + " |",
    "| " + " | ".join(["---"] * len(columns)) + " |"
]
for row in table.itertuples(index=False, name=None):
    lines.append("| " + " | ".join(map(str, row)) + " |")

heading = "## Final Week 1 assumption checks"

note = f"""
{heading}

Two one-factor diagnostic scenarios were evaluated using frozen policies:
zero fixed cloud latency, and equal cloud/fog execution duration.
Cloud capacity remained 96 CPU / 100 normalized memory units.
All three policies were evaluated; no retraining occurred.

{chr(10).join(lines)}

Removing fixed latency reduced, but did not eliminate, the deadline
advantage over cloud-only on this interval. Removing the cloud execution
speed advantage produced severe congestion for all policies.

Cloud placement counts and WAN traffic remained unchanged across these
checks. The existing DDQN state has no explicit cloud congestion features.
Its hold rate remained 71.909%.

These results cover one development interval and one DDQN training seed.
They test sensitivity; they do not calibrate the assumed 4-second latency
or 0.3 runtime multiplier. No general superiority or statistical
significance is established.

Week 1 experimental work is closed for now. Numerical parameter
calibration remains a documented limitation. No further hold tuning
is planned. Next: Week 2 RF forecast integration and matched ablation.
"""

report = summary_folder / "WEEK1_SUMMARY.md"
text = report.read_text(encoding="utf-8")
if heading not in text:
    report.write_text(text + "\n" + note, encoding="utf-8")

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup = shutil.make_archive(
    f"/kaggle/working/fog_week1_checkpoint_{stamp}",
    "zip",
    root_dir=str(project.parent),
    base_dir=project.name
)
with zipfile.ZipFile(backup) as archive:
    assert archive.testzip() is None

print("Week 1 summary updated with assumption checks.")
print("Verified checkpoint:", backup)

Week 1 summary updated with assumption checks.
Verified checkpoint: /kaggle/working/fog_week1_checkpoint_20261103_112951.zip


In [18]:
from pathlib import Path
import json
import joblib
import sklearn

project = Path("/kaggle/working/something_unique")

print("===== SAVED FORECAST MODELS =====")
for path in sorted((project / "models").glob("*forecast*.joblib")):
    print(path.name)

model_path = project / "models/forecasting_sim_rf_v2_120s.joblib"
assert model_path.is_file(), (
    "Expected forecast model not found. Send the model list above."
)

bundle = joblib.load(model_path)
assert isinstance(bundle, dict), "Unexpected forecast model format."

print("\n===== FORECAST MODEL METADATA =====")
for key in [
    "window",
    "step_seconds",
    "horizon_seconds",
    "feature_order",
    "training_episodes",
    "training_windows",
    "sklearn_version",
    "status"
]:
    print(f"{key}: {bundle.get(key, 'not recorded')}")

model = bundle["model"]
print("Model class:", type(model).__name__)
print("Input features:", getattr(model, "n_features_in_", "not recorded"))
print("Outputs:", getattr(model, "n_outputs_", "not recorded"))
print("Current scikit-learn:", sklearn.__version__)

print("\n===== AVAILABLE EPISODE LENGTHS =====")

for filename in [
    "ddqn_training_episodes.json",
    "forecast_simulation_episodes.json"
]:
    path = project / "data" / filename
    if not path.exists():
        print(filename, "— missing")
        continue

    payload = json.loads(path.read_text())
    print(f"\n{filename}")

    groups = (
        {"episodes": payload}
        if isinstance(payload, list)
        else {
            key: payload[key]
            for key in ["episodes", "train", "validation"]
            if key in payload
        }
    )

    for split, episodes in groups.items():
        for index, episode in enumerate(episodes):
            tasks = episode["tasks"] if isinstance(episode, dict) else episode
            if not tasks:
                continue

            arrivals = [float(task["arrival_time"]) for task in tasks]
            name = (
                episode.get("name", episode.get("episode_id", index))
                if isinstance(episode, dict) else index
            )

            print(
                f"{split} | {name} | {len(tasks):,} tasks | "
                f"first arrival={min(arrivals):g}s | "
                f"last arrival={max(arrivals):g}s"
            )

print("\nRead-only check complete. No models or files changed.")

===== SAVED FORECAST MODELS =====
forecasting.joblib
forecasting_rf_120s.joblib
forecasting_sim_rf_120s.joblib
forecasting_sim_rf_v2_120s.joblib

===== FORECAST MODEL METADATA =====
window: 6
step_seconds: 30
horizon_seconds: 120
feature_order: oldest to newest; CPU then memory; fractions 0–1
training_episodes: ['train_0', 'train_1', 'train_extra']
training_windows: 912
sklearn_version: 1.6.1
status: development candidate; DDQN inputs unchanged
Model class: RandomForestRegressor
Input features: 12
Outputs: 2
Current scikit-learn: 1.6.1

===== AVAILABLE EPISODE LENGTHS =====

ddqn_training_episodes.json
episodes | 0 | 103 tasks | first arrival=0s | last arrival=59s
episodes | 1 | 70 tasks | first arrival=0s | last arrival=59s
episodes | 2 | 192 tasks | first arrival=3s | last arrival=59s
episodes | 3 | 1,004 tasks | first arrival=0s | last arrival=59s
episodes | 4 | 549 tasks | first arrival=0s | last arrival=59s

forecast_simulation_episodes.json
train | train_0 | 4,993 tasks | first a

In [19]:
from pathlib import Path
import json
import joblib
import sklearn

project = Path("/kaggle/working/something_unique")

print("===== SAVED FORECAST MODELS =====")
for path in sorted((project / "models").glob("*forecast*.joblib")):
    print(path.name)

model_path = project / "models/forecasting_sim_rf_v2_120s.joblib"
assert model_path.is_file(), (
    "Expected forecast model not found. Send the model list above."
)

bundle = joblib.load(model_path)
assert isinstance(bundle, dict), "Unexpected forecast model format."

print("\n===== FORECAST MODEL METADATA =====")
for key in [
    "window",
    "step_seconds",
    "horizon_seconds",
    "feature_order",
    "training_episodes",
    "training_windows",
    "sklearn_version",
    "status"
]:
    print(f"{key}: {bundle.get(key, 'not recorded')}")

model = bundle["model"]
print("Model class:", type(model).__name__)
print("Input features:", getattr(model, "n_features_in_", "not recorded"))
print("Outputs:", getattr(model, "n_outputs_", "not recorded"))
print("Current scikit-learn:", sklearn.__version__)

print("\n===== AVAILABLE EPISODE LENGTHS =====")

for filename in [
    "ddqn_training_episodes.json",
    "forecast_simulation_episodes.json"
]:
    path = project / "data" / filename
    if not path.exists():
        print(filename, "— missing")
        continue

    payload = json.loads(path.read_text())
    print(f"\n{filename}")

    groups = (
        {"episodes": payload}
        if isinstance(payload, list)
        else {
            key: payload[key]
            for key in ["episodes", "train", "validation"]
            if key in payload
        }
    )

    for split, episodes in groups.items():
        for index, episode in enumerate(episodes):
            tasks = episode["tasks"] if isinstance(episode, dict) else episode
            if not tasks:
                continue

            arrivals = [float(task["arrival_time"]) for task in tasks]
            name = (
                episode.get("name", episode.get("episode_id", index))
                if isinstance(episode, dict) else index
            )

            print(
                f"{split} | {name} | {len(tasks):,} tasks | "
                f"first arrival={min(arrivals):g}s | "
                f"last arrival={max(arrivals):g}s"
            )

print("\nRead-only check complete. No models or files changed.")

===== SAVED FORECAST MODELS =====
forecasting.joblib
forecasting_rf_120s.joblib
forecasting_sim_rf_120s.joblib
forecasting_sim_rf_v2_120s.joblib

===== FORECAST MODEL METADATA =====
window: 6
step_seconds: 30
horizon_seconds: 120
feature_order: oldest to newest; CPU then memory; fractions 0–1
training_episodes: ['train_0', 'train_1', 'train_extra']
training_windows: 912
sklearn_version: 1.6.1
status: development candidate; DDQN inputs unchanged
Model class: RandomForestRegressor
Input features: 12
Outputs: 2
Current scikit-learn: 1.6.1

===== AVAILABLE EPISODE LENGTHS =====

ddqn_training_episodes.json
episodes | 0 | 103 tasks | first arrival=0s | last arrival=59s
episodes | 1 | 70 tasks | first arrival=0s | last arrival=59s
episodes | 2 | 192 tasks | first arrival=3s | last arrival=59s
episodes | 3 | 1,004 tasks | first arrival=0s | last arrival=59s
episodes | 4 | 549 tasks | first arrival=0s | last arrival=59s

forecast_simulation_episodes.json
train | train_0 | 4,993 tasks | first a